# PhotoGraphiQ manuscript experiment suite (R1-R42)

One executable notebook combining every script in `paper_experiments/` (`common.py`, `metadata.py`, and experiments R1-R42, in order). Each experiment's own module docstring is reproduced as its section heading.

**Runtime**: running the whole notebook top-to-bottom executes every experiment, including the large-Fock and long-benchmark ones excluded from `run_all_safe.py` (see `README.md`) -- expect roughly 30-45 minutes in total, dominated by R28 (~9 min), R36 (~4 min) and the cutoff=24-96 non-Gaussian/GKP scripts (R18, R20-R23, R26-R28, R33-R34).

**Isolation note**: `run_all_safe.py` runs each script in its own subprocess so failures/warnings can't leak between experiments; this notebook instead runs everything in one kernel for a single linear narrative. Every script's `main()` is still called immediately after its own constants/helpers are (re)defined in the same cell, so redefining a same-named helper (e.g. `independent_rotation`) in a later cell cannot affect an earlier cell's already-completed run.

**Optional dependencies**: R17 (Graphix), R41-R42 (JAX) print `SKIPPED` and raise `SystemExit(0)` if their optional package is missing -- Jupyter shows this as a brief traceback-like message but the notebook keeps going. The install cell below installs both, so on a fresh environment they run for real.

## Install dependencies

In [1]:
# Core + optional extras (graphix, jax) + matplotlib, from the local checkout.
# Run once per environment; safe to re-run (pip no-ops on already-satisfied requirements).
import subprocess
import sys

try:
    import pip  # noqa: F401
except ImportError:
    # Some environments (e.g. a uv-provisioned interpreter) ship no pip at
    # all; bootstrap it from the standard library before installing anything.
    print("pip not found in this interpreter; bootstrapping via ensurepip...")
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "..[dev,autodiff]"],
    check=True,
)


CompletedProcess(args=['D:\\Research\\QML\\results_1\\cvqc\\thesis_part_2\\softwares\\PhotoGraphiQ\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-e', '..[dev,autodiff]'], returncode=0)

## Setup: make `common` and `metadata` importable

In [2]:
import sys
from pathlib import Path

PAPER_EXPERIMENTS_DIR = Path.cwd()
if not (PAPER_EXPERIMENTS_DIR / "common.py").exists():
    # Fall back to this file's location if the notebook's cwd differs
    # from paper_experiments/ (e.g. launched from the repo root).
    PAPER_EXPERIMENTS_DIR = Path("paper_experiments").resolve()
sys.path.insert(0, str(PAPER_EXPERIMENTS_DIR))

import common
import metadata

print("paper_experiments dir:", PAPER_EXPERIMENTS_DIR)
print("HAS_GRAPHIX:", common.HAS_GRAPHIX, " HAS_JAX:", common.HAS_JAX)


paper_experiments dir: D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\paper_experiments
HAS_GRAPHIX: True  HAS_JAX: True


## R1: Basic CV wire / one-step teleportation.

Verifies the elementary CV-MBQC transport primitive: a single p-homodyne
measurement on a two-mode momentum-squeezed resource teleports a Fourier
transform of the input mode (``photographiq.protocols.wire`` with one shear).

Exact functionality: the unconditional affine channel (S, N, d) is computed
independently by ``photographiq.analysis.gaussian_channel``, which propagates
Wigner variables through the pattern and is a separate code path from the
conditional trajectory simulator (``simulate``/``run_shots``). Agreement
between the two is evidence that finite-squeezing ensemble simulation matches
the analytic finite-resource channel derived in docs/theory.md.

Finite-resource approximation: resource squeezing r is finite (r=1.0 here);
noise diag(0, exp(-2r)) is physical, not a truncation artifact.

In [3]:
"""R1: Basic CV wire / one-step teleportation.

Verifies the elementary CV-MBQC transport primitive: a single p-homodyne
measurement on a two-mode momentum-squeezed resource teleports a Fourier
transform of the input mode (``photographiq.protocols.wire`` with one shear).

Exact functionality: the unconditional affine channel (S, N, d) is computed
independently by ``photographiq.analysis.gaussian_channel``, which propagates
Wigner variables through the pattern and is a separate code path from the
conditional trajectory simulator (``simulate``/``run_shots``). Agreement
between the two is evidence that finite-squeezing ensemble simulation matches
the analytic finite-resource channel derived in docs/theory.md.

Finite-resource approximation: resource squeezing r is finite (r=1.0 here);
noise diag(0, exp(-2r)) is physical, not a truncation artifact.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SHOTS = 4000
SQUEEZING = 1.0
SEED = 20260913

COHERENT_INPUTS = [0.0 + 0.0j, 0.3 + 0.2j, -0.5 + 0.1j, 1.0 + 0.0j, 0.2 - 0.4j, -0.8 - 0.6j]


def main():
    plt = common.setup_style()
    pattern = pg.protocols.wire([0.0], squeezing=SQUEEZING)  # one-step Fourier wire
    channel = pg.gaussian_channel(pattern)
    output_node = pattern.outputs[0]

    rows = []
    for index, alpha in enumerate(COHERENT_INPUTS):
        input_state = pg.GaussianInput.coherent(alpha)
        ideal = channel.apply(input_state.state(0))

        ensemble = {}
        for backend in ("gaussian", "piquasso"):
            shots = pg.run_shots(
                pattern,
                SHOTS,
                backend=backend,
                inputs={0: input_state},
                seed=SEED + 97 * index,
            )
            ensemble[backend] = shots.ensemble_state()

        single = pg.simulate(pattern, backend="gaussian", inputs={0: input_state}, seed=SEED + index)
        q_mean_single, q_var_single = single.state.quadrature(output_node)
        p_mean_single, p_var_single = single.state.quadrature(output_node, np.pi / 2)

        row = {
            "alpha_real": alpha.real,
            "alpha_imag": alpha.imag,
            "ideal_q_mean": ideal.mean[0],
            "ideal_p_mean": ideal.mean[1],
            "ideal_q_var": ideal.covariance[0, 0],
            "ideal_p_var": ideal.covariance[1, 1],
            "single_trajectory_q_mean": q_mean_single,
            "single_trajectory_p_mean": p_mean_single,
            "single_trajectory_q_var": q_var_single,
            "single_trajectory_p_var": p_var_single,
        }
        for backend, state in ensemble.items():
            row[f"{backend}_ensemble_q_mean"] = state.mean[0]
            row[f"{backend}_ensemble_p_mean"] = state.mean[1]
            row[f"{backend}_ensemble_q_var"] = state.covariance[0, 0]
            row[f"{backend}_ensemble_p_var"] = state.covariance[1, 1]
            row[f"{backend}_mean_error"] = common.frobenius_error(state.mean, ideal.mean)
            row[f"{backend}_covariance_error"] = common.frobenius_error(state.covariance, ideal.covariance)
        rows.append(row)

    # Shot-noise-scaled tolerance: SEM of a variance estimate from SHOTS draws
    # scales roughly like sqrt(2/SHOTS) times the variance itself; use a
    # generous multiple to avoid a flaky assertion while still catching bugs.
    tolerance = 0.35
    failures = [
        r for r in rows if r["gaussian_mean_error"] > tolerance or r["gaussian_covariance_error"] > tolerance
    ]
    if failures:
        common.save_csv(failures, "R1_wire_teleportation_FAILURES")
        raise AssertionError(f"{len(failures)} coherent inputs exceeded tolerance {tolerance}; see FAILURES csv")

    common.save_result(
        rows,
        "R1_wire_teleportation",
        extra={
            "protocol": "one-step Fourier wire (k=0)",
            "squeezing": SQUEEZING,
            "shots": SHOTS,
            "seed_base": SEED,
            "channel_matrix": channel.matrix.tolist(),
            "channel_noise": channel.noise.tolist(),
        },
    )

    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))
    x = np.arange(len(rows))
    labels = [f"{r['alpha_real']:.1f}{r['alpha_imag']:+.1f}i" for r in rows]
    width = 0.25
    axes[0].bar(x - width, [r["ideal_q_var"] for r in rows], width, label="Analytic channel", color="#24677b")
    axes[0].bar(x, [r["gaussian_ensemble_q_var"] for r in rows], width, label="Gaussian ensemble", color="#c66d27")
    axes[0].bar(x + width, [r["piquasso_ensemble_q_var"] for r in rows], width, label="Piquasso ensemble", color="#627a36")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    axes[0].set_ylabel("Output q variance")
    axes[0].legend(fontsize=7)

    axes[1].bar(x - width, [r["ideal_p_var"] for r in rows], width, label="Analytic channel", color="#24677b")
    axes[1].bar(x, [r["gaussian_ensemble_p_var"] for r in rows], width, label="Gaussian ensemble", color="#c66d27")
    axes[1].bar(x + width, [r["piquasso_ensemble_p_var"] for r in rows], width, label="Piquasso ensemble", color="#627a36")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    axes[1].set_ylabel("Output p variance")
    axes[1].legend(fontsize=7)
    fig.suptitle(f"One-step Fourier wire, resource squeezing r={SQUEEZING}")
    common.save_figure(fig, "R1_wire_teleportation")
    plt.close(fig)

    common.print_summary(
        "R1 wire teleportation",
        inputs=len(rows),
        max_gaussian_mean_error=max(r["gaussian_mean_error"] for r in rows),
        max_gaussian_covariance_error=max(r["gaussian_covariance_error"] for r in rows),
        max_piquasso_mean_error=max(r["piquasso_mean_error"] for r in rows),
        max_piquasso_covariance_error=max(r["piquasso_covariance_error"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R1 wire teleportation ===
  inputs: 6
  max_gaussian_mean_error: 0.010124365847056171
  max_gaussian_covariance_error: 0.007180999865738169
  max_piquasso_mean_error: 0.01012436584705595
  max_piquasso_covariance_error: 0.007180999865738169


## R2: Identity wire versus finite squeezing.

Sweeps resource squeezing r for the package's actual four-step identity wire
(``photographiq.protocols.identity``, i.e. four Fourier teleportation steps
whose ideal composed map is identity) and measures how the output mean error
and added covariance noise shrink as r grows.

Exact functionality: ``gaussian_channel`` computes the unconditional affine
map (S, N, d) analytically for the fixed-angle pattern; per docs/theory.md an
L-step wire accumulates noise recursively and four k=0 steps implement
identity with noise 2*exp(-2r)*I. That closed form is reproduced here
independently (via ``photographiq.gaussian.wire_channel``, a separate
function from the pattern-level analyzer) as a second oracle.

In [4]:
"""R2: Identity wire versus finite squeezing.

Sweeps resource squeezing r for the package's actual four-step identity wire
(``photographiq.protocols.identity``, i.e. four Fourier teleportation steps
whose ideal composed map is identity) and measures how the output mean error
and added covariance noise shrink as r grows.

Exact functionality: ``gaussian_channel`` computes the unconditional affine
map (S, N, d) analytically for the fixed-angle pattern; per docs/theory.md an
L-step wire accumulates noise recursively and four k=0 steps implement
identity with noise 2*exp(-2r)*I. That closed form is reproduced here
independently (via ``photographiq.gaussian.wire_channel``, a separate
function from the pattern-level analyzer) as a second oracle.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402
from photographiq.gaussian import wire_channel  # noqa: E402

SQUEEZING_VALUES = [0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0]
INPUT_ALPHA = 0.4 + 0.3j


def main():
    plt = common.setup_style()
    rows = []
    for r in SQUEEZING_VALUES:
        pattern = pg.protocols.identity(squeezing=r)
        output_node = pattern.outputs[0]
        channel = pg.gaussian_channel(pattern)

        # Independent closed-form oracle: four k=0 wire steps.
        closed_form_matrix, closed_form_noise = wire_channel([0.0] * 4, r)

        input_state = pg.GaussianInput.coherent(INPUT_ALPHA)
        applied = channel.apply(input_state.state(0))

        matrix_error = common.frobenius_error(channel.matrix, np.eye(2))
        oracle_matrix_error = common.frobenius_error(channel.matrix, closed_form_matrix)
        oracle_noise_error = common.frobenius_error(channel.noise, closed_form_noise)

        mean_error = common.frobenius_error(applied.mean, np.array([2 * INPUT_ALPHA.real, 2 * INPUT_ALPHA.imag]))
        added_noise_trace = float(np.trace(channel.noise))

        single = pg.simulate(pattern, backend="gaussian", inputs={0: input_state}, seed=7)
        _, q_var = single.state.quadrature(output_node)
        _, p_var = single.state.quadrature(output_node, np.pi / 2)

        rows.append(
            {
                "squeezing_r": r,
                "identity_matrix_error": matrix_error,
                "channel_vs_closed_form_matrix_error": oracle_matrix_error,
                "channel_vs_closed_form_noise_error": oracle_noise_error,
                "output_mean_error": mean_error,
                "added_variance_trace": added_noise_trace,
                "predicted_added_variance": 2 * np.exp(-2 * r),
                "output_q_variance": q_var,
                "output_p_variance": p_var,
                "log10_added_variance": common.safe_log10(added_noise_trace),
            }
        )

    tolerance = 1e-9
    failures = [r for r in rows if r["channel_vs_closed_form_matrix_error"] > tolerance
                or r["channel_vs_closed_form_noise_error"] > tolerance or r["identity_matrix_error"] > tolerance]
    if failures:
        common.save_csv(failures, "R2_identity_finite_squeezing_FAILURES")
        raise AssertionError("Identity wire channel disagreed with the closed-form oracle; see FAILURES csv")

    common.save_result(
        rows,
        "R2_identity_finite_squeezing",
        extra={"input_alpha": [INPUT_ALPHA.real, INPUT_ALPHA.imag], "protocol": "four-step identity wire"},
    )

    fig, ax = plt.subplots(figsize=(5.2, 3.6))
    rs = np.asarray(SQUEEZING_VALUES)
    ax.semilogy(rs, [r["added_variance_trace"] for r in rows], "o-", label="Measured Tr(N)", color="#24677b")
    ax.semilogy(rs, [r["predicted_added_variance"] for r in rows], "--", label=r"$2e^{-2r}$ (theory)", color="#c66d27")
    ax.set_xlabel("Resource squeezing r")
    ax.set_ylabel("Added output variance (vacuum units)")
    ax.legend()
    ax.set_title("Four-step identity wire: finite-squeezing noise")
    common.save_figure(fig, "R2_identity_finite_squeezing")
    plt.close(fig)

    common.print_summary(
        "R2 identity vs finite squeezing",
        squeezing_values=SQUEEZING_VALUES,
        max_channel_vs_closed_form_error=max(r["channel_vs_closed_form_matrix_error"] for r in rows),
        max_added_variance=max(r["added_variance_trace"] for r in rows),
        min_added_variance=min(r["added_variance_trace"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R2 identity vs finite squeezing ===
  squeezing_values: [0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0]
  max_channel_vs_closed_form_error: 1.7319121124709866e-16
  max_added_variance: 2.6812801841425573
  min_added_variance: 0.07326255555493671


## R3: Rotation compiler validation.

For each rotation angle theta, compiles ``Circuit(1).rotate(0, theta)`` to a
finite-squeezing MBQC pattern, extracts its exact unconditional Gaussian
channel, and compares the resulting linear map against an independently
written NumPy rotation matrix (not ``photographiq.gaussian.rotation``, which
is the function the compiler itself is built from).

Exact functionality: in the r -> infinity limit the compiled channel matrix
converges to the ideal rotation; finite r contributes physical additive noise
N, not a truncation artifact. A second sweep fixes theta and varies resource
squeezing to show the noise floor separately from the angle dependence.

In [5]:
"""R3: Rotation compiler validation.

For each rotation angle theta, compiles ``Circuit(1).rotate(0, theta)`` to a
finite-squeezing MBQC pattern, extracts its exact unconditional Gaussian
channel, and compares the resulting linear map against an independently
written NumPy rotation matrix (not ``photographiq.gaussian.rotation``, which
is the function the compiler itself is built from).

Exact functionality: in the r -> infinity limit the compiled channel matrix
converges to the ideal rotation; finite r contributes physical additive noise
N, not a truncation artifact. A second sweep fixes theta and varies resource
squeezing to show the noise floor separately from the angle dependence.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

THETAS = np.linspace(-np.pi, np.pi, 25)
SQUEEZING_SWEEP = [0.3, 0.5, 0.8, 1.2, 1.6, 2.0, 2.5]
FIXED_THETA_FOR_SQUEEZING_SWEEP = 0.7
FIXED_SQUEEZING_FOR_THETA_SWEEP = 1.2


def independent_rotation_matrix(theta: float) -> np.ndarray:
    """Independent NumPy rotation matrix; not photographiq.gaussian.rotation."""
    c, s = float(np.cos(theta)), float(np.sin(theta))
    return np.array([[c, -s], [s, c]])


def main():
    plt = common.setup_style()

    theta_rows = []
    for theta in THETAS:
        circuit = pg.Circuit(1).rotate(0, float(theta))
        pattern = circuit.compile(squeezing=FIXED_SQUEEZING_FOR_THETA_SWEEP)
        channel = pg.gaussian_channel(pattern)
        target = independent_rotation_matrix(float(theta))
        map_error = common.frobenius_error(channel.matrix, target)
        noise_norm = float(np.linalg.norm(channel.noise, ord=2))
        theta_rows.append(
            {
                "theta": float(theta),
                "squeezing": FIXED_SQUEEZING_FOR_THETA_SWEEP,
                "map_frobenius_error": map_error,
                "noise_spectral_norm": noise_norm,
                "noise_trace": float(np.trace(channel.noise)),
                "commands": len(pattern.commands),
            }
        )

    squeezing_rows = []
    target = independent_rotation_matrix(FIXED_THETA_FOR_SQUEEZING_SWEEP)
    for r in SQUEEZING_SWEEP:
        circuit = pg.Circuit(1).rotate(0, FIXED_THETA_FOR_SQUEEZING_SWEEP)
        pattern = circuit.compile(squeezing=r)
        channel = pg.gaussian_channel(pattern)
        squeezing_rows.append(
            {
                "theta": FIXED_THETA_FOR_SQUEEZING_SWEEP,
                "squeezing": r,
                "map_frobenius_error": common.frobenius_error(channel.matrix, target),
                "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
                "noise_trace": float(np.trace(channel.noise)),
            }
        )

    map_tolerance = 1e-9  # squeezing does not affect the ideal linear map, only noise
    failures = [r for r in theta_rows + squeezing_rows if r["map_frobenius_error"] > map_tolerance]
    if failures:
        common.save_csv(failures, "R3_rotation_compile_FAILURES")
        raise AssertionError("Compiled rotation map disagreed with the analytic target; see FAILURES csv")

    common.save_result(theta_rows, "R3_rotation_compile_theta_sweep")
    common.save_result(squeezing_rows, "R3_rotation_compile_squeezing_sweep")

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].semilogy(
        [r["theta"] for r in theta_rows], [r["noise_spectral_norm"] for r in theta_rows], "o-", color="#24677b"
    )
    axes[0].set_xlabel(r"Rotation angle $\theta$ (rad)")
    axes[0].set_ylabel("Noise spectral norm ||N||")
    axes[0].set_title(f"Squeezing r={FIXED_SQUEEZING_FOR_THETA_SWEEP} fixed")

    axes[1].semilogy(
        [r["squeezing"] for r in squeezing_rows], [r["noise_spectral_norm"] for r in squeezing_rows], "o-", color="#c66d27"
    )
    axes[1].set_xlabel("Resource squeezing r")
    axes[1].set_ylabel("Noise spectral norm ||N||")
    axes[1].set_title(rf"$\theta$={FIXED_THETA_FOR_SQUEEZING_SWEEP:.2f} fixed")
    fig.suptitle("Rotation gate compilation: finite-resource noise")
    common.save_figure(fig, "R3_rotation_compile")
    plt.close(fig)

    common.print_summary(
        "R3 rotation compiler validation",
        theta_points=len(theta_rows),
        max_map_error=max(r["map_frobenius_error"] for r in theta_rows + squeezing_rows),
        max_noise_norm=max(r["noise_spectral_norm"] for r in theta_rows),
    )


if __name__ == "__main__":
    main()



=== R3 rotation compiler validation ===
  theta_points: 25
  max_map_error: 2.3235379165971276e-14
  max_noise_norm: 5.504649663154229


## R4: Squeezing compiler validation.

Compiles ``Circuit(1).squeeze(0, r_logical)`` for a sweep of logical squeezing
parameters, extracts the exact unconditional Gaussian channel and compares it
to an independently written NumPy squeezing matrix diag(exp(-r), exp(r))
(``photographiq.gaussian.squeezing`` uses the identical formula but is the
function the compiler is built from, so an inline copy is used as the
independent oracle here). Numerical stability is monitored via the condition
number of the compiled map and the growth of the added noise as |r| grows.

In [6]:
"""R4: Squeezing compiler validation.

Compiles ``Circuit(1).squeeze(0, r_logical)`` for a sweep of logical squeezing
parameters, extracts the exact unconditional Gaussian channel and compares it
to an independently written NumPy squeezing matrix diag(exp(-r), exp(r))
(``photographiq.gaussian.squeezing`` uses the identical formula but is the
function the compiler is built from, so an inline copy is used as the
independent oracle here). Numerical stability is monitored via the condition
number of the compiled map and the growth of the added noise as |r| grows.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

LOGICAL_SQUEEZING = [-1.5, -1.0, -0.5, -0.2, 0.0, 0.2, 0.5, 1.0, 1.5]
RESOURCE_SQUEEZING = 1.2


def independent_squeezing_matrix(r: float) -> np.ndarray:
    """Independent NumPy squeezing matrix; not photographiq.gaussian.squeezing."""
    return np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])


def main():
    plt = common.setup_style()
    rows = []
    for r_logical in LOGICAL_SQUEEZING:
        circuit = pg.Circuit(1).squeeze(0, r_logical)
        pattern = circuit.compile(squeezing=RESOURCE_SQUEEZING)
        channel = pg.gaussian_channel(pattern)
        target = independent_squeezing_matrix(r_logical)
        map_error = common.frobenius_error(channel.matrix, target)
        condition_number = float(np.linalg.cond(channel.matrix))
        rows.append(
            {
                "logical_squeezing_r": r_logical,
                "resource_squeezing": RESOURCE_SQUEEZING,
                "map_frobenius_error": map_error,
                "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
                "noise_trace": float(np.trace(channel.noise)),
                "matrix_condition_number": condition_number,
                "matrix_finite": bool(np.isfinite(channel.matrix).all()),
                "noise_finite": bool(np.isfinite(channel.noise).all()),
            }
        )

    tolerance = 1e-9
    failures = [r for r in rows if r["map_frobenius_error"] > tolerance or not r["matrix_finite"] or not r["noise_finite"]]
    if failures:
        common.save_csv(failures, "R4_squeezing_compile_FAILURES")
        raise AssertionError("Compiled squeezing map disagreed with the analytic target; see FAILURES csv")

    common.save_result(rows, "R4_squeezing_compile", extra={"resource_squeezing": RESOURCE_SQUEEZING})

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    rs = [r["logical_squeezing_r"] for r in rows]
    axes[0].semilogy(rs, [r["noise_spectral_norm"] for r in rows], "o-", color="#24677b")
    axes[0].set_xlabel("Logical squeezing r")
    axes[0].set_ylabel("Noise spectral norm ||N||")
    axes[1].semilogy(rs, [r["matrix_condition_number"] for r in rows], "o-", color="#c66d27")
    axes[1].set_xlabel("Logical squeezing r")
    axes[1].set_ylabel("Compiled map condition number")
    fig.suptitle(f"Squeezing gate compilation, resource squeezing r={RESOURCE_SQUEEZING}")
    common.save_figure(fig, "R4_squeezing_compile")
    plt.close(fig)

    common.print_summary(
        "R4 squeezing compiler validation",
        points=len(rows),
        max_map_error=max(r["map_frobenius_error"] for r in rows),
        max_condition_number=max(r["matrix_condition_number"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R4 squeezing compiler validation ===
  points: 9
  max_map_error: 3.847518542748416e-15
  max_condition_number: 20.08553692318767


## R5: Random single-mode symplectic compilation (major validation result).

Draws many random valid single-mode Gaussian transformations
S = R(phi1) S(r) R(phi2) with a deterministic RNG, compiles each through
``Circuit(1).gaussian(0, S).compile(...)``, and compares the compiled affine
map against the exact target S using the independent Gaussian-channel
analyzer. ``decompose_symplectic`` can raise for numerically unstable near-
singular requests (docs/theory.md); those cases are recorded as decomposition
failures rather than silently skipped.

In [7]:
"""R5: Random single-mode symplectic compilation (major validation result).

Draws many random valid single-mode Gaussian transformations
S = R(phi1) S(r) R(phi2) with a deterministic RNG, compiles each through
``Circuit(1).gaussian(0, S).compile(...)``, and compares the compiled affine
map against the exact target S using the independent Gaussian-channel
analyzer. ``decompose_symplectic`` can raise for numerically unstable near-
singular requests (docs/theory.md); those cases are recorded as decomposition
failures rather than silently skipped.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

N_SAMPLES = 400
SEED = 20260913
SQUEEZING = 1.0
R_RANGE = (-1.6, 1.6)


def independent_rotation(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def independent_squeezing(r):
    return np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])


def random_symplectic(generator: np.random.Generator):
    phi1 = generator.uniform(-np.pi, np.pi)
    phi2 = generator.uniform(-np.pi, np.pi)
    r = generator.uniform(*R_RANGE)
    matrix = independent_rotation(phi1) @ independent_squeezing(r) @ independent_rotation(phi2)
    return matrix, phi1, r, phi2


def main():
    plt = common.setup_style()
    generator = common.rng(SEED)
    rows = []
    for i in range(N_SAMPLES):
        target, phi1, r, phi2 = random_symplectic(generator)
        row = {
            "sample": i,
            "phi1": phi1,
            "r": r,
            "phi2": phi2,
            "determinant": float(np.linalg.det(target)),
            "target_matrix": target.tolist(),
        }
        try:
            circuit = pg.Circuit(1).gaussian(0, target)
            pattern, trace = circuit.compile(squeezing=SQUEEZING, return_trace=True)
            channel = pg.gaussian_channel(pattern)
            error = common.frobenius_error(channel.matrix, target)
            step = trace.steps[0]
            row.update(
                {
                    "decomposition_failed": False,
                    "compiled_matrix": channel.matrix.tolist(),
                    "frobenius_error": error,
                    "log10_error": common.safe_log10(error),
                    "resource_nodes": len(step.resource_nodes),
                    "measurements": len(step.measurements),
                    "commands": len(pattern.commands),
                }
            )
        except ValueError as exc:
            row.update(
                {
                    "decomposition_failed": True,
                    "error_message": str(exc),
                    "compiled_matrix": None,
                    "frobenius_error": None,
                    "log10_error": None,
                    "resource_nodes": None,
                    "measurements": None,
                    "commands": None,
                }
            )
        rows.append(row)

    successes = [r for r in rows if not r["decomposition_failed"]]
    failed = [r for r in rows if r["decomposition_failed"]]
    tolerance = 1e-7
    bad = [r for r in successes if r["frobenius_error"] > tolerance]
    if bad:
        common.save_csv(bad, "R5_random_symplectic_compile_FAILURES")
        raise AssertionError(f"{len(bad)} compiled maps exceeded tolerance {tolerance}; see FAILURES csv")

    common.save_result(
        rows,
        "R5_random_symplectic_compile",
        extra={
            "n_samples": N_SAMPLES,
            "seed": SEED,
            "squeezing": SQUEEZING,
            "r_range": list(R_RANGE),
            "n_successes": len(successes),
            "n_decomposition_failures": len(failed),
        },
    )
    if failed:
        common.save_csv(failed, "R5_random_symplectic_compile_decomposition_failures")

    fig, ax = plt.subplots(figsize=(5.4, 3.8))
    errors = [r["log10_error"] for r in successes]
    ax.hist(errors, bins=30, color="#24677b", edgecolor="white")
    ax.set_xlabel(r"$\log_{10}$(Frobenius error)")
    ax.set_ylabel("Count")
    ax.set_title(
        f"Random single-mode symplectic compilation (n={len(successes)}, "
        f"{len(failed)} decomposition failures)"
    )
    common.save_figure(fig, "R5_random_symplectic_compile")
    plt.close(fig)

    common.print_summary(
        "R5 random symplectic compilation",
        n_samples=N_SAMPLES,
        n_successes=len(successes),
        n_decomposition_failures=len(failed),
        max_error=max((r["frobenius_error"] for r in successes), default=None),
        median_log10_error=float(np.median(errors)) if errors else None,
    )


if __name__ == "__main__":
    main()



=== R5 random symplectic compilation ===
  n_samples: 400
  n_successes: 400
  n_decomposition_failures: 0
  max_error: 7.815727244050343e-12
  median_log10_error: -15.139741702151106


## R6: Two-mode CZ compilation.

Sweeps the logical controlled-Z weight in ``Circuit(2).cz(0, 1, weight)``,
compiles each circuit, and compares the exact unconditional two-mode Gaussian
channel to an independently written analytic CZ symplectic matrix
CZ(g): p_i -> p_i + g*q_j (photographiq.gaussian.cz implements the identical
convention; the copy here is written independently as the cross-check
target). For representative coherent inputs, mean/covariance and the
generated cross-mode correlations are also compared.

In [8]:
"""R6: Two-mode CZ compilation.

Sweeps the logical controlled-Z weight in ``Circuit(2).cz(0, 1, weight)``,
compiles each circuit, and compares the exact unconditional two-mode Gaussian
channel to an independently written analytic CZ symplectic matrix
CZ(g): p_i -> p_i + g*q_j (photographiq.gaussian.cz implements the identical
convention; the copy here is written independently as the cross-check
target). For representative coherent inputs, mean/covariance and the
generated cross-mode correlations are also compared.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402
from photographiq.gaussian import wire_channel  # noqa: E402

WEIGHTS = [-2.0, -1.0, -0.5, -0.1, 0.1, 0.5, 1.0, 1.5, 2.0]
SQUEEZING = 1.0
INPUT_ALPHAS = [(0.3 + 0.1j, -0.2 + 0.4j), (0.0, 0.5 + 0.0j)]


def independent_cz(weight: float) -> np.ndarray:
    """Independent CZ(g) symplectic matrix on (q0,p0,q1,p1); p_i -> p_i+g*q_j."""
    matrix = np.eye(4)
    matrix[1, 2] = weight  # p0 += g q1
    matrix[3, 0] = weight  # p1 += g q0
    return matrix


def expected_noise(squeezing: float) -> np.ndarray:
    """Independent noise oracle: logical CZ = instantaneous Entangle followed by
    an identity wire (four k=0 teleportation steps) on each output mode; the
    entangling step itself is a direct physical transform and adds no noise,
    so the total noise is block-diagonal in the per-mode wire noise.
    """
    _, per_mode_noise = wire_channel([0.0] * 4, squeezing)
    noise = np.zeros((4, 4))
    noise[:2, :2] = per_mode_noise
    noise[2:, 2:] = per_mode_noise
    return noise


def main():
    plt = common.setup_style()
    rows = []
    for weight in WEIGHTS:
        circuit = pg.Circuit(2).cz(0, 1, weight)
        pattern = circuit.compile(squeezing=SQUEEZING)
        channel = pg.gaussian_channel(pattern)
        target = independent_cz(weight)
        map_error = common.frobenius_error(channel.matrix, target)
        noise_oracle = expected_noise(SQUEEZING)
        noise_error = common.frobenius_error(channel.noise, noise_oracle)

        state_errors = []
        cross_correlations = []
        for alpha0, alpha1 in INPUT_ALPHAS:
            inputs = pg.GaussianState(
                np.array(
                    [2 * alpha0.real, 2 * alpha0.imag, 2 * alpha1.real, 2 * alpha1.imag]
                ),
                np.eye(4),
                (0, 1),
            )
            actual = channel.apply(inputs)
            ideal_mean = target @ inputs.mean
            ideal_cov = target @ inputs.covariance @ target.T + noise_oracle
            state_errors.append(
                {
                    "mean_error": common.frobenius_error(actual.mean, ideal_mean),
                    "covariance_error": common.frobenius_error(actual.covariance, ideal_cov),
                }
            )
            cross_correlations.append(float(actual.covariance[0, 2]))  # q0-q1 correlation

        rows.append(
            {
                "weight": weight,
                "squeezing": SQUEEZING,
                "map_frobenius_error": map_error,
                "noise_oracle_frobenius_error": noise_error,
                "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
                "noise_trace": float(np.trace(channel.noise)),
                "max_state_mean_error": max(s["mean_error"] for s in state_errors),
                "max_state_covariance_error": max(s["covariance_error"] for s in state_errors),
                "cross_correlations": cross_correlations,
            }
        )

    tolerance = 1e-8
    failures = [r for r in rows if r["map_frobenius_error"] > tolerance or r["noise_oracle_frobenius_error"] > tolerance
                or r["max_state_mean_error"] > tolerance or r["max_state_covariance_error"] > tolerance]
    if failures:
        common.save_csv(failures, "R6_cz_compile_FAILURES")
        raise AssertionError("Compiled CZ map disagreed with the analytic target; see FAILURES csv")

    common.save_result(rows, "R6_cz_compile", extra={"squeezing": SQUEEZING})

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    ws = [r["weight"] for r in rows]
    axes[0].plot(ws, [r["map_frobenius_error"] for r in rows], "o-", color="#24677b")
    axes[0].set_xlabel("CZ weight g")
    axes[0].set_ylabel("Compiled map Frobenius error")
    axes[1].semilogy(ws, [r["noise_spectral_norm"] for r in rows], "o-", color="#c66d27")
    axes[1].set_xlabel("CZ weight g")
    axes[1].set_ylabel("Noise spectral norm ||N||")
    fig.suptitle(f"CZ gate compilation, resource squeezing r={SQUEEZING}")
    common.save_figure(fig, "R6_cz_compile")
    plt.close(fig)

    common.print_summary(
        "R6 CZ compilation",
        weights=len(rows),
        max_map_error=max(r["map_frobenius_error"] for r in rows),
        max_noise_norm=max(r["noise_spectral_norm"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R6 CZ compilation ===
  weights: 9
  max_map_error: 4.2423009548996277e-16
  max_noise_norm: 0.2706705664732254


## R7: Beamsplitter compilation.

Sweeps the beamsplitter mixing angle theta in ``Circuit(2).beamsplitter``,
compiles the (deliberately resource-expensive, per docs/theory.md) SUM-gate
decomposition, and compares the exact unconditional Gaussian channel to an
independently written analytic beamsplitter symplectic matrix (the package's
own convention: c=cos, s=sin on (q0,p0,q1,p1)). Records the MBQC resource
overhead (nodes/measurements/commands), which the docs flag as especially
costly for this gate.

In [9]:
"""R7: Beamsplitter compilation.

Sweeps the beamsplitter mixing angle theta in ``Circuit(2).beamsplitter``,
compiles the (deliberately resource-expensive, per docs/theory.md) SUM-gate
decomposition, and compares the exact unconditional Gaussian channel to an
independently written analytic beamsplitter symplectic matrix (the package's
own convention: c=cos, s=sin on (q0,p0,q1,p1)). Records the MBQC resource
overhead (nodes/measurements/commands), which the docs flag as especially
costly for this gate.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

THETAS = np.linspace(-np.pi + 0.05, np.pi - 0.05, 17)  # avoid the singular +-pi boundary
SQUEEZING = 1.2


def independent_beamsplitter(theta: float) -> np.ndarray:
    """Independent beamsplitter symplectic matrix (package sign convention)."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array(
        [
            [c, 0, -s, 0],
            [0, c, 0, -s],
            [s, 0, c, 0],
            [0, s, 0, c],
        ]
    )


def main():
    plt = common.setup_style()
    rows = []
    for theta in THETAS:
        circuit = pg.Circuit(2).beamsplitter(0, 1, float(theta))
        pattern, trace = circuit.compile(squeezing=SQUEEZING, return_trace=True)
        channel = pg.gaussian_channel(pattern)
        target = independent_beamsplitter(float(theta))
        map_error = common.frobenius_error(channel.matrix, target)
        step = trace.steps[0]
        rows.append(
            {
                "theta": float(theta),
                "squeezing": SQUEEZING,
                "map_frobenius_error": map_error,
                "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
                "noise_trace": float(np.trace(channel.noise)),
                "resource_nodes": len(step.resource_nodes),
                "measurements": len(step.measurements),
                "commands": len(pattern.commands),
            }
        )

    tolerance = 1e-6
    failures = [r for r in rows if r["map_frobenius_error"] > tolerance]
    if failures:
        common.save_csv(failures, "R7_beamsplitter_compile_FAILURES")
        raise AssertionError("Compiled beamsplitter map disagreed with the analytic target; see FAILURES csv")

    common.save_result(rows, "R7_beamsplitter_compile", extra={"squeezing": SQUEEZING})

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    ts = [r["theta"] for r in rows]
    axes[0].semilogy(ts, [r["map_frobenius_error"] for r in rows], "o-", color="#24677b")
    axes[0].set_xlabel(r"Beamsplitter angle $\theta$ (rad)")
    axes[0].set_ylabel("Compiled map Frobenius error")
    axes[1].plot(ts, [r["resource_nodes"] for r in rows], "o-", color="#c66d27", label="Resource nodes")
    axes[1].plot(ts, [r["measurements"] for r in rows], "s--", color="#627a36", label="Measurements")
    axes[1].set_xlabel(r"Beamsplitter angle $\theta$ (rad)")
    axes[1].set_ylabel("MBQC resource count")
    axes[1].legend(fontsize=8)
    fig.suptitle(f"Beamsplitter compilation, resource squeezing r={SQUEEZING}")
    common.save_figure(fig, "R7_beamsplitter_compile")
    plt.close(fig)

    common.print_summary(
        "R7 beamsplitter compilation",
        thetas=len(rows),
        max_map_error=max(r["map_frobenius_error"] for r in rows),
        max_resource_nodes=max(r["resource_nodes"] for r in rows),
        max_commands=max(r["commands"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R7 beamsplitter compilation ===
  thetas: 17
  max_map_error: 3.978430722306971e-15
  max_resource_nodes: 96
  max_commands: 391


## R8: Multi-mode circuit-to-MBQC compilation showcase (main-text figure candidate).

Builds a five-mode Gaussian circuit mixing rotations, a squeezer, a CZ gate,
a beamsplitter and a displacement, compiles it with an explicit provenance
trace, and renders the package's own gate-to-resource visualization
(``photographiq.visualize_compilation``). Also records the compiled
resource-graph statistics (nodes, measurements, entanglement edges,
corrections, commands) that accompany the figure.

Beamsplitter compilation is deliberately resource-expensive (SUM-gate
decomposition via three CZ/rotation sandwiches per docs/theory.md), so the
circuit here is kept small enough to stay under
``visualize_compilation``'s built-in size guard (<=24 gates, <=150 nodes).

In [10]:
"""R8: Multi-mode circuit-to-MBQC compilation showcase (main-text figure candidate).

Builds a five-mode Gaussian circuit mixing rotations, a squeezer, a CZ gate,
a beamsplitter and a displacement, compiles it with an explicit provenance
trace, and renders the package's own gate-to-resource visualization
(``photographiq.visualize_compilation``). Also records the compiled
resource-graph statistics (nodes, measurements, entanglement edges,
corrections, commands) that accompany the figure.

Beamsplitter compilation is deliberately resource-expensive (SUM-gate
decomposition via three CZ/rotation sandwiches per docs/theory.md), so the
circuit here is kept small enough to stay under
``visualize_compilation``'s built-in size guard (<=24 gates, <=150 nodes).
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SQUEEZING = 1.0


def build_circuit() -> pg.Circuit:
    circuit = pg.Circuit(5)
    circuit.rotate(0, 0.4)
    circuit.squeeze(1, 0.3)
    circuit.rotate(2, -0.6)
    circuit.cz(0, 1, 0.8)
    circuit.displace(3, q=0.2, p=-0.1)
    circuit.beamsplitter(2, 3, 0.3)
    circuit.rotate(4, 0.5)
    circuit.cz(3, 4, -0.5)
    return circuit


def main():
    common.setup_style()
    circuit = build_circuit()
    pattern, trace = circuit.compile(squeezing=SQUEEZING, return_trace=True)
    graph = pattern.graph

    entangle_edges = [c for c in pattern.commands if isinstance(c, pg.Entangle)]
    measurements = [c for c in pattern.commands if isinstance(c, pg.Measure)]
    corrections = [c for c in pattern.commands if isinstance(c, pg.Displace)]

    summary = {
        "logical_modes": circuit.modes,
        "source_gates": len(circuit.gates),
        "mbqc_nodes": len(graph.nodes),
        "measurements": len(measurements),
        "entanglement_edges": len(entangle_edges),
        "corrections": len(corrections),
        "commands": len(pattern.commands),
    }
    if summary["mbqc_nodes"] > 150 or summary["source_gates"] > 24:
        raise AssertionError("Showcase circuit exceeds visualize_compilation's size guard")

    steps_rows = [
        {
            "gate_index": s.gate_index,
            "source_gate": s.source_gate,
            "source_modes": s.source_modes,
            "resource_nodes": len(s.resource_nodes),
            "measurements": len(s.measurements),
            "corrections": len(s.corrections),
            "commands": len(s.command_indices),
        }
        for s in trace.steps
    ]

    common.save_json(summary, "R8_multimode_showcase_summary")
    common.save_csv(steps_rows, "R8_multimode_showcase_steps")
    common.write_metadata("R8_multimode_showcase")
    common.save_raw(pattern.to_json(), "R8_multimode_showcase_pattern.json")

    fig = pg.visualize_compilation(
        circuit,
        pattern,
        trace=trace,
        output=str(common.FIG_PDF / "R8_multimode_showcase.pdf"),
    )
    fig.savefig(common.FIG_PNG / "R8_multimode_showcase.png", bbox_inches="tight", dpi=300)
    import matplotlib.pyplot as plt

    plt.close(fig)

    common.print_summary("R8 multimode showcase", **summary)


if __name__ == "__main__":
    main()



=== R8 multimode showcase ===
  logical_modes: 5
  source_gates: 8
  mbqc_nodes: 138
  measurements: 133
  entanglement_edges: 141
  corrections: 134
  commands: 542


## R9: Compilation provenance / trace.

Uses the same five-mode showcase circuit as R8 and exports the full
gate-by-gate compilation provenance (``CompilationTrace``/``CompilationStep``)
as a machine-readable CSV plus a human-readable text summary.

In [11]:
"""R9: Compilation provenance / trace.

Uses the same five-mode showcase circuit as R8 and exports the full
gate-by-gate compilation provenance (``CompilationTrace``/``CompilationStep``)
as a machine-readable CSV plus a human-readable text summary.
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SQUEEZING = 1.0


def build_circuit() -> pg.Circuit:
    """Identical to 08_multimode_showcase.build_circuit (kept standalone per script)."""
    circuit = pg.Circuit(5)
    circuit.rotate(0, 0.4)
    circuit.squeeze(1, 0.3)
    circuit.rotate(2, -0.6)
    circuit.cz(0, 1, 0.8)
    circuit.displace(3, q=0.2, p=-0.1)
    circuit.beamsplitter(2, 3, 0.3)
    circuit.rotate(4, 0.5)
    circuit.cz(3, 4, -0.5)
    return circuit


def main():
    circuit = build_circuit()
    pattern, trace = circuit.compile(squeezing=SQUEEZING, return_trace=True)

    rows = []
    summary_lines = []
    for step in trace.steps:
        rows.append(
            {
                "gate_index": step.gate_index,
                "source_gate": step.source_gate,
                "source_modes": step.source_modes,
                "source_parameters": step.source_parameters,
                "input_nodes": step.input_nodes,
                "output_nodes": step.output_nodes,
                "resource_nodes": step.resource_nodes,
                "n_resource_nodes": len(step.resource_nodes),
                "command_indices": step.command_indices,
                "n_commands": len(step.command_indices),
                "edges": step.edges,
                "n_edges": len(step.edges),
                "measurements": step.measurements,
                "n_measurements": len(step.measurements),
                "corrections": step.corrections,
                "n_corrections": len(step.corrections),
                "synthesis_report": None if step.synthesis_report is None else str(step.synthesis_report),
            }
        )
        summary_lines.append(
            f"Gate {step.gate_index:2d} [{step.source_gate}] modes={step.source_modes} "
            f"params={step.source_parameters} : {len(step.resource_nodes)} resource nodes, "
            f"{len(step.measurements)} measurements, {len(step.corrections)} corrections, "
            f"{len(step.command_indices)} commands "
            f"({step.input_nodes} -> {step.output_nodes})"
        )

    # Consistency check: every command index appears in exactly one step's
    # command_indices list plus the trailing Output command.
    all_indices = sorted(i for step in trace.steps for i in step.command_indices)
    expected = list(range(len(pattern.commands) - 1))  # exclude trailing Output
    if all_indices != expected:
        raise AssertionError("Compilation trace command indices do not partition the command list")

    common.save_csv(rows, "R9_compilation_trace")
    common.save_json({"steps": rows, "pattern_signature": trace.pattern_signature}, "R9_compilation_trace")
    common.write_metadata("R9_compilation_trace")
    common.save_raw("\n".join(summary_lines), "R9_compilation_trace_summary.txt")

    common.print_summary(
        "R9 compilation provenance",
        gates=len(trace.steps),
        total_commands=len(pattern.commands),
        total_resource_nodes=sum(len(s.resource_nodes) for s in trace.steps),
    )


if __name__ == "__main__":
    main()



=== R9 compilation provenance ===
  gates: 8
  total_commands: 542
  total_resource_nodes: 133


## R10: Compiler resource scaling.

Builds circuits with an increasing number of repeated gates (rotation,
squeezing, CZ, beamsplitter) and records how compiled resource-node count,
measurement count, entanglement-edge count, command count and compile time
grow. Beamsplitter compilation is a fixed, angle-independent SUM-gate
decomposition that is markedly more resource-expensive per gate (see R7), so
its gate-count sweep is capped lower to keep total runtime practical.

In [12]:
"""R10: Compiler resource scaling.

Builds circuits with an increasing number of repeated gates (rotation,
squeezing, CZ, beamsplitter) and records how compiled resource-node count,
measurement count, entanglement-edge count, command count and compile time
grow. Beamsplitter compilation is a fixed, angle-independent SUM-gate
decomposition that is markedly more resource-expensive per gate (see R7), so
its gate-count sweep is capped lower to keep total runtime practical.
"""

from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SQUEEZING = 1.0
GATE_COUNTS = [1, 2, 5, 10, 20, 50, 100]
BEAMSPLITTER_COUNTS = [1, 2, 5, 10, 20]


def build(kind: str, n: int) -> pg.Circuit:
    if kind in ("rotation", "squeeze"):
        circuit = pg.Circuit(1)
        for i in range(n):
            if kind == "rotation":
                circuit.rotate(0, 0.1 + 0.01 * i)
            else:
                circuit.squeeze(0, 0.05 + 0.005 * i)
        return circuit
    circuit = pg.Circuit(2)
    for i in range(n):
        if kind == "cz":
            circuit.cz(0, 1, 0.3 + 0.01 * i)
        else:
            circuit.beamsplitter(0, 1, 0.2 + 0.01 * i)
    return circuit


def measure_compile(circuit):
    repeats = 3 if len(circuit.gates) <= 20 else 1
    times = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = circuit.compile(squeezing=SQUEEZING, return_trace=True)
        times.append(time.perf_counter() - start)
    return result, times


def main():
    plt = common.setup_style()
    all_rows = []
    for kind, counts in (
        ("rotation", GATE_COUNTS),
        ("squeeze", GATE_COUNTS),
        ("cz", GATE_COUNTS),
        ("beamsplitter", BEAMSPLITTER_COUNTS),
    ):
        for n in counts:
            circuit = build(kind, n)
            (pattern, trace), times = measure_compile(circuit)
            entangle_edges = sum(1 for c in pattern.commands if isinstance(c, pg.Entangle))
            measurements = sum(1 for c in pattern.commands if isinstance(c, pg.Measure))
            row = {
                "gate_kind": kind,
                "gate_count": n,
                "resource_nodes": len(pattern.graph.nodes) - circuit.modes,
                "measurements": measurements,
                "edges": entangle_edges,
                "commands": len(pattern.commands),
                "compile_time_median_seconds": float(np.median(times)) if len(times) > 1 else times[0],
                "compile_time_seconds_raw": times,
            }
            all_rows.append(row)
            print(f"  {kind} n={n}: {row['resource_nodes']} nodes, {row['compile_time_median_seconds']:.4f}s", flush=True)

    common.save_result(all_rows, "R10_compiler_scaling")

    fig, axes = plt.subplots(2, 2, figsize=(9.5, 7))
    metrics = [
        ("resource_nodes", "Resource nodes"),
        ("measurements", "Measurements"),
        ("commands", "Commands"),
        ("compile_time_median_seconds", "Compile time (s)"),
    ]
    for ax, (key, label) in zip(axes.flat, metrics, strict=True):
        for kind in ("rotation", "squeeze", "cz", "beamsplitter"):
            rows = [r for r in all_rows if r["gate_kind"] == kind]
            ax.plot([r["gate_count"] for r in rows], [r[key] for r in rows], "o-", label=kind, markersize=4)
        ax.set_xlabel("Number of gates")
        ax.set_ylabel(label)
        ax.set_xscale("log")
        ax.set_yscale("log")
    axes[0, 0].legend(fontsize=7)
    fig.suptitle("Compiler resource scaling")
    common.save_figure(fig, "R10_compiler_scaling")
    plt.close(fig)

    common.print_summary("R10 compiler resource scaling", rows=len(all_rows))


if __name__ == "__main__":
    main()


  rotation n=1: 4 nodes, 0.0041s


  rotation n=2: 8 nodes, 0.0063s


  rotation n=5: 20 nodes, 0.0147s


  rotation n=10: 40 nodes, 0.0277s

  rotation n=20: 80 nodes, 0.0543s


...

  beamsplitter n=2: 192 nodes, 0.1446s


  beamsplitter n=5: 480 nodes, 0.3269s


  beamsplitter n=10: 960 nodes, 0.8130s


  beamsplitter n=20: 1920 nodes, 1.5175s



=== R10 compiler resource scaling ===
  rows: 26


## R11: Adaptive feed-forward.

Builds a three-node line resource where the *measurement angle* of the
second homodyne detector depends on the first measurement's outcome
(``pg.Outcome("m0")``), in addition to the standard outcome-dependent
displacement corrections. This is genuinely adaptive: the affine
``gaussian_channel`` analyzer explicitly rejects outcome-dependent gate
angles (docs/theory.md), which is used here as a structural check that the
pattern is not secretly representable as a fixed-angle Gaussian channel.
Reproducibility is achieved by fixing the trajectory seed (Gaussian homodyne
has no explicit-outcome postselection API, unlike the Fock backends).

In [13]:
"""R11: Adaptive feed-forward.

Builds a three-node line resource where the *measurement angle* of the
second homodyne detector depends on the first measurement's outcome
(``pg.Outcome("m0")``), in addition to the standard outcome-dependent
displacement corrections. This is genuinely adaptive: the affine
``gaussian_channel`` analyzer explicitly rejects outcome-dependent gate
angles (docs/theory.md), which is used here as a structural check that the
pattern is not secretly representable as a fixed-angle Gaussian channel.
Reproducibility is achieved by fixing the trajectory seed (Gaussian homodyne
has no explicit-outcome postselection API, unlike the Fock backends).
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SQUEEZING = 1.0
THETA_PARAMETER = 0.3
SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]


def build_pattern():
    graph = pg.CVGraph.line(3, squeezing=SQUEEZING, inputs=(0,))
    pattern = pg.Pattern(graph)
    pattern.measure(0, pg.Homodyne.q(), key="m0")
    pattern.displace(1, q=-pg.Outcome("m0"))
    # Adaptive: the homodyne angle itself is a function of the earlier outcome.
    pattern.measure(1, pg.Homodyne(angle=0.05 * pg.Outcome("m0") + pg.Parameter("theta")), key="m1")
    pattern.displace(2, q=-pg.Outcome("m1"))
    pattern.append(pg.Output((2,)))
    return pattern.validate()


def main():
    common.setup_style()
    pattern = build_pattern()

    dependency_graph = pattern.dependencies()
    m0_index = next(i for i, c in enumerate(pattern.commands) if isinstance(c, pg.Measure) and c.result_key == "m0")
    m1_index = next(i for i, c in enumerate(pattern.commands) if isinstance(c, pg.Measure) and c.result_key == "m1")
    has_dependency_edge = dependency_graph.has_edge(m0_index, m1_index)
    if not has_dependency_edge:
        raise AssertionError("Expected a causal dependency edge from m0's measurement to m1's")

    # Structural proof of genuine adaptivity: the affine analyzer must reject
    # this pattern because the second homodyne angle depends on an outcome.
    rejected_by_affine_analyzer = False
    try:
        pg.gaussian_channel(pattern)
    except NotImplementedError:
        rejected_by_affine_analyzer = True
    if not rejected_by_affine_analyzer:
        raise AssertionError("Expected the affine Gaussian-channel analyzer to reject this adaptive pattern")

    rows = []
    input_state = pg.GaussianInput.coherent(0.3 + 0.1j)
    for seed in SEEDS:
        result = pg.simulate(
            pattern, backend="gaussian", inputs={0: input_state}, parameters={"theta": THETA_PARAMETER}, seed=seed
        )
        expected_angle = 0.05 * result.records["m0"] + THETA_PARAMETER
        rows.append(
            {
                "seed": seed,
                "m0_outcome": result.records["m0"],
                "m1_outcome": result.records["m1"],
                "adaptive_angle_used": expected_angle,
                "output_q_mean": result.state.quadrature(2)[0],
                "output_p_mean": result.state.quadrature(2, expected_angle)[0],
            }
        )

    common.save_result(
        rows,
        "R11_adaptive_feedforward",
        extra={
            "pattern_json": pattern.to_json(),
            "dependency_edges": sorted(dependency_graph.edges()),
            "m0_command_index": m0_index,
            "m1_command_index": m1_index,
            "affine_analyzer_rejected_pattern": rejected_by_affine_analyzer,
            "theta_parameter": THETA_PARAMETER,
        },
    )
    common.save_raw(pattern.to_json(), "R11_adaptive_feedforward_pattern.json")

    from photographiq.visualization import draw_dependencies, draw_pattern

    ax = draw_pattern(pattern)
    common.save_figure(ax.figure, "R11_adaptive_feedforward_pattern")
    import matplotlib.pyplot as plt

    plt.close(ax.figure)

    ax = draw_dependencies(pattern)
    common.save_figure(ax.figure, "R11_adaptive_feedforward_dependencies")
    plt.close(ax.figure)

    common.print_summary(
        "R11 adaptive feed-forward",
        seeds=len(rows),
        dependency_edge_m0_to_m1=has_dependency_edge,
        affine_analyzer_rejected_pattern=rejected_by_affine_analyzer,
    )


if __name__ == "__main__":
    main()



=== R11 adaptive feed-forward ===
  seeds: 8
  dependency_edge_m0_to_m1: True
  affine_analyzer_rejected_pattern: True


## R12: Dependency DAG / causal schedule.

Builds a four-node adaptive pattern with multiple classical dependencies
(two homodyne outcomes feeding three downstream corrections/measurements),
extracts ``Pattern.dependencies()`` (a command DAG) and
``Pattern.schedule()`` (a topological command ordering), and checks that
every dependency edge points strictly backward-to-forward in that schedule.

In [14]:
"""R12: Dependency DAG / causal schedule.

Builds a four-node adaptive pattern with multiple classical dependencies
(two homodyne outcomes feeding three downstream corrections/measurements),
extracts ``Pattern.dependencies()`` (a command DAG) and
``Pattern.schedule()`` (a topological command ordering), and checks that
every dependency edge points strictly backward-to-forward in that schedule.
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SQUEEZING = 1.0


def build_pattern():
    graph = pg.CVGraph.line(4, squeezing=SQUEEZING, inputs=(0,))
    pattern = pg.Pattern(graph)
    pattern.measure(0, pg.Homodyne.q(), key="m0")
    pattern.displace(1, q=-pg.Outcome("m0"))
    pattern.measure(1, pg.Homodyne(angle=0.1 * pg.Outcome("m0")), key="m1")
    # Node 2's correction depends on BOTH earlier outcomes.
    pattern.displace(2, q=-pg.Outcome("m1"), p=0.2 * pg.Outcome("m0"))
    pattern.measure(2, pg.Homodyne.p(), key="m2")
    pattern.displace(3, q=-pg.Outcome("m2"), p=-pg.Outcome("m0") + pg.Outcome("m1"))
    pattern.append(pg.Output((3,)))
    return pattern.validate()


def main():
    common.setup_style()
    pattern = build_pattern()
    dag = pattern.dependencies()
    schedule = pattern.schedule()

    position = {command_index: position for position, command_index in enumerate(schedule)}
    edges = sorted(dag.edges())
    violations = [(u, v) for u, v in edges if position[u] >= position[v]]

    rows = [
        {
            "command_index": i,
            "command_type": type(c).__name__,
            "schedule_position": position[i],
            "dependencies": sorted(dag.predecessors(i)),
        }
        for i, c in enumerate(pattern.commands)
    ]

    common.save_result(
        rows,
        "R12_dependency_dag",
        extra={
            "dependency_edges": edges,
            "topological_schedule": list(schedule),
            "backward_violations": violations,
            "pattern_json": pattern.to_json(),
        },
    )

    if violations:
        common.save_json(violations, "R12_dependency_dag_VIOLATIONS")
        raise AssertionError(f"{len(violations)} dependency edges violate the topological schedule")

    from photographiq.visualization import draw_dependencies

    ax = draw_dependencies(pattern)
    common.save_figure(ax.figure, "R12_dependency_dag")
    import matplotlib.pyplot as plt

    plt.close(ax.figure)

    common.print_summary(
        "R12 dependency DAG / causal schedule",
        commands=len(pattern.commands),
        dependency_edges=len(edges),
        backward_violations=len(violations),
    )


if __name__ == "__main__":
    main()



=== R12 dependency DAG / causal schedule ===
  commands: 13
  dependency_edges: 19
  backward_violations: 0


## R13: Invalid causality / software-safety test (appendix result).

Constructs several intentionally invalid patterns and confirms that
``Pattern.validate()`` (or construction itself) rejects each one with the
expected, actual validator error — not a fabricated exception. This audits
the software's causal/structural safety checks rather than its physics.

In [15]:
"""R13: Invalid causality / software-safety test (appendix result).

Constructs several intentionally invalid patterns and confirms that
``Pattern.validate()`` (or construction itself) rejects each one with the
expected, actual validator error — not a fabricated exception. This audits
the software's causal/structural safety checks rather than its physics.
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402

import photographiq as pg  # noqa: E402
from photographiq.commands import Displace, Entangle, Measure, Output, Prepare  # noqa: E402
from photographiq.expressions import Outcome  # noqa: E402
from photographiq.measurements import Homodyne  # noqa: E402


def case_duplicate_inputs():
    return pg.Pattern(inputs=(0, 0)).validate()


def case_duplicate_resource_node():
    graph = pg.CVGraph.line(2, inputs=(0,))
    pattern = pg.Pattern(graph)
    pattern.append(Prepare(1, 1.0))  # node 1 already prepared by the graph
    return pattern.validate()


def case_self_cz():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Entangle(0, 0, 1.0))
    return pattern.validate()


def case_output_not_final():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Output((0,)))
    pattern.append(Displace(0, q=1.0))
    return pattern.validate()


def case_duplicate_outputs():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Output((0, 0)))
    return pattern.validate()


def case_missing_or_measured_node():
    pattern = pg.Pattern(inputs=(0,))
    pattern.measure(0, Homodyne.q(), key="m0")
    pattern.append(Displace(0, q=1.0))  # 0 was destructively measured
    return pattern.validate()


def case_future_outcome_dependency():
    pattern = pg.Pattern(inputs=(0, 1))
    pattern.append(Displace(0, q=Outcome("future")))  # "future" not yet produced
    pattern.append(Measure(1, Homodyne.q(), "future"))
    return pattern.validate()


def case_cyclic_dependency():
    pattern = pg.Pattern(inputs=(0, 1))
    pattern.append(Measure(0, Homodyne(angle=Outcome("m1")), "m0"))
    pattern.append(Measure(1, Homodyne(angle=Outcome("m0")), "m1"))
    return pattern.validate()


def case_undefined_classical_key():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Displace(0, q=Outcome("nonexistent")))
    pattern.append(Output((0,)))
    return pattern.validate()


def case_duplicate_classical_key():
    pattern = pg.Pattern(inputs=(0, 1))
    pattern.append(Measure(0, Homodyne.q(), "m"))
    pattern.append(Measure(1, Homodyne.q(), "m"))  # same key twice
    return pattern.validate()


def case_unsupported_measurement_description():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Measure(0, "not a measurement", "m"))
    return pattern.validate()


def case_ideal_graph_pattern():
    graph = pg.CVGraph.line(2, inputs=(0,), ideal=True)
    return pg.Pattern(graph)


CASES = [
    ("duplicate_inputs", case_duplicate_inputs),
    ("duplicate_resource_node", case_duplicate_resource_node),
    ("self_cz", case_self_cz),
    ("output_not_final", case_output_not_final),
    ("duplicate_outputs", case_duplicate_outputs),
    ("missing_or_measured_node", case_missing_or_measured_node),
    ("future_outcome_dependency", case_future_outcome_dependency),
    ("cyclic_dependency", case_cyclic_dependency),
    ("undefined_classical_key", case_undefined_classical_key),
    ("duplicate_classical_key", case_duplicate_classical_key),
    ("unsupported_measurement_description", case_unsupported_measurement_description),
    ("ideal_graph_pattern", case_ideal_graph_pattern),
]


def main():
    rows = []
    unexpected = []
    for name, builder in CASES:
        try:
            builder()
            rows.append({"case": name, "raised": False, "exception_type": None, "message": None})
            unexpected.append(name)
        except (ValueError, NotImplementedError, TypeError) as exc:
            rows.append(
                {"case": name, "raised": True, "exception_type": type(exc).__name__, "message": str(exc)}
            )

    common.save_result(rows, "R13_invalid_causality")

    if unexpected:
        raise AssertionError(f"Expected these invalid patterns to be rejected but they were not: {unexpected}")

    common.print_summary(
        "R13 invalid causality / software safety",
        cases=len(rows),
        all_rejected=len(unexpected) == 0,
    )


if __name__ == "__main__":
    main()



=== R13 invalid causality / software safety ===
  cases: 12
  all_rejected: True


## R14: CV-flow validation.

Uses the current, supplied-total-order CV-flow certificate
(``photographiq.cvflow.certify_cv_flow``/``flow_pattern``), which implements
the real-linear condition of Booth and Markham (Quantum 7, 1146, 2023) for
one given measurement order. It does NOT search over orders, so this script
only demonstrates certification/rejection for two concrete orders on the
same three-node line resource -- not a general optimal-flow finder.

Valid example: measuring the line graph 0->1->2 (input 0, output 2) in the
natural order (0, 1) yields a zero-residual certificate. Invalid example:
measuring the same graph in the reversed order (1, 0) has no exact real-linear
correction (node 1 alone cannot supply the needed influence on the surviving
output once node 0, its only other neighbor, is excluded as an input), so
certification returns None with reported residuals in the diagnostic case.

In [16]:
"""R14: CV-flow validation.

Uses the current, supplied-total-order CV-flow certificate
(``photographiq.cvflow.certify_cv_flow``/``flow_pattern``), which implements
the real-linear condition of Booth and Markham (Quantum 7, 1146, 2023) for
one given measurement order. It does NOT search over orders, so this script
only demonstrates certification/rejection for two concrete orders on the
same three-node line resource -- not a general optimal-flow finder.

Valid example: measuring the line graph 0->1->2 (input 0, output 2) in the
natural order (0, 1) yields a zero-residual certificate. Invalid example:
measuring the same graph in the reversed order (1, 0) has no exact real-linear
correction (node 1 alone cannot supply the needed influence on the surviving
output once node 0, its only other neighbor, is excluded as an input), so
certification returns None with reported residuals in the diagnostic case.
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402

import photographiq as pg  # noqa: E402
from photographiq.cvflow import certify_cv_flow, flow_pattern  # noqa: E402

SQUEEZING = 1.0


def main():
    graph = pg.CVGraph.line(3, squeezing=SQUEEZING, inputs=(0,), outputs=(2,))

    valid_order = (0, 1)
    valid_certificate = certify_cv_flow(graph, valid_order)
    if valid_certificate is None:
        raise AssertionError("Expected order (0,1) to admit a CV-flow certificate")

    invalid_order = (1, 0)
    invalid_certificate = certify_cv_flow(graph, invalid_order)
    if invalid_certificate is not None:
        raise AssertionError("Expected order (1,0) to have no CV-flow certificate")

    # Diagnose *why* the invalid order fails: recompute the same linear system
    # certify_cv_flow solves internally, to report an explicit residual.
    import numpy as np

    a = graph.adjacency()
    indices = {n: i for i, n in enumerate(graph.nodes)}
    past = invalid_order[:2]
    future = tuple(n for n in graph.nodes if n not in past and n not in graph.inputs)
    matrix = a[np.ix_([indices[n] for n in past], [indices[n] for n in future])]
    target = np.zeros(len(past))
    target[-1] = 1
    solution, *_ = np.linalg.lstsq(matrix, target, rcond=None)
    residual = float(np.linalg.norm(matrix @ solution - target, ord=np.inf))

    pattern = flow_pattern(graph, valid_order)
    input_state = pg.GaussianInput.coherent(0.3 + 0.1j)
    result = pg.simulate(pattern, backend="gaussian", inputs={0: input_state}, seed=1)
    ideal_channel = pg.gaussian_channel(pattern)
    ideal = ideal_channel.apply(input_state.state(0))

    summary = {
        "graph": "line(3), inputs=(0,), outputs=(2,)",
        "valid_order": valid_order,
        "valid_certificate_corrections": {str(k): v for k, v in valid_certificate.corrections.items()},
        "valid_certificate_residuals": valid_certificate.residuals,
        "invalid_order": invalid_order,
        "invalid_certificate_result": None,
        "invalid_order_diagnostic_residual": residual,
        "invalid_order_diagnostic_solution": solution.tolist(),
        "flow_pattern_commands": len(pattern.commands),
        "flow_pattern_output_mean": result.state.mean.tolist(),
        "flow_pattern_output_covariance": result.state.covariance.tolist(),
        "affine_channel_mean": ideal.mean.tolist(),
        "affine_channel_covariance": ideal.covariance.tolist(),
    }
    common.save_json(summary, "R14_cvflow")
    common.write_metadata("R14_cvflow")
    common.save_raw(pattern.to_json(), "R14_cvflow_pattern.json")

    common.print_summary(
        "R14 CV-flow validation",
        valid_certificate_found=True,
        invalid_certificate_found=False,
        invalid_order_residual=residual,
    )


if __name__ == "__main__":
    main()



=== R14 CV-flow validation ===
  valid_certificate_found: True
  invalid_certificate_found: False
  invalid_order_residual: 1.0


## R15: Raw Piquasso Gaussian reference (independent, not through PhotoGraphiQ).

For displacement, rotation, squeezing, beamsplitter, CZ (native
``pq.ControlledZ``, the correct Gaussian-regime representation; Piquasso's
``GaussianTransform`` is instead used by PhotoGraphiQ's *Fock* backend) and a
short compound circuit, this script:

1. Builds a raw ``piquasso`` program directly with the Piquasso public API
   (no PhotoGraphiQ wrapper call in the numerical path) starting from vacuum,
   preparing the same coherent input and applying the same native gate.
2. Runs the identical physical operation through PhotoGraphiQ's
   ``backend="piquasso"`` path.
3. Compares mean-vector norm error, covariance Frobenius error and mean
   photon number (via the documented analytic Gaussian formula, applied
   identically to both sides) across many random parameter draws.

In [17]:
"""R15: Raw Piquasso Gaussian reference (independent, not through PhotoGraphiQ).

For displacement, rotation, squeezing, beamsplitter, CZ (native
``pq.ControlledZ``, the correct Gaussian-regime representation; Piquasso's
``GaussianTransform`` is instead used by PhotoGraphiQ's *Fock* backend) and a
short compound circuit, this script:

1. Builds a raw ``piquasso`` program directly with the Piquasso public API
   (no PhotoGraphiQ wrapper call in the numerical path) starting from vacuum,
   preparing the same coherent input and applying the same native gate.
2. Runs the identical physical operation through PhotoGraphiQ's
   ``backend="piquasso"`` path.
3. Compares mean-vector norm error, covariance Frobenius error and mean
   photon number (via the documented analytic Gaussian formula, applied
   identically to both sides) across many random parameter draws.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import piquasso as pq

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SEED = 20260913
N_RANDOM = 20
HBAR = 2.0


def mean_photon_number(mean: np.ndarray, covariance: np.ndarray) -> float:
    """Independent analytic single-mode formula: (Tr(V)+mu.mu-2)/4 (docs/theory.md)."""
    return float((np.trace(covariance) + mean @ mean - 2) / 4)


def raw_single_mode(alpha: complex, gate_builder):
    """Prepare a coherent state and apply one native gate via a fresh raw program."""
    with pq.Program() as program:
        pq.Q(0) | pq.Vacuum()
        pq.Q(0) | pq.Displacement(r=abs(alpha), phi=np.angle(alpha))
        gate_builder(program)
    state = pq.GaussianSimulator(d=1, config=pq.Config(hbar=HBAR)).execute(program).state
    return np.array(state.xpxp_mean_vector), np.array(state.xpxp_covariance_matrix) / 2


def raw_two_mode(alpha0: complex, alpha1: complex, gate_builder):
    with pq.Program() as program:
        pq.Q(0, 1) | pq.Vacuum()
        pq.Q(0) | pq.Displacement(r=abs(alpha0), phi=np.angle(alpha0))
        pq.Q(1) | pq.Displacement(r=abs(alpha1), phi=np.angle(alpha1))
        gate_builder(program)
    state = pq.GaussianSimulator(d=2, config=pq.Config(hbar=HBAR)).execute(program).state
    return np.array(state.xpxp_mean_vector), np.array(state.xpxp_covariance_matrix) / 2


def pg_single_mode(alpha: complex, command):
    pattern = pg.Pattern(inputs=(0,)).append(command).append(pg.Output((0,)))
    result = pg.simulate(pattern, backend="piquasso", inputs={0: pg.GaussianInput.coherent(alpha)}, seed=0)
    return result.state.mean, result.state.covariance


def pg_two_mode(alpha0: complex, alpha1: complex, commands):
    pattern = pg.Pattern(inputs=(0, 1))
    pattern.extend(commands)
    pattern.append(pg.Output((0, 1)))
    inputs = {0: pg.GaussianInput.coherent(alpha0), 1: pg.GaussianInput.coherent(alpha1)}
    result = pg.simulate(pattern, backend="piquasso", inputs=inputs, seed=0)
    return result.state.mean, result.state.covariance


def compare(name, alpha_desc, raw_mean, raw_cov, pg_mean, pg_cov):
    mean_error = common.frobenius_error(pg_mean, raw_mean)
    cov_error = common.frobenius_error(pg_cov, raw_cov)
    raw_n = mean_photon_number(raw_mean[:2], raw_cov[:2, :2]) if len(raw_mean) == 2 else None
    pg_n = mean_photon_number(pg_mean[:2], pg_cov[:2, :2]) if len(pg_mean) == 2 else None
    return {
        "gate": name,
        "params": alpha_desc,
        "mean_vector_norm_error": mean_error,
        "covariance_frobenius_error": cov_error,
        "photon_number_error": None if raw_n is None else abs(raw_n - pg_n),
    }


def main():
    plt = common.setup_style()
    generator = common.rng(SEED)
    rows = []

    for _ in range(N_RANDOM):
        alpha = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        q, p = generator.uniform(-1, 1), generator.uniform(-1, 1)
        shift = complex(q, p) / 2

        def gate_builder(program, shift=shift):
            pq.Q(0) | pq.Displacement(r=abs(shift), phi=np.angle(shift))

        raw_mean, raw_cov = raw_single_mode(alpha, gate_builder)
        pg_mean, pg_cov = pg_single_mode(alpha, pg.Displace(0, q=q, p=p))
        rows.append(compare("displacement", {"alpha": [alpha.real, alpha.imag], "q": q, "p": p}, raw_mean, raw_cov, pg_mean, pg_cov))

    for _ in range(N_RANDOM):
        alpha = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        angle = generator.uniform(-np.pi, np.pi)
        raw_mean, raw_cov = raw_single_mode(alpha, lambda program, angle=angle: (pq.Q(0) | pq.Phaseshifter(phi=angle)))
        pg_mean, pg_cov = pg_single_mode(alpha, pg.Rotate(0, angle))
        rows.append(compare("rotation", {"alpha": [alpha.real, alpha.imag], "angle": angle}, raw_mean, raw_cov, pg_mean, pg_cov))

    for _ in range(N_RANDOM):
        alpha = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        r = generator.uniform(-1.2, 1.2)
        raw_mean, raw_cov = raw_single_mode(alpha, lambda program, r=r: (pq.Q(0) | pq.Squeezing(r=r)))
        pg_mean, pg_cov = pg_single_mode(alpha, pg.Squeeze(0, r))
        rows.append(compare("squeezing", {"alpha": [alpha.real, alpha.imag], "r": r}, raw_mean, raw_cov, pg_mean, pg_cov))

    for _ in range(N_RANDOM):
        alpha0 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        alpha1 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        theta = generator.uniform(-np.pi / 2, np.pi / 2)
        raw_mean, raw_cov = raw_two_mode(alpha0, alpha1, lambda program, theta=theta: (pq.Q(0, 1) | pq.Beamsplitter(theta=theta, phi=0.0)))
        pg_mean, pg_cov = pg_two_mode(alpha0, alpha1, [pg.BeamSplitter(0, 1, theta)])
        rows.append(
            compare(
                "beamsplitter",
                {"alpha0": [alpha0.real, alpha0.imag], "alpha1": [alpha1.real, alpha1.imag], "theta": theta},
                raw_mean, raw_cov, pg_mean, pg_cov,
            )
        )

    for _ in range(N_RANDOM):
        alpha0 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        alpha1 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        weight = generator.uniform(-2, 2)
        raw_mean, raw_cov = raw_two_mode(alpha0, alpha1, lambda program, weight=weight: (pq.Q(0, 1) | pq.ControlledZ(s=weight)))
        pg_mean, pg_cov = pg_two_mode(alpha0, alpha1, [pg.Entangle(0, 1, weight)])
        rows.append(
            compare(
                "cz",
                {"alpha0": [alpha0.real, alpha0.imag], "alpha1": [alpha1.real, alpha1.imag], "weight": weight},
                raw_mean, raw_cov, pg_mean, pg_cov,
            )
        )

    for _ in range(N_RANDOM):
        alpha0 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        alpha1 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
        r, angle, weight, q, p = (
            generator.uniform(-0.6, 0.6),
            generator.uniform(-np.pi, np.pi),
            generator.uniform(-1, 1),
            generator.uniform(-1, 1),
            generator.uniform(-1, 1),
        )

        def raw_compound(program, r=r, angle=angle, weight=weight, q=q, p=p):
            shift = complex(q, p) / 2
            pq.Q(0) | pq.Squeezing(r=r)
            pq.Q(0) | pq.Phaseshifter(phi=angle)
            pq.Q(0, 1) | pq.ControlledZ(s=weight)
            pq.Q(1) | pq.Displacement(r=abs(shift), phi=np.angle(shift))

        raw_mean, raw_cov = raw_two_mode(alpha0, alpha1, raw_compound)
        pg_mean, pg_cov = pg_two_mode(
            alpha0, alpha1, [pg.Squeeze(0, r), pg.Rotate(0, angle), pg.Entangle(0, 1, weight), pg.Displace(1, q=q, p=p)]
        )
        rows.append(
            compare(
                "compound(squeeze+rotate+cz+displace)",
                {"r": r, "angle": angle, "weight": weight, "q": q, "p": p},
                raw_mean, raw_cov, pg_mean, pg_cov,
            )
        )

    tolerance = 1e-9
    failures = [r for r in rows if r["mean_vector_norm_error"] > tolerance or r["covariance_frobenius_error"] > tolerance]
    if failures:
        common.save_csv(failures, "R15_raw_piquasso_gaussian_FAILURES")
        raise AssertionError(f"{len(failures)} cases disagreed with raw Piquasso beyond tolerance; see FAILURES csv")

    common.save_result(rows, "R15_raw_piquasso_gaussian", extra={"n_random_per_gate": N_RANDOM, "seed": SEED})

    gates = sorted(set(r["gate"] for r in rows))
    summary_table = [
        {
            "gate": gate,
            "n": len([r for r in rows if r["gate"] == gate]),
            "max_mean_error": max(r["mean_vector_norm_error"] for r in rows if r["gate"] == gate),
            "max_covariance_error": max(r["covariance_frobenius_error"] for r in rows if r["gate"] == gate),
        }
        for gate in gates
    ]
    common.save_csv(summary_table, "R15_raw_piquasso_gaussian_summary_table")

    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    x = np.arange(len(gates))
    ax.bar(x, [common.safe_log10(t["max_covariance_error"]) for t in summary_table], color="#24677b")
    ax.set_xticks(x)
    ax.set_xticklabels(gates, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel(r"$\log_{10}$(max covariance error)")
    ax.set_title("PhotoGraphiQ (Piquasso backend) vs. raw Piquasso")
    common.save_figure(fig, "R15_raw_piquasso_gaussian")
    plt.close(fig)

    common.print_summary(
        "R15 raw Piquasso Gaussian reference",
        total_cases=len(rows),
        max_mean_error=max(r["mean_vector_norm_error"] for r in rows),
        max_covariance_error=max(r["covariance_frobenius_error"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R15 raw Piquasso Gaussian reference ===
  total_cases: 120
  max_mean_error: 2.7303643765501543e-15
  max_covariance_error: 6.329245045284193e-16


## R16: Independent NumPy analytic Gaussian reference.

Builds small symplectic target matrices directly with NumPy (not
``photographiq.gaussian``), applies them analytically to a coherent-state
input, and compares the result against three execution paths: PhotoGraphiQ's
NumPy Gaussian backend, PhotoGraphiQ's Piquasso Gaussian backend, and a raw
Piquasso program built independently of PhotoGraphiQ's backend adapters.

In [18]:
"""R16: Independent NumPy analytic Gaussian reference.

Builds small symplectic target matrices directly with NumPy (not
``photographiq.gaussian``), applies them analytically to a coherent-state
input, and compares the result against three execution paths: PhotoGraphiQ's
NumPy Gaussian backend, PhotoGraphiQ's Piquasso Gaussian backend, and a raw
Piquasso program built independently of PhotoGraphiQ's backend adapters.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import piquasso as pq

import common  # noqa: E402

import photographiq as pg  # noqa: E402

HBAR = 2.0


# --------------------------------------------------------------------------- #
# Independent NumPy target matrices (not photographiq.gaussian)
# --------------------------------------------------------------------------- #


def target_rotation(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def target_squeezing(r):
    return np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])


def target_displacement_delta(q, p):
    return np.array([q, p])


def target_beamsplitter(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, -s, 0], [0, c, 0, -s], [s, 0, c, 0], [0, s, 0, c]])


def target_cz(weight):
    matrix = np.eye(4)
    matrix[1, 2] = weight
    matrix[3, 0] = weight
    return matrix


# --------------------------------------------------------------------------- #
# Raw Piquasso helper (independent of photographiq.backends.piquasso)
# --------------------------------------------------------------------------- #


def raw_piquasso(alphas, gate_builder, modes):
    with pq.Program() as program:
        pq.Q(*range(modes)) | pq.Vacuum()
        for i, alpha in enumerate(alphas):
            pq.Q(i) | pq.Displacement(r=abs(alpha), phi=np.angle(alpha))
        gate_builder()
    state = pq.GaussianSimulator(d=modes, config=pq.Config(hbar=HBAR)).execute(program).state
    return np.array(state.xpxp_mean_vector), np.array(state.xpxp_covariance_matrix) / 2


def coherent_mean_cov(alpha: complex):
    return np.array([2 * alpha.real, 2 * alpha.imag]), np.eye(2)


def run_case(name, params, modes, alphas, target_matrix, target_delta, pg_commands, raw_builder):
    means = [2 * a.real for a in alphas], [2 * a.imag for a in alphas]
    mean_in = np.empty(2 * modes)
    mean_in[0::2] = [2 * a.real for a in alphas]
    mean_in[1::2] = [2 * a.imag for a in alphas]
    cov_in = np.eye(2 * modes)

    analytic_mean = target_matrix @ mean_in + target_delta
    analytic_cov = target_matrix @ cov_in @ target_matrix.T

    inputs = {i: pg.GaussianInput.coherent(a) for i, a in enumerate(alphas)}
    rows = {}
    for backend in ("gaussian", "piquasso"):
        pattern = pg.Pattern(inputs=tuple(range(modes)))
        pattern.extend(pg_commands)
        pattern.append(pg.Output(tuple(range(modes))))
        result = pg.simulate(pattern, backend=backend, inputs=inputs, seed=0)
        rows[f"{backend}_mean_error"] = common.frobenius_error(result.state.mean, analytic_mean)
        rows[f"{backend}_covariance_error"] = common.frobenius_error(result.state.covariance, analytic_cov)

    raw_mean, raw_cov = raw_piquasso(alphas, raw_builder, modes)
    rows["raw_piquasso_mean_error"] = common.frobenius_error(raw_mean, analytic_mean)
    rows["raw_piquasso_covariance_error"] = common.frobenius_error(raw_cov, analytic_cov)

    return {"gate": name, "params": params, **rows}


def main():
    plt = common.setup_style()
    rows = []

    rows.append(
        run_case(
            "rotation", {"theta": 0.6}, 1, [0.4 + 0.2j],
            target_rotation(0.6), np.zeros(2), [pg.Rotate(0, 0.6)],
            lambda: (pq.Q(0) | pq.Phaseshifter(phi=0.6)),
        )
    )
    rows.append(
        run_case(
            "squeezing", {"r": 0.5}, 1, [0.3 - 0.1j],
            target_squeezing(0.5), np.zeros(2), [pg.Squeeze(0, 0.5)],
            lambda: (pq.Q(0) | pq.Squeezing(r=0.5)),
        )
    )
    rows.append(
        run_case(
            "displacement", {"q": 0.4, "p": -0.3}, 1, [0.1 + 0.1j],
            np.eye(2), target_displacement_delta(0.4, -0.3), [pg.Displace(0, q=0.4, p=-0.3)],
            lambda: (pq.Q(0) | pq.Displacement(r=abs(complex(0.4, -0.3) / 2), phi=np.angle(complex(0.4, -0.3) / 2))),
        )
    )
    rows.append(
        run_case(
            "beamsplitter", {"theta": 0.35}, 2, [0.3 + 0.1j, -0.2 + 0.2j],
            target_beamsplitter(0.35), np.zeros(4), [pg.BeamSplitter(0, 1, 0.35)],
            lambda: (pq.Q(0, 1) | pq.Beamsplitter(theta=0.35, phi=0.0)),
        )
    )
    rows.append(
        run_case(
            "cz", {"weight": 0.7}, 2, [0.2 + 0.3j, 0.1 - 0.4j],
            target_cz(0.7), np.zeros(4), [pg.Entangle(0, 1, 0.7)],
            lambda: (pq.Q(0, 1) | pq.ControlledZ(s=0.7)),
        )
    )
    compound_matrix = target_cz(0.4) @ np.block(
        [[target_rotation(0.3), np.zeros((2, 2))], [np.zeros((2, 2)), np.eye(2)]]
    ) @ np.block([[target_squeezing(0.2), np.zeros((2, 2))], [np.zeros((2, 2)), np.eye(2)]])
    rows.append(
        run_case(
            "compound(squeeze+rotate+cz)", {"r": 0.2, "angle": 0.3, "weight": 0.4}, 2,
            [0.2 + 0.0j, -0.1 + 0.2j], compound_matrix, np.zeros(4),
            [pg.Squeeze(0, 0.2), pg.Rotate(0, 0.3), pg.Entangle(0, 1, 0.4)],
            lambda: (
                pq.Q(0) | pq.Squeezing(r=0.2),
                pq.Q(0) | pq.Phaseshifter(phi=0.3),
                pq.Q(0, 1) | pq.ControlledZ(s=0.4),
            ),
        )
    )

    tolerance = 1e-9
    error_keys = [k for k in rows[0] if k.endswith("_error")]
    failures = [r for r in rows if any(r[k] > tolerance for k in error_keys)]
    if failures:
        common.save_csv(failures, "R16_numpy_analytic_reference_FAILURES")
        raise AssertionError("A backend disagreed with the independent analytic target; see FAILURES csv")

    common.save_result(rows, "R16_numpy_analytic_reference")

    fig, ax = plt.subplots(figsize=(7, 3.8))
    gates = [r["gate"] for r in rows]
    x = np.arange(len(gates))
    width = 0.25
    ax.bar(x - width, [common.safe_log10(r["gaussian_covariance_error"]) for r in rows], width, label="NumPy Gaussian backend")
    ax.bar(x, [common.safe_log10(r["piquasso_covariance_error"]) for r in rows], width, label="Piquasso Gaussian backend")
    ax.bar(x + width, [common.safe_log10(r["raw_piquasso_covariance_error"]) for r in rows], width, label="Raw Piquasso")
    ax.set_xticks(x)
    ax.set_xticklabels(gates, rotation=25, ha="right", fontsize=7)
    ax.set_ylabel(r"$\log_{10}$(covariance error vs. NumPy analytic target)")
    ax.legend(fontsize=7)
    common.save_figure(fig, "R16_numpy_analytic_reference")
    plt.close(fig)

    common.print_summary(
        "R16 independent NumPy analytic reference",
        cases=len(rows),
        max_error=max(max(r[k] for k in error_keys) for r in rows),
    )


if __name__ == "__main__":
    main()



=== R16 independent NumPy analytic reference ===
  cases: 6
  max_error: 4.577566798522237e-16


## R17: Graphix structural comparison (structure only, never CV amplitudes).

For a line and a star resource topology, builds a hand-matched Graphix
qubit-MBQC ``Pattern`` (N/E/M/X/Z commands) alongside the equivalent
PhotoGraphiQ CV pattern, and compares only *structural* invariants: resource
graph isomorphism/edges, input/output node sets, measurement order, and
classical-dependency domains. Continuous-variable amplitudes, quadrature
statistics or any numerical physics are never compared against Graphix,
per the package's documented validation policy (docs/validation/index.md).

In [19]:
"""R17: Graphix structural comparison (structure only, never CV amplitudes).

For a line and a star resource topology, builds a hand-matched Graphix
qubit-MBQC ``Pattern`` (N/E/M/X/Z commands) alongside the equivalent
PhotoGraphiQ CV pattern, and compares only *structural* invariants: resource
graph isomorphism/edges, input/output node sets, measurement order, and
classical-dependency domains. Continuous-variable amplitudes, quadrature
statistics or any numerical physics are never compared against Graphix,
per the package's documented validation policy (docs/validation/index.md).
"""

from __future__ import annotations
try:

    import sys
    from pathlib import Path

    import networkx as nx

    import common  # noqa: E402

    if not common.HAS_GRAPHIX:
        print("SKIPPED: graphix is not installed; see requirements_notes.md")
        sys.exit(0)

    import graphix  # noqa: E402
    from graphix import command  # noqa: E402

    import photographiq as pg  # noqa: E402


    def build_line_case(n: int):
        graph = pg.CVGraph.line(n, inputs=(0,))
        cv = pg.Pattern(graph)
        cv.measure(0, pg.Homodyne.q())
        for i in range(1, n - 1):
            cv.measure(i, pg.Homodyne(0.2 + 0.1 * i + pg.Outcome(i - 1)))
        cv.displace(n - 1, q=pg.Outcome(n - 2), p=(pg.Outcome(0) if n > 2 else 0.0))
        cv.append(pg.Output((n - 1,)))
        cv.validate()

        dv = graphix.Pattern(input_nodes=[0])
        for i in range(1, n):
            dv.add(command.N(i))
        for i in range(n - 1):
            dv.add(command.E((i, i + 1)))
        dv.add(command.M(0))
        for i in range(1, n - 1):
            dv.add(command.M(i, s_domain={i - 1}))
        dv.add(command.X(n - 1, domain={n - 2}))
        if n > 2:
            dv.add(command.Z(n - 1, domain={0}))
        return cv, dv


    def build_star_case(leaves: int):
        graph = pg.CVGraph.star(leaves, inputs=(0,))
        leaf_nodes = tuple(range(1, leaves + 1))
        cv = pg.Pattern(graph)
        cv.measure(0, pg.Homodyne.q())
        for leaf in leaf_nodes:
            cv.displace(leaf, q=pg.Outcome(0))
        cv.append(pg.Output(leaf_nodes))
        cv.validate()

        dv = graphix.Pattern(input_nodes=[0])
        for leaf in leaf_nodes:
            dv.add(command.N(leaf))
        for leaf in leaf_nodes:
            dv.add(command.E((0, leaf)))
        dv.add(command.M(0))
        for leaf in leaf_nodes:
            dv.add(command.X(leaf, domain={0}))
        return cv, dv


    def compare(name, cv, dv):
        og = dv.to_opengraph()
        isomorphic = nx.is_isomorphic(og.graph, cv.graph.network)
        edges_match = common.canonical_edges(og.graph) == common.canonical_edges(cv.graph.network)
        io_match = tuple(dv.input_nodes) == cv.inputs and tuple(dv.output_nodes) == cv.outputs
        graphix_measure_order = [c.node for c in dv if isinstance(c, command.M)]
        cv_measure_order = [c.node for c in cv.commands if isinstance(c, pg.Measure)]
        order_match = graphix_measure_order == cv_measure_order

        cv_measurements = [(i, c) for i, c in enumerate(cv.commands) if isinstance(c, pg.Measure)]
        dependency_graph = cv.dependencies()
        domain_checks = []
        for gx_command in dv:
            if isinstance(gx_command, command.M) and gx_command.s_domain:
                target_index = next(i for i, c in cv_measurements if c.node == gx_command.node)
                for source_node in gx_command.s_domain:
                    source_index = next(i for i, c in cv_measurements if c.node == source_node)
                    domain_checks.append(dependency_graph.has_edge(source_index, target_index))
            if isinstance(gx_command, command.X) and gx_command.domain:
                for source_node in gx_command.domain:
                    source_index = next(i for i, c in cv_measurements if c.node == source_node)
                    # The corresponding correction is the Displace on gx_command.node.
                    target_index = next(
                        i for i, c in enumerate(cv.commands)
                        if isinstance(c, pg.Displace) and c.node == gx_command.node
                    )
                    domain_checks.append(dependency_graph.has_edge(source_index, target_index))
        domains_match = all(domain_checks) and len(domain_checks) > 0

        return {
            "case": name,
            "nodes": len(cv.graph.nodes),
            "edges": cv.graph.network.number_of_edges(),
            "graph_isomorphic": isomorphic,
            "canonical_edges_match": edges_match,
            "input_output_match": io_match,
            "measurement_order_match": order_match,
            "dependency_domain_checks": len(domain_checks),
            "dependency_domains_match": domains_match,
        }


    def main():
        rows = [
            compare("line_n3", *build_line_case(3)),
            compare("line_n5", *build_line_case(5)),
            compare("star_leaves_3", *build_star_case(3)),
            compare("star_leaves_5", *build_star_case(5)),
        ]

        failures = [
            r for r in rows
            if not (r["graph_isomorphic"] and r["canonical_edges_match"] and r["input_output_match"]
                    and r["measurement_order_match"] and r["dependency_domains_match"])
        ]
        if failures:
            common.save_csv(failures, "R17_graphix_structure_FAILURES")
            raise AssertionError("Structural comparison against Graphix failed; see FAILURES csv")

        common.save_result(
            rows,
            "R17_graphix_structure",
            extra={
                "note": "Structural comparison only: node/edge/I-O/order/domain equivalence. "
                "Graphix never validates CV amplitudes or quadrature statistics."
            },
        )

        common.print_summary("R17 Graphix structural comparison", cases=len(rows), all_passed=not failures)


    if __name__ == "__main__":
        main()
except SystemExit:
    pass



=== R17 Graphix structural comparison ===
  cases: 4
  all_passed: True


## R18: Gaussian backend cross-check.

Runs the same random, low-energy, purely physical (non-adaptive) two-mode
Gaussian workloads through the NumPy Gaussian backend and the Piquasso
Gaussian backend (both exact) and compares state mean/covariance/photon
number/quadrature exactly. It additionally runs the same preparation through
the experimental ``piquasso-mixed-fock`` backend at a modest cutoff and
compares photon number/quadrature -- this third comparison is a *finite Fock
truncation approximation*, not an exact-backend agreement, and uses a looser,
explicitly reported tolerance.

In [20]:
"""R18: Gaussian backend cross-check.

Runs the same random, low-energy, purely physical (non-adaptive) two-mode
Gaussian workloads through the NumPy Gaussian backend and the Piquasso
Gaussian backend (both exact) and compares state mean/covariance/photon
number/quadrature exactly. It additionally runs the same preparation through
the experimental ``piquasso-mixed-fock`` backend at a modest cutoff and
compares photon number/quadrature -- this third comparison is a *finite Fock
truncation approximation*, not an exact-backend agreement, and uses a looser,
explicitly reported tolerance.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SEED = 20260913
N_RANDOM = 15
FOCK_CUTOFF = 14
GATE_KINDS = ("rotate", "squeeze", "beamsplitter", "cz", "displace")


def random_pattern(generator, n_gates=4):
    pattern = pg.Pattern(inputs=(0, 1))
    for _ in range(n_gates):
        kind = generator.choice(GATE_KINDS)
        if kind == "rotate":
            pattern.append(pg.Rotate(int(generator.integers(0, 2)), float(generator.uniform(-np.pi, np.pi))))
        elif kind == "squeeze":
            pattern.append(pg.Squeeze(int(generator.integers(0, 2)), float(generator.uniform(-0.4, 0.4))))
        elif kind == "beamsplitter":
            pattern.append(pg.BeamSplitter(0, 1, float(generator.uniform(-np.pi / 2, np.pi / 2))))
        elif kind == "cz":
            pattern.append(pg.Entangle(0, 1, float(generator.uniform(-1, 1))))
        else:
            pattern.append(
                pg.Displace(int(generator.integers(0, 2)), q=float(generator.uniform(-0.3, 0.3)), p=float(generator.uniform(-0.3, 0.3)))
            )
    pattern.append(pg.Output((0, 1)))
    return pattern.validate()


def main():
    plt = common.setup_style()
    generator = common.rng(SEED)
    rows = []
    for i in range(N_RANDOM):
        pattern = random_pattern(generator)
        alpha0 = generator.uniform(-0.25, 0.25) + 1j * generator.uniform(-0.25, 0.25)
        alpha1 = generator.uniform(-0.25, 0.25) + 1j * generator.uniform(-0.25, 0.25)
        inputs = {0: pg.GaussianInput.coherent(alpha0), 1: pg.GaussianInput.coherent(alpha1)}

        gaussian = pg.simulate(pattern, backend="gaussian", inputs=inputs, seed=0).state
        piquasso = pg.simulate(pattern, backend="piquasso", inputs=inputs, seed=0).state

        mean_error = common.frobenius_error(piquasso.mean, gaussian.mean)
        cov_error = common.frobenius_error(piquasso.covariance, gaussian.covariance)
        photon_error = max(abs(piquasso.photon_number(n) - gaussian.photon_number(n)) for n in (0, 1))

        row = {
            "sample": i,
            "n_gates": len(pattern.commands) - 1,
            "gaussian_vs_piquasso_mean_error": mean_error,
            "gaussian_vs_piquasso_covariance_error": cov_error,
            "gaussian_vs_piquasso_photon_number_error": photon_error,
        }

        fock_failure = common.safe_cutoff_run(
            lambda: pg.simulate(pattern, backend="piquasso-mixed-fock", cutoff=FOCK_CUTOFF, inputs=inputs, seed=0).state,
            label=f"sample {i} mixed-fock",
        )
        if fock_failure["ok"]:
            fock_state = fock_failure["value"]
            row["mixed_fock_photon_number_error"] = max(
                abs(fock_state.photon_number(n) - gaussian.photon_number(n)) for n in (0, 1)
            )
            row["mixed_fock_q_error"] = max(
                abs(fock_state.quadrature(n)[0] - gaussian.quadrature(n)[0]) for n in (0, 1)
            )
            row["mixed_fock_cutoff"] = FOCK_CUTOFF
            row["mixed_fock_ok"] = True
        else:
            row.update(
                mixed_fock_photon_number_error=None, mixed_fock_q_error=None,
                mixed_fock_cutoff=FOCK_CUTOFF, mixed_fock_ok=False, mixed_fock_error=fock_failure["error"],
            )
        rows.append(row)

    exact_tolerance = 1e-8
    exact_failures = [
        r for r in rows
        if r["gaussian_vs_piquasso_mean_error"] > exact_tolerance
        or r["gaussian_vs_piquasso_covariance_error"] > exact_tolerance
        or r["gaussian_vs_piquasso_photon_number_error"] > exact_tolerance
    ]
    if exact_failures:
        common.save_csv(exact_failures, "R18_gaussian_backends_FAILURES")
        raise AssertionError("NumPy Gaussian and Piquasso Gaussian backends disagreed beyond tolerance; see FAILURES csv")

    common.save_result(rows, "R18_gaussian_backends", extra={"seed": SEED, "fock_cutoff": FOCK_CUTOFF})

    ok_rows = [r for r in rows if r["mixed_fock_ok"]]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].hist([common.safe_log10(r["gaussian_vs_piquasso_covariance_error"]) for r in rows], bins=15, color="#24677b")
    axes[0].set_xlabel(r"$\log_{10}$(covariance error): NumPy vs. Piquasso (exact)")
    axes[0].set_ylabel("Count")
    if ok_rows:
        axes[1].hist([common.safe_log10(r["mixed_fock_photon_number_error"]) for r in ok_rows], bins=15, color="#c66d27")
    axes[1].set_xlabel(rf"$\log_{{10}}$(photon-number error): Gaussian vs. mixed-Fock ($c={FOCK_CUTOFF}$)")
    fig.suptitle("Backend cross-checks: exact (left) vs. finite-cutoff approximation (right)")
    common.save_figure(fig, "R18_gaussian_backends")
    plt.close(fig)

    common.print_summary(
        "R18 Gaussian backend cross-check",
        samples=len(rows),
        max_exact_covariance_error=max(r["gaussian_vs_piquasso_covariance_error"] for r in rows),
        mixed_fock_successes=len(ok_rows),
        median_mixed_fock_photon_error=float(np.median([r["mixed_fock_photon_number_error"] for r in ok_rows])) if ok_rows else None,
    )


if __name__ == "__main__":
    main()


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:494: RuntimeWarning: Squeezing: retained Fock norm 0.99999996; test a larger cutoff
  self._gate((node,), pq.Squeezing(r=r))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999951; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99999994; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999954; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999761; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999999; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999705; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999854; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:494: RuntimeWarning: Squeezing: retained Fock norm 0.99999998; test a larger cutoff
  self._gate((node,), pq.Squeezing(r=r))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99999999; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:494: RuntimeWarning: Squeezing: retained Fock norm 0.99998953; test a larger cutoff
  self._gate((node,), pq.Squeezing(r=r))



=== R18 Gaussian backend cross-check ===
  samples: 15
  max_exact_covariance_error: 7.506179160044328e-16
  mixed_fock_successes: 14
  median_mixed_fock_photon_error: 4.104472805382953e-08


## R19: Photon subtraction analytic probability.

Uses the package's physical vacuum-tap subtraction pattern
(``photographiq.non_gaussian.photon_subtraction``): a beamsplitter of angle
theta couples input |n> to a vacuum ancilla, which is then photon-counted.
For input |n>, heralding count k has the exact binomial probability
C(n,k) sin(theta)^(2k) cos(theta)^(2(n-k)) -- this is the analytic reference,
derived independently here with SciPy, and the resulting signal-mode photon
number is deterministically n-k (a number-conserving beamsplitter on a
definite Fock input produces a photon-number-correlated joint state).

In [21]:
"""R19: Photon subtraction analytic probability.

Uses the package's physical vacuum-tap subtraction pattern
(``photographiq.non_gaussian.photon_subtraction``): a beamsplitter of angle
theta couples input |n> to a vacuum ancilla, which is then photon-counted.
For input |n>, heralding count k has the exact binomial probability
C(n,k) sin(theta)^(2k) cos(theta)^(2(n-k)) -- this is the analytic reference,
derived independently here with SciPy, and the resulting signal-mode photon
number is deterministically n-k (a number-conserving beamsplitter on a
definite Fock input produces a photon-number-correlated joint state).
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from scipy.special import comb

import common  # noqa: E402

import photographiq as pg  # noqa: E402

THETAS = np.linspace(0.05, np.pi / 2 - 0.05, 15)
PHOTON_NUMBERS = (1, 2, 3)
CUTOFF = 8


def analytic_probability(n, k, theta):
    return float(comb(n, k) * np.sin(theta) ** (2 * k) * np.cos(theta) ** (2 * (n - k)))


def main():
    plt = common.setup_style()
    rows = []
    for n in PHOTON_NUMBERS:
        for theta in THETAS:
            pattern = pg.non_gaussian.photon_subtraction(float(theta))
            for k in range(n + 1):
                result = pg.simulate(
                    pattern,
                    backend="piquasso-fock",
                    cutoff=CUTOFF,
                    inputs={"in": pg.FockInput.number(n)},
                    measurement_outcomes={"count": k},
                    seed=0,
                )
                pg_probability = result.measurement_statistics["count"]["value"]
                analytic = analytic_probability(n, k, float(theta))
                conditional_n = result.state.photon_number("in")
                rows.append(
                    {
                        "n": n,
                        "theta": float(theta),
                        "k": k,
                        "analytic_probability": analytic,
                        "photographiq_probability": pg_probability,
                        "absolute_error": abs(pg_probability - analytic),
                        "relative_error": common.relative_error(pg_probability, analytic, floor=1e-12),
                        "conditional_photon_number": conditional_n,
                        "expected_conditional_photon_number": n - k,
                        "conditional_photon_number_error": abs(conditional_n - (n - k)),
                    }
                )

    tolerance = 1e-9
    photon_tolerance = 1e-9
    failures = [
        r for r in rows
        if r["absolute_error"] > tolerance or r["conditional_photon_number_error"] > photon_tolerance
    ]
    if failures:
        common.save_csv(failures, "R19_photon_subtraction_probability_FAILURES")
        raise AssertionError(f"{len(failures)} cases disagreed with analytic predictions; see FAILURES csv")

    common.save_result(rows, "R19_photon_subtraction_probability", extra={"cutoff": CUTOFF})

    fig, axes = plt.subplots(1, len(PHOTON_NUMBERS), figsize=(4 * len(PHOTON_NUMBERS), 3.6), sharey=True)
    for ax, n in zip(axes, PHOTON_NUMBERS, strict=True):
        for k in range(n + 1):
            subset = [r for r in rows if r["n"] == n and r["k"] == k]
            ts = [r["theta"] for r in subset]
            ax.plot(ts, [r["analytic_probability"] for r in subset], "-", label=f"k={k} analytic")
            ax.plot(ts, [r["photographiq_probability"] for r in subset], "o", markersize=3, label=f"k={k} PhotoGraphiQ")
        ax.set_xlabel(r"Tap angle $\theta$")
        ax.set_title(f"n={n}")
        ax.legend(fontsize=6)
    axes[0].set_ylabel("Herald probability")
    fig.suptitle("Photon subtraction: analytic binomial vs. simulated probability")
    common.save_figure(fig, "R19_photon_subtraction_probability")
    plt.close(fig)

    common.print_summary(
        "R19 photon subtraction analytic probability",
        cases=len(rows),
        max_absolute_error=max(r["absolute_error"] for r in rows),
        max_conditional_photon_error=max(r["conditional_photon_number_error"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R19 photon subtraction analytic probability ===
  cases: 135
  max_absolute_error: 4.440892098500626e-16
  max_conditional_photon_error: 8.881784197001252e-16


## R20: Photon subtraction on squeezed vacuum.

Prepares a squeezed-vacuum input (``photographiq.states.GaussianInput.squeezed``,
which the pure-Fock backend accepts directly and converts to a truncated
Fock vector) and performs one-photon heralded subtraction via
``photographiq.non_gaussian.photon_subtraction``. Compares the Fock
distribution, parity and mean photon number before/after, and demonstrates
the expected even-to-odd parity flip (squeezed vacuum has only even photon
numbers; single-photon subtraction produces a cat-like odd-photon-dominant
state). Uses ``cutoff_convergence`` to check the finite-cutoff behavior of
this filtering/attenuating physical subtraction (not an ideal a-operator).

In [22]:
"""R20: Photon subtraction on squeezed vacuum.

Prepares a squeezed-vacuum input (``photographiq.states.GaussianInput.squeezed``,
which the pure-Fock backend accepts directly and converts to a truncated
Fock vector) and performs one-photon heralded subtraction via
``photographiq.non_gaussian.photon_subtraction``. Compares the Fock
distribution, parity and mean photon number before/after, and demonstrates
the expected even-to-odd parity flip (squeezed vacuum has only even photon
numbers; single-photon subtraction produces a cat-like odd-photon-dominant
state). Uses ``cutoff_convergence`` to check the finite-cutoff behavior of
this filtering/attenuating physical subtraction (not an ideal a-operator).
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SQUEEZING_R = 0.6
THETA = 0.15  # small tap angle: closer to ideal (low-attenuation) subtraction
CUTOFF = 24
CONVERGENCE_CUTOFFS = [12, 16, 20, 28, 36]


def main():
    plt = common.setup_style()
    squeezed_input = pg.GaussianInput.squeezed(-SQUEEZING_R)  # q-squeezed: even support, like docs' resource convention

    before_pattern = pg.Pattern(inputs=("in",)).append(pg.Output(("in",))).validate()
    before = pg.simulate(
        before_pattern, backend="piquasso-fock", cutoff=CUTOFF, inputs={"in": squeezed_input}, seed=0
    ).state

    subtraction_pattern = pg.non_gaussian.photon_subtraction(THETA)
    after_result = pg.simulate(
        subtraction_pattern,
        backend="piquasso-fock",
        cutoff=CUTOFF,
        inputs={"in": squeezed_input},
        measurement_outcomes={"count": 1},
        seed=0,
    )
    after = after_result.state

    before_parity = before.parity()
    after_parity = after.parity()
    before_mean_n = before.photon_number("in")
    after_mean_n = after.photon_number("in")

    parity_flipped = before_parity > 0.5 and after_parity < -0.5
    if not parity_flipped:
        raise AssertionError(
            f"Expected an even-to-odd parity flip; got before={before_parity:.4f}, after={after_parity:.4f}"
        )

    before_probs = before.probabilities
    after_probs = after.probabilities
    max_n = max(max(b[0] for b in before_probs), max(b[0] for b in after_probs)) + 1
    distribution_rows = [
        {
            "n": n,
            "before_probability": before_probs.get((n,), 0.0),
            "after_probability": after_probs.get((n,), 0.0),
        }
        for n in range(max_n)
    ]

    summary = {
        "squeezing_r": SQUEEZING_R,
        "tap_theta": THETA,
        "cutoff": CUTOFF,
        "before_parity": before_parity,
        "after_parity": after_parity,
        "before_mean_photon_number": before_mean_n,
        "after_mean_photon_number": after_mean_n,
        "herald_probability": after_result.measurement_statistics["count"]["value"],
        "parity_flipped_even_to_odd": parity_flipped,
    }
    common.save_json(summary, "R20_subtracted_squeezed_state_summary")
    common.save_csv(distribution_rows, "R20_subtracted_squeezed_state_distribution")
    common.write_metadata("R20_subtracted_squeezed_state")

    q_axis = np.linspace(-5, 5, 121)
    p_axis = np.linspace(-5, 5, 121)
    before_wigner = before.wigner(q_axis, p_axis, node="in")
    after_wigner = after.wigner(q_axis, p_axis, node="in")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    width = 0.35
    ns = [r["n"] for r in distribution_rows]
    axes[0].bar(np.array(ns) - width / 2, [r["before_probability"] for r in distribution_rows], width, label="Before", color="#24677b")
    axes[0].bar(np.array(ns) + width / 2, [r["after_probability"] for r in distribution_rows], width, label="After subtraction", color="#c66d27")
    axes[0].set_xlabel("Photon number n")
    axes[0].set_ylabel("Probability")
    axes[0].legend(fontsize=7)
    axes[0].set_title("Fock distribution")

    for ax, grid, title in ((axes[1], before_wigner, "Before (squeezed vacuum)"), (axes[2], after_wigner, "After subtraction")):
        extent = max(abs(grid.values.min()), abs(grid.values.max()))
        mesh = ax.pcolormesh(grid.q, grid.p, grid.values, shading="auto", cmap="RdBu_r", vmin=-extent, vmax=extent)
        ax.set(xlabel="q", ylabel="p", aspect="equal", title=title)
        fig.colorbar(mesh, ax=ax, fraction=0.046)
    fig.suptitle(f"Photon subtraction on squeezed vacuum (r={SQUEEZING_R}, theta={THETA})")
    common.save_figure(fig, "R20_subtracted_squeezed_state")
    plt.close(fig)

    # Cutoff convergence for this fixed conditional branch (measurement_outcomes
    # fixes count=1 so every cutoff compares the same branch).
    study = pg.cutoff_convergence(
        subtraction_pattern,
        CONVERGENCE_CUTOFFS,
        backend="piquasso-fock",
        inputs={"in": squeezed_input},
        measurement_outcomes={"count": 1},
    )
    convergence_rows = [
        {
            "cutoff": row["cutoff"],
            "norm": row["norm"],
            "minimum_retained_norm": row["minimum_retained_norm"],
            "maximum_boundary_population": row["maximum_boundary_population"],
            "fidelity_to_previous": row["fidelity_to_previous"],
            "trace_distance_to_previous": row["trace_distance_to_previous"],
            "comparable_to_previous": row["comparable_to_previous"],
        }
        for row in study.rows
    ]
    common.save_csv(convergence_rows, "R20_subtracted_squeezed_state_convergence")

    common.print_summary(
        "R20 photon subtraction on squeezed vacuum",
        before_parity=before_parity,
        after_parity=after_parity,
        parity_flipped=parity_flipped,
        final_fidelity_to_previous_cutoff=convergence_rows[-1]["fidelity_to_previous"],
    )


if __name__ == "__main__":
    main()


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:212: RuntimeWarning: preparation: retained Fock norm 0.99999994; test a larger cutoff
  engine.prepare(node, squeezing=0, state=inputs.get(node, GaussianInput()))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:212: RuntimeWarning: preparation: retained Fock norm 0.9998502; test a larger cutoff
  engine.prepare(node, squeezing=0, state=inputs.get(node, GaussianInput()))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:212: RuntimeWarning: preparation: retained Fock norm 0.99998909; test a larger cutoff
  engine.prepare(node, squeezing=0, state=inputs.get(node, GaussianInput()))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:212: RuntimeWarning: preparation: retained Fock norm 0.99999918; test a larger cutoff
  engine.prepare(node, squeezing=0, state=inputs.get(node, GaussianInput()))



=== R20 photon subtraction on squeezed vacuum ===
  before_parity: 1.0
  after_parity: -0.9999999999999993
  parity_flipped: True
  final_fidelity_to_previous_cutoff: 0.9999998094976486


## R21: Cat state.

Generates even and odd finite-Fock cat resources (``photographiq.resources.
CatResource``, which internally calls ``FockInput.cat``) for several alpha
values, and cross-checks against an independently written NumPy coherent-
state expansion (not reusing ``FockInput.cat``). Reports parity,
photon-number distribution, normalization/retained mass and Wigner functions,
and checks the expected even/odd parity sign.

In [23]:
"""R21: Cat state.

Generates even and odd finite-Fock cat resources (``photographiq.resources.
CatResource``, which internally calls ``FockInput.cat``) for several alpha
values, and cross-checks against an independently written NumPy coherent-
state expansion (not reusing ``FockInput.cat``). Reports parity,
photon-number distribution, normalization/retained mass and Wigner functions,
and checks the expected even/odd parity sign.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from scipy.special import gammaln

import common  # noqa: E402

import photographiq as pg  # noqa: E402

ALPHAS = [0.5, 1.0, 1.5, 2.0]
CUTOFF = 32
REPRESENTATIVE_ALPHA = 1.5


def analytic_cat(alpha: float, cutoff: int, parity: int) -> np.ndarray:
    """Independent NumPy |alpha>+parity|-alpha> expansion (not FockInput.cat)."""
    n = np.arange(cutoff)
    log_coherent = n * np.log(alpha) - gammaln(n + 1) / 2 - 0.5 * alpha**2 if alpha > 0 else None
    if alpha == 0:
        vector = np.zeros(cutoff)
        vector[0] = 1 + parity
    else:
        vector = np.exp(log_coherent) * (1 + parity * (-1.0) ** n)
    norm = np.linalg.norm(vector)
    return vector / norm


def main():
    plt = common.setup_style()
    rows = []
    wigner_states = {}
    for alpha in ALPHAS:
        for parity in (1, -1):
            resource = pg.CatResource(alpha, parity=parity)
            pattern = pg.Pattern(inputs=()).append(pg.Prepare(0, state=resource)).append(pg.Output((0,))).validate()
            result = pg.simulate(pattern, backend="piquasso-fock", cutoff=CUTOFF, seed=0)
            state = result.state

            analytic = analytic_cat(alpha, CUTOFF, parity)
            actual_amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
            actual_vector = np.array([actual_amplitudes.get((n,), 0j) for n in range(CUTOFF)])
            overlap = np.vdot(analytic, actual_vector)
            fidelity = float(abs(overlap) ** 2)

            expected_parity = float(parity)
            actual_parity = state.parity()

            rows.append(
                {
                    "alpha": alpha,
                    "parity_label": "even" if parity == 1 else "odd",
                    "fidelity_vs_independent_analytic": fidelity,
                    "expected_ideal_parity": expected_parity,
                    "measured_parity": actual_parity,
                    "mean_photon_number": state.photon_number(0),
                    "norm": state.norm,
                    "retained_norm": state.retained_norms[-1] if state.retained_norms else None,
                    "boundary_population": state.diagnostics[-1]["boundary_population"] if state.diagnostics else None,
                }
            )
            if alpha == REPRESENTATIVE_ALPHA:
                wigner_states[parity] = state

    fidelity_tolerance = 1e-6
    parity_sign_failures = [r for r in rows if np.sign(r["measured_parity"]) != np.sign(r["expected_ideal_parity"])]
    fidelity_failures = [r for r in rows if r["fidelity_vs_independent_analytic"] < 1 - fidelity_tolerance]
    if parity_sign_failures or fidelity_failures:
        common.save_csv(parity_sign_failures + fidelity_failures, "R21_cat_state_FAILURES")
        raise AssertionError("Cat state parity sign or independent-fidelity check failed; see FAILURES csv")

    common.save_result(rows, "R21_cat_state", extra={"cutoff": CUTOFF})

    q_axis = np.linspace(-6, 6, 121)
    p_axis = np.linspace(-6, 6, 121)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for ax, (parity, label) in zip(axes, ((1, "Even cat"), (-1, "Odd cat")), strict=True):
        grid = wigner_states[parity].wigner(q_axis, p_axis, node=0)
        extent = max(abs(grid.values.min()), abs(grid.values.max()))
        mesh = ax.pcolormesh(grid.q, grid.p, grid.values, shading="auto", cmap="RdBu_r", vmin=-extent, vmax=extent)
        ax.set(xlabel="q", ylabel="p", aspect="equal", title=f"{label}, alpha={REPRESENTATIVE_ALPHA}")
        fig.colorbar(mesh, ax=ax, fraction=0.046)
    fig.suptitle("Cat-state Wigner functions")
    common.save_figure(fig, "R21_cat_state")
    plt.close(fig)

    common.print_summary(
        "R21 cat state",
        cases=len(rows),
        min_fidelity_to_analytic=min(r["fidelity_vs_independent_analytic"] for r in rows),
        all_parity_signs_correct=not parity_sign_failures,
    )


if __name__ == "__main__":
    main()



=== R21 cat state ===
  cases: 8
  min_fidelity_to_analytic: 0.9999999999999998
  all_parity_signs_correct: True


## R22: Cubic phase direct execution.

Applies ``photographiq.commands.CubicPhase(gamma)`` (native Piquasso
``exp(i*gamma*q^3/6)``, hbar=2) to a low-energy coherent input on the pure
Fock backend, and compares the resulting truncated state vector against an
independently built finite-dimensional reference: a dense ladder operator
``a`` constructed directly with NumPy, ``q = a + a^T``, and
``U = scipy.linalg.expm(i*gamma*q^3/6)`` applied to the SAME truncated input
vector. This is a finite-Fock-truncation comparison at matched cutoff, not an
infinite-cutoff convergence claim (see R33/R34 for that).

In [24]:
"""R22: Cubic phase direct execution.

Applies ``photographiq.commands.CubicPhase(gamma)`` (native Piquasso
``exp(i*gamma*q^3/6)``, hbar=2) to a low-energy coherent input on the pure
Fock backend, and compares the resulting truncated state vector against an
independently built finite-dimensional reference: a dense ladder operator
``a`` constructed directly with NumPy, ``q = a + a^T``, and
``U = scipy.linalg.expm(i*gamma*q^3/6)`` applied to the SAME truncated input
vector. This is a finite-Fock-truncation comparison at matched cutoff, not an
infinite-cutoff convergence claim (see R33/R34 for that).
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from scipy.linalg import expm

import common  # noqa: E402

import photographiq as pg  # noqa: E402

GAMMAS = [0.02, 0.05, 0.1, 0.2]
CUTOFFS = [16, 24, 32, 48]
ALPHA = 0.3 + 0.1j


def independent_cubic_unitary(gamma: float, cutoff: int) -> np.ndarray:
    """Independent dense NumPy/SciPy reference: exp(i*gamma*q^3/6) at fixed cutoff."""
    a = np.diag(np.sqrt(np.arange(1, cutoff)), k=1)
    q = a + a.T
    return expm(1j * gamma * (q @ q @ q) / 6)


def truncated_input_vector(cutoff: int) -> np.ndarray:
    pattern = pg.Pattern(inputs=(0,)).append(pg.Output((0,))).validate()
    state = pg.simulate(
        pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0
    ).state
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    return np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])


def main():
    plt = common.setup_style()
    rows = []
    for cutoff in CUTOFFS:
        before = truncated_input_vector(cutoff)
        for gamma in GAMMAS:
            pattern = pg.Pattern(inputs=(0,)).append(pg.CubicPhase(0, gamma)).append(pg.Output((0,))).validate()
            state = pg.simulate(
                pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0
            ).state
            amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
            actual = np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])

            unitary = independent_cubic_unitary(gamma, cutoff)
            predicted = unitary @ before
            predicted /= np.linalg.norm(predicted)
            overlap = np.vdot(predicted, actual)
            fidelity = float(np.clip(abs(overlap) ** 2, 0, 1))
            state_vector_norm_error = float(np.linalg.norm(actual - predicted * (overlap.conjugate() / abs(overlap))))

            rows.append(
                {
                    "gamma": gamma,
                    "cutoff": cutoff,
                    "fidelity_to_independent_reference": fidelity,
                    "state_vector_norm_error_phase_aligned": state_vector_norm_error,
                    "trace_distance_estimate": float(np.sqrt(max(0.0, 1 - fidelity))),
                    "retained_norm": state.retained_norms[-1] if state.retained_norms else None,
                }
            )

    common.save_result(rows, "R22_cubic_phase_direct", extra={"alpha": [ALPHA.real, ALPHA.imag]})

    fig, ax = plt.subplots(figsize=(6, 4))
    for gamma in GAMMAS:
        subset = [r for r in rows if r["gamma"] == gamma]
        ax.semilogy(
            [r["cutoff"] for r in subset], [max(1e-16, 1 - r["fidelity_to_independent_reference"]) for r in subset],
            "o-", label=f"gamma={gamma}",
        )
    ax.set_xlabel("Fock cutoff")
    ax.set_ylabel("1 - fidelity (independent reference)")
    ax.legend(fontsize=8)
    ax.set_title(f"Cubic phase, coherent input alpha={ALPHA}")
    common.save_figure(fig, "R22_cubic_phase_direct")
    plt.close(fig)

    common.print_summary(
        "R22 cubic phase direct execution",
        cases=len(rows),
        min_fidelity=min(r["fidelity_to_independent_reference"] for r in rows),
        max_fidelity=max(r["fidelity_to_independent_reference"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R22 cubic phase direct execution ===
  cases: 16
  min_fidelity: 1.0
  max_fidelity: 1.0


## R23: Cubic MBQC injection (finite-resource approximation -- clearly labelled).

Uses ``photographiq.non_gaussian.cubic_injection(gamma, squeezing=r)`` with a
FIXED ancilla homodyne outcome m=0 (so the nonlinear feed-forward corrections
QuadraticPhase(-2*gamma*m) and Displace(p=-gamma*m^2) vanish identically,
isolating the residual finite-resource Gaussian envelope). The result is
compared against an ideal direct CubicPhase(gamma) applied to the same
truncated input, using the same independent NumPy/SciPy matrix-exponential
reference as R22. Per docs/non_gaussian.md this injection is a conditional
filter times a cubic phase, NOT an exact finite-energy unitary cubic gate;
fidelity to the ideal gate is expected to improve as resource squeezing r
grows (the envelope widens toward the infinite-squeezing/no-filter limit).

In [25]:
"""R23: Cubic MBQC injection (finite-resource approximation -- clearly labelled).

Uses ``photographiq.non_gaussian.cubic_injection(gamma, squeezing=r)`` with a
FIXED ancilla homodyne outcome m=0 (so the nonlinear feed-forward corrections
QuadraticPhase(-2*gamma*m) and Displace(p=-gamma*m^2) vanish identically,
isolating the residual finite-resource Gaussian envelope). The result is
compared against an ideal direct CubicPhase(gamma) applied to the same
truncated input, using the same independent NumPy/SciPy matrix-exponential
reference as R22. Per docs/non_gaussian.md this injection is a conditional
filter times a cubic phase, NOT an exact finite-energy unitary cubic gate;
fidelity to the ideal gate is expected to improve as resource squeezing r
grows (the envelope widens toward the infinite-squeezing/no-filter limit).
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from scipy.linalg import expm

import common  # noqa: E402

import photographiq as pg  # noqa: E402

GAMMA = 0.15
ALPHA = 0.2 + 0.0j
SQUEEZING_VALUES = [0.1, 0.3, 0.6, 1.0, 1.5, 2.0]
CUTOFFS = [24, 32, 48]
FIXED_OUTCOME = 0.0


def independent_cubic_unitary(gamma: float, cutoff: int) -> np.ndarray:
    a = np.diag(np.sqrt(np.arange(1, cutoff)), k=1)
    q = a + a.T
    return expm(1j * gamma * (q @ q @ q) / 6)


def truncated_input_vector(cutoff: int) -> np.ndarray:
    pattern = pg.Pattern(inputs=(0,)).append(pg.Output((0,))).validate()
    state = pg.simulate(
        pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0
    ).state
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    return np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])


def vector_from_state(state, node, cutoff):
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    return np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])


def main():
    plt = common.setup_style()
    rows = []
    for cutoff in CUTOFFS:
        before = truncated_input_vector(cutoff)
        ideal_after = independent_cubic_unitary(GAMMA, cutoff) @ before
        ideal_after /= np.linalg.norm(ideal_after)

        for r in SQUEEZING_VALUES:
            pattern = pg.non_gaussian.cubic_injection(GAMMA, squeezing=r)
            failure = common.safe_cutoff_run(
                lambda: pg.simulate(
                    pattern, backend="piquasso-fock", cutoff=cutoff,
                    inputs={"in": pg.GaussianInput.coherent(ALPHA)},
                    measurement_outcomes={"m": FIXED_OUTCOME}, seed=0,
                ),
                label=f"cutoff={cutoff} r={r}",
            )
            if not failure["ok"]:
                rows.append(
                    {"gamma": GAMMA, "resource_squeezing": r, "cutoff": cutoff, "ok": False, "error": failure["error"]}
                )
                continue
            result = failure["value"]
            actual = vector_from_state(result.state, "in", cutoff)
            overlap = np.vdot(ideal_after, actual)
            fidelity = float(np.clip(abs(overlap) ** 2, 0, 1))
            rows.append(
                {
                    "gamma": GAMMA,
                    "resource_squeezing": r,
                    "cutoff": cutoff,
                    "ok": True,
                    "fidelity_to_ideal_cubic_phase": fidelity,
                    "trace_distance_estimate": float(np.sqrt(max(0.0, 1 - fidelity))),
                    "outcome_density": result.measurement_statistics["m"]["value"],
                    "outcome_kind": result.measurement_statistics["m"]["kind"],
                    "retained_norm": result.state.retained_norms[-1] if result.state.retained_norms else None,
                }
            )

    common.save_result(rows, "R23_cubic_injection", extra={"gamma": GAMMA, "alpha": [ALPHA.real, ALPHA.imag], "fixed_outcome": FIXED_OUTCOME})

    ok_rows = [r for r in rows if r["ok"]]
    fig, ax = plt.subplots(figsize=(6, 4))
    for cutoff in CUTOFFS:
        subset = [r for r in ok_rows if r["cutoff"] == cutoff]
        ax.plot([r["resource_squeezing"] for r in subset], [r["fidelity_to_ideal_cubic_phase"] for r in subset], "o-", label=f"cutoff={cutoff}")
    ax.set_xlabel("Resource squeezing r")
    ax.set_ylabel("Fidelity to ideal CubicPhase(gamma)")
    ax.legend(fontsize=8)
    ax.set_title(f"Finite-resource cubic injection, gamma={GAMMA} (m fixed at 0)")
    common.save_figure(fig, "R23_cubic_injection")
    plt.close(fig)

    common.print_summary(
        "R23 cubic MBQC injection (finite-resource approximation)",
        cases=len(rows),
        ok=len(ok_rows),
        min_fidelity=min((r["fidelity_to_ideal_cubic_phase"] for r in ok_rows), default=None),
        max_fidelity=max((r["fidelity_to_ideal_cubic_phase"] for r in ok_rows), default=None),
    )


if __name__ == "__main__":
    main()


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999865; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99996303; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99999994; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99999474; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99965619; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99999979; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99981295; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999992; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999471; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.9999992; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99944778; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99996586; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99999998; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)
D:\Research\QML\results

D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.9999998; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99999997; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99991102; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99999964; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99999854; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))



=== R23 cubic MBQC injection (finite-resource approximation) ===
  cases: 18
  ok: 8
  min_fidelity: 0.9469294912977205
  max_fidelity: 0.988996020664211


## R24: Kerr direct execution.

Applies ``photographiq.commands.Kerr(kappa)`` (native Piquasso
``exp(i*kappa*n^2)``) to the explicit custom superposition
``(|0>+|1>+|2>)/sqrt(3)`` (``FockInput`` with arbitrary amplitudes) and
compares each output amplitude against the exact analytic phase rule
``|n> -> exp(i*kappa*n^2)|n>``, computed independently with NumPy.

In [26]:
"""R24: Kerr direct execution.

Applies ``photographiq.commands.Kerr(kappa)`` (native Piquasso
``exp(i*kappa*n^2)``) to the explicit custom superposition
``(|0>+|1>+|2>)/sqrt(3)`` (``FockInput`` with arbitrary amplitudes) and
compares each output amplitude against the exact analytic phase rule
``|n> -> exp(i*kappa*n^2)|n>``, computed independently with NumPy.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

KAPPAS = np.linspace(-2.0, 2.0, 17)
CUTOFF = 8


def main():
    plt = common.setup_style()
    amplitudes_in = np.array([1, 1, 1], dtype=complex) / np.sqrt(3)
    fock_input = pg.FockInput(tuple(amplitudes_in))

    rows = []
    for kappa in KAPPAS:
        pattern = pg.Pattern(inputs=(0,)).append(pg.Kerr(0, float(kappa))).append(pg.Output((0,))).validate()
        state = pg.simulate(pattern, backend="piquasso-fock", cutoff=CUTOFF, inputs={0: fock_input}, seed=0).state
        actual_map = dict(zip(state.basis, state.state_vector, strict=True))
        actual = np.array([actual_map.get((n,), 0j) for n in range(CUTOFF)])

        expected = np.zeros(CUTOFF, dtype=complex)
        for n in range(len(amplitudes_in)):
            expected[n] = amplitudes_in[n] * np.exp(1j * kappa * n**2)

        overlap = np.vdot(expected, actual)
        fidelity = float(np.clip(abs(overlap) ** 2, 0, 1))
        vector_error = float(np.linalg.norm(actual - expected))
        rows.append(
            {
                "kappa": float(kappa),
                "fidelity_to_exact_phase_rule": fidelity,
                "state_vector_error": vector_error,
            }
        )

    tolerance_fidelity = 1 - 1e-10
    tolerance_vector = 1e-8
    failures = [r for r in rows if r["fidelity_to_exact_phase_rule"] < tolerance_fidelity or r["state_vector_error"] > tolerance_vector]
    if failures:
        common.save_csv(failures, "R24_kerr_direct_FAILURES")
        raise AssertionError(f"{len(failures)} Kerr cases disagreed with the exact phase rule; see FAILURES csv")

    common.save_result(rows, "R24_kerr_direct", extra={"input_amplitudes": [[a.real, a.imag] for a in amplitudes_in], "cutoff": CUTOFF})

    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    ax.semilogy([r["kappa"] for r in rows], [max(1e-16, r["state_vector_error"]) for r in rows], "o-", color="#24677b")
    ax.set_xlabel(r"Kerr coefficient $\kappa$")
    ax.set_ylabel("State-vector error vs. exact phase rule")
    ax.set_title(r"Kerr on $(|0\rangle+|1\rangle+|2\rangle)/\sqrt{3}$")
    common.save_figure(fig, "R24_kerr_direct")
    plt.close(fig)

    common.print_summary(
        "R24 Kerr direct execution",
        kappas=len(rows),
        max_state_vector_error=max(r["state_vector_error"] for r in rows),
        min_fidelity=min(r["fidelity_to_exact_phase_rule"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R24 Kerr direct execution ===
  kappas: 17
  max_state_vector_error: 2.0062885096040707e-16
  min_fidelity: 1.0


## R25: Kerr synthesis convergence (approximate; synthesis-only, no MBQC noise).

``photographiq.synthesis.synthesize_kerr`` returns an ordinary ``Circuit``
of native rotation/quadratic-phase/cubic-phase gates approximating
exp(i*kappa*n^2) by a Lie product formula. To isolate the product-formula
approximation error from separate finite-resource MBQC injection error
(covered by R23), this script translates the synthesized circuit's gates
directly into PHYSICAL Fock-backend commands (Rotate/QuadraticPhase/
CubicPhase/Displace) and executes them without compiling to an MBQC pattern.
Fidelity to the exact phase rule is compared across synthesis_steps.

In [27]:
"""R25: Kerr synthesis convergence (approximate; synthesis-only, no MBQC noise).

``photographiq.synthesis.synthesize_kerr`` returns an ordinary ``Circuit``
of native rotation/quadratic-phase/cubic-phase gates approximating
exp(i*kappa*n^2) by a Lie product formula. To isolate the product-formula
approximation error from separate finite-resource MBQC injection error
(covered by R23), this script translates the synthesized circuit's gates
directly into PHYSICAL Fock-backend commands (Rotate/QuadraticPhase/
CubicPhase/Displace) and executes them without compiling to an MBQC pattern.
Fidelity to the exact phase rule is compared across synthesis_steps.
"""

from __future__ import annotations

import sys
import time
import warnings
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402
from photographiq.synthesis import synthesize_kerr  # noqa: E402

KAPPA = 0.05
CUTOFF = 16
STEPS = [1, 2, 4, 8, 16, 32, 64]


def physical_command_for(name, mode, params):
    """Translate one synthesized Circuit gate into a direct physical command."""
    if name == "displace":
        return pg.Displace(mode, *params)
    if name == "cubic_phase":
        return pg.CubicPhase(mode, params[0])
    if name == "symplectic":
        matrix = np.asarray(params[0])
        if np.allclose(matrix, [[1.0, 0.0], [matrix[1, 0], 1.0]], atol=1e-12):
            return pg.QuadraticPhase(mode, float(matrix[1, 0]))
        if np.allclose(matrix @ matrix.T, np.eye(2), atol=1e-10):
            angle = float(np.arctan2(matrix[1, 0], matrix[0, 0]))
            return pg.Rotate(mode, angle)
        raise NotImplementedError("Unexpected symplectic gate shape in Kerr synthesis output")
    raise NotImplementedError(f"Unexpected synthesized gate kind: {name}")


def main():
    plt = common.setup_style()
    amplitudes_in = np.array([1, 1, 1], dtype=complex) / np.sqrt(3)
    fock_input = pg.FockInput(tuple(amplitudes_in))
    expected = np.zeros(CUTOFF, dtype=complex)
    for n in range(len(amplitudes_in)):
        expected[n] = amplitudes_in[n] * np.exp(1j * KAPPA * n**2)

    rows = []
    for steps in STEPS:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            circuit, report = synthesize_kerr(KAPPA, steps=steps, modes=1, mode=0)
            pattern = pg.Pattern(inputs=(0,))
            for name, modes, params in circuit.gates:
                pattern.append(physical_command_for(name, modes[0], params))
            pattern.append(pg.Output((0,)))
            pattern.validate()

            start = time.perf_counter()
            state = pg.simulate(pattern, backend="piquasso-fock", cutoff=CUTOFF, inputs={0: fock_input}, seed=0).state
            elapsed = time.perf_counter() - start

        actual_map = dict(zip(state.basis, state.state_vector, strict=True))
        actual = np.array([actual_map.get((n,), 0j) for n in range(CUTOFF)])
        overlap = np.vdot(expected, actual)
        fidelity = float(np.clip(abs(overlap) ** 2, 0, 1))
        rows.append(
            {
                "synthesis_steps": steps,
                "fidelity_to_exact_kerr": fidelity,
                "trace_distance_estimate": float(np.sqrt(max(0.0, 1 - fidelity))),
                "primitive_count": report.primitive_count,
                "synthesis_order": report.order,
                "runtime_seconds": elapsed,
                "retained_norm": state.retained_norms[-1] if state.retained_norms else None,
            }
        )
        print(f"  steps={steps}: fidelity={fidelity:.8f}, primitives={report.primitive_count}, {elapsed:.3f}s", flush=True)

    common.save_result(
        rows, "R25_kerr_synthesis",
        extra={"kappa": KAPPA, "cutoff": CUTOFF, "note": "synthesis-only execution; MBQC injection noise excluded (see R23)"},
    )

    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    ax.loglog([r["synthesis_steps"] for r in rows], [max(1e-16, 1 - r["fidelity_to_exact_kerr"]) for r in rows], "o-", color="#24677b")
    ax.set_xlabel("Synthesis steps")
    ax.set_ylabel("1 - fidelity to exact Kerr")
    ax.set_title(f"Kerr Lie-product synthesis, kappa={KAPPA} (direct physical execution)")
    common.save_figure(fig, "R25_kerr_synthesis")
    plt.close(fig)

    common.print_summary(
        "R25 Kerr synthesis convergence",
        steps=STEPS,
        min_infidelity=min(1 - r["fidelity_to_exact_kerr"] for r in rows),
        max_infidelity=max(1 - r["fidelity_to_exact_kerr"] for r in rows),
    )


if __name__ == "__main__":
    main()


  steps=1: fidelity=0.99640225, primitives=182, 0.771s


  steps=2: fidelity=0.99868848, primitives=364, 1.524s


  steps=4: fidelity=0.99944510, primitives=728, 3.057s


  steps=8: fidelity=0.99974762, primitives=1456, 5.899s


  steps=16: fidelity=0.99988070, primitives=2912, 12.133s


  steps=32: fidelity=0.99994234, primitives=5824, 23.911s


  steps=64: fidelity=0.99997177, primitives=11648, 48.502s



=== R25 Kerr synthesis convergence ===
  steps: [1, 2, 4, 8, 16, 32, 64]
  min_infidelity: 2.823460216117013e-05
  max_infidelity: 0.0035977532365163434


## R26: GKP resource characterization.

Generates finite-energy square-lattice GKP resources
(``photographiq.gkp.GKPResource``), characterizes their q-quadrature
probability structure (comparing the intended smooth target wavefunction to
its finite-Fock projection, reconstructed from the Hermite basis functions
the package itself uses -- ``photographiq.fock_measurements.wavefunctions``),
stabilizer expectations (``photographiq.gkp.stabilizers``), logical-codeword
overlap (``GKPCode.gram``), and normalization/cutoff sensitivity via
``GKPResource.project``. This makes NO fault-tolerance or threshold claim.

In [28]:
"""R26: GKP resource characterization.

Generates finite-energy square-lattice GKP resources
(``photographiq.gkp.GKPResource``), characterizes their q-quadrature
probability structure (comparing the intended smooth target wavefunction to
its finite-Fock projection, reconstructed from the Hermite basis functions
the package itself uses -- ``photographiq.fock_measurements.wavefunctions``),
stabilizer expectations (``photographiq.gkp.stabilizers``), logical-codeword
overlap (``GKPCode.gram``), and normalization/cutoff sensitivity via
``GKPResource.project``. This makes NO fault-tolerance or threshold claim.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402
from photographiq.fock_measurements import wavefunctions  # noqa: E402

PARAMETER_SETS = [
    {"peak_width": 0.3, "envelope": 0.3, "label": "sharp"},
    {"peak_width": 0.4, "envelope": 0.4, "label": "default"},
    {"peak_width": 0.6, "envelope": 0.6, "label": "fuzzy"},
]
DEFAULT_CUTOFF = 48
CUTOFF_SENSITIVITY = [24, 32, 48, 64, 96]


def reconstructed_probability(amplitudes: np.ndarray, q_grid: np.ndarray) -> np.ndarray:
    """|sum_n c_n phi_n(q)|^2 using the package's own Hermite wavefunctions."""
    cutoff = len(amplitudes)
    basis = np.array([wavefunctions(x, cutoff) for x in q_grid])  # (len(q), cutoff)
    values = basis @ amplitudes
    return np.abs(values) ** 2


def main():
    plt = common.setup_style()
    rows = []
    representative_data = None

    for params in PARAMETER_SETS:
        for logical in (0, 1):
            resource = pg.GKPResource(logical, peak_width=params["peak_width"], envelope=params["envelope"])
            fock_input, captured_mass = resource.project(DEFAULT_CUTOFF)

            pattern = pg.Pattern(inputs=(0,)).append(pg.Output((0,))).validate()
            state = pg.simulate(pattern, backend="piquasso-fock", cutoff=DEFAULT_CUTOFF, inputs={0: fock_input}, seed=0).state
            stabilizer_values = pg.gkp.stabilizers(state)

            q_target, psi_target = resource.wavefunction()
            q_grid = np.linspace(q_target.min(), q_target.max(), 2000)
            target_probability = np.interp(q_grid, q_target, psi_target**2)
            fock_probability = reconstructed_probability(np.array(fock_input.amplitudes), q_grid)
            l1_distance = float(np.trapezoid(np.abs(target_probability - fock_probability), q_grid))

            rows.append(
                {
                    "label": params["label"],
                    "peak_width": params["peak_width"],
                    "envelope": params["envelope"],
                    "logical": logical,
                    "cutoff": DEFAULT_CUTOFF,
                    "captured_mass": captured_mass,
                    "q_translation_stabilizer_abs": abs(stabilizer_values["q_translation"]),
                    "p_translation_stabilizer_abs": abs(stabilizer_values["p_translation"]),
                    "target_vs_fock_probability_l1": l1_distance,
                    "mean_photon_number": state.photon_number(0),
                }
            )
            if params["label"] == "default" and logical == 0:
                representative_data = (q_grid, target_probability, fock_probability, params)

        code = pg.GKPCode(peak_width=params["peak_width"], envelope=params["envelope"], cutoff=DEFAULT_CUTOFF)
        rows.append(
            {
                "label": params["label"], "peak_width": params["peak_width"], "envelope": params["envelope"],
                "logical": "overlap|0><1|", "cutoff": DEFAULT_CUTOFF, "captured_mass": None,
                "q_translation_stabilizer_abs": None, "p_translation_stabilizer_abs": None,
                "target_vs_fock_probability_l1": None,
                "mean_photon_number": None,
                "logical_overlap_abs": abs(code.gram[0, 1]),
            }
        )

    cutoff_rows = []
    for cutoff in CUTOFF_SENSITIVITY:
        resource = pg.GKPResource(0, peak_width=0.4, envelope=0.4)
        _, mass = resource.project(cutoff)
        cutoff_rows.append({"cutoff": cutoff, "captured_mass": mass})

    common.save_result(rows, "R26_gkp_resource")
    common.save_csv(cutoff_rows, "R26_gkp_resource_cutoff_sensitivity")

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    q_grid, target_probability, fock_probability, params = representative_data
    axes[0].plot(q_grid, target_probability, label="Target wavefunction |psi(q)|^2", color="#24677b")
    axes[0].plot(q_grid, fock_probability, "--", label=f"Fock reconstruction (c={DEFAULT_CUTOFF})", color="#c66d27")
    axes[0].set_xlabel("q")
    axes[0].set_ylabel("Probability density")
    axes[0].legend(fontsize=7)
    axes[0].set_title(f"GKP |0>_L, {params['label']} (width={params['peak_width']})")

    axes[1].semilogy(
        [r["cutoff"] for r in cutoff_rows], [max(1e-16, 1 - r["captured_mass"]) for r in cutoff_rows], "o-", color="#627a36"
    )
    axes[1].set_xlabel("Fock cutoff")
    axes[1].set_ylabel("1 - captured mass")
    axes[1].set_title("Cutoff sensitivity, peak_width=envelope=0.4")
    fig.suptitle("GKP resource characterization (no fault-tolerance claim)")
    common.save_figure(fig, "R26_gkp_resource")
    plt.close(fig)

    numeric_rows = [r for r in rows if r["captured_mass"] is not None]
    common.print_summary(
        "R26 GKP resource characterization",
        cases=len(rows),
        min_captured_mass=min(r["captured_mass"] for r in numeric_rows),
        max_stabilizer_deviation_from_unity=max(1 - r["q_translation_stabilizer_abs"] for r in numeric_rows),
    )


if __name__ == "__main__":
    main()



=== R26 GKP resource characterization ===
  cases: 9
  min_captured_mass: 0.9998082959646712
  max_stabilizer_deviation_from_unity: 0.7959531570831201


## R27: GKP logical shifts/stabilizers.

Applies logical lattice displacements (``photographiq.gkp.logical_displacement``,
X: q -> q+sqrt(2*pi), Z: p -> p+sqrt(2*pi)) to finite-energy GKP codewords and
checks the resulting stabilizer expectations and decoded bit against the
ideal square-lattice relations, computed independently with NumPy. Finite
energy means these are approximate, not exact, permutations of the codeword.

In [29]:
"""R27: GKP logical shifts/stabilizers.

Applies logical lattice displacements (``photographiq.gkp.logical_displacement``,
X: q -> q+sqrt(2*pi), Z: p -> p+sqrt(2*pi)) to finite-energy GKP codewords and
checks the resulting stabilizer expectations and decoded bit against the
ideal square-lattice relations, computed independently with NumPy. Finite
energy means these are approximate, not exact, permutations of the codeword.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402
from photographiq.gkp import SPACING, logical_displacement, stabilizers  # noqa: E402

CUTOFF = 48
PEAK_WIDTH = ENVELOPE = 0.35


def apply_logical_gate(fock_input, gate, cutoff):
    pattern = pg.Pattern(inputs=(0,)).append(gate).append(pg.Output((0,))).validate()
    return pg.simulate(pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: fock_input}, seed=0).state


def main():
    plt = common.setup_style()
    rows = []
    decoder = pg.NearestCellDecoder()

    for logical in (0, 1):
        resource = pg.GKPResource(logical, peak_width=PEAK_WIDTH, envelope=ENVELOPE)
        fock_input, mass = resource.project(CUTOFF)
        base_state = apply_logical_gate(fock_input, pg.Displace(0, q=0.0, p=0.0), CUTOFF)
        base_stabilizers = stabilizers(base_state)

        for gate_name in ("X", "Z"):
            shifted_state = apply_logical_gate(fock_input, logical_displacement(0, gate_name), CUTOFF)
            shifted_stabilizers = stabilizers(shifted_state)

            # Independent expectation: an X shift (q -> q+SPACING) multiplies the
            # p-translation stabilizer phase exp(i*SPACING*q) by
            # exp(i*SPACING^2) = exp(i*2*pi) = 1 (identity, since SPACING^2=2*pi);
            # a Z shift analogously leaves q_translation's phase unchanged, apart
            # from finite-energy/truncation deviations from the ideal comb.
            independent_phase_prediction = complex(np.exp(1j * SPACING**2)) if gate_name == "X" else complex(np.exp(-1j * SPACING**2))

            rows.append(
                {
                    "logical": logical,
                    "gate": gate_name,
                    "cutoff": CUTOFF,
                    "captured_mass": mass,
                    "base_q_stabilizer_abs": abs(base_stabilizers["q_translation"]),
                    "base_p_stabilizer_abs": abs(base_stabilizers["p_translation"]),
                    "shifted_q_stabilizer_abs": abs(shifted_stabilizers["q_translation"]),
                    "shifted_p_stabilizer_abs": abs(shifted_stabilizers["p_translation"]),
                    "independent_phase_prediction_real": independent_phase_prediction.real,
                    "independent_phase_prediction_imag": independent_phase_prediction.imag,
                }
            )

    # Decode a family of raw homodyne-like outcomes around the ideal 0/1 combs
    # and check the nearest-cell rule against an independent formula.
    decode_rows = []
    for outcome in np.linspace(-3 * SPACING, 3 * SPACING, 61):
        result = decoder.decode(float(outcome))
        independent_cell = int(np.floor(outcome / SPACING + 0.5))
        independent_residual = float(outcome - independent_cell * SPACING)
        independent_bit = independent_cell % 2
        decode_rows.append(
            {
                "outcome": float(outcome),
                "decoded_bit": result.bit,
                "independent_bit": independent_bit,
                "residual": result.residual,
                "independent_residual": independent_residual,
                "residual_error": abs(result.residual - independent_residual),
                "bit_match": result.bit == independent_bit,
            }
        )

    bit_failures = [r for r in decode_rows if not r["bit_match"] or r["residual_error"] > 1e-9]
    if bit_failures:
        common.save_csv(bit_failures, "R27_gkp_stabilizers_decode_FAILURES")
        raise AssertionError("Nearest-cell decoder disagreed with the independent floor-based formula; see FAILURES csv")

    common.save_result(rows, "R27_gkp_stabilizers", extra={"cutoff": CUTOFF, "peak_width": PEAK_WIDTH, "envelope": ENVELOPE})
    common.save_csv(decode_rows, "R27_gkp_stabilizers_decode_sweep")

    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.plot([r["outcome"] for r in decode_rows], [r["decoded_bit"] for r in decode_rows], "o", markersize=3, color="#24677b")
    for k in range(-3, 4):
        ax.axvline((k + 0.5) * SPACING, color="grey", linestyle=":", linewidth=0.7)
    ax.set_xlabel("Raw homodyne outcome")
    ax.set_ylabel("Decoded logical bit")
    ax.set_title("Nearest-cell GKP decoding")
    common.save_figure(fig, "R27_gkp_stabilizers")
    plt.close(fig)

    common.print_summary(
        "R27 GKP logical shifts/stabilizers",
        cases=len(rows),
        decode_points=len(decode_rows),
        min_base_stabilizer=min(min(r["base_q_stabilizer_abs"], r["base_p_stabilizer_abs"]) for r in rows),
    )


if __name__ == "__main__":
    main()


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99993094; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99994217; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99993299; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99990485; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))



=== R27 GKP logical shifts/stabilizers ===
  cases: 4
  decode_points: 61
  min_base_stabilizer: 0.6760063670187486


## R28: GKP syndrome extraction (finite-energy/Fock approximation).

Prepares a finite-energy GKP |0>_L codeword, physically displaces it by a
known small q-shift, then runs the package's actual one-ancilla SUM syndrome
extraction (``photographiq.gkp.correction_pattern``) with a finite "logical
plus" ancilla. A single trajectory's raw ancilla homodyne outcome samples one
of many comb peaks of the (also finite-energy) ancilla, so the decoded
residual has substantial single-shot noise set by the resource's intrinsic
peak width; this script instead reports the ENSEMBLE (circular) mean and
spread over repeated seeded trajectories, which is the physically meaningful
way to test whether extraction is centered on the true displacement. This is
a finite-ancilla, finite-energy approximation, not a fault-tolerant claim.

In [30]:
"""R28: GKP syndrome extraction (finite-energy/Fock approximation).

Prepares a finite-energy GKP |0>_L codeword, physically displaces it by a
known small q-shift, then runs the package's actual one-ancilla SUM syndrome
extraction (``photographiq.gkp.correction_pattern``) with a finite "logical
plus" ancilla. A single trajectory's raw ancilla homodyne outcome samples one
of many comb peaks of the (also finite-energy) ancilla, so the decoded
residual has substantial single-shot noise set by the resource's intrinsic
peak width; this script instead reports the ENSEMBLE (circular) mean and
spread over repeated seeded trajectories, which is the physically meaningful
way to test whether extraction is centered on the true displacement. This is
a finite-ancilla, finite-energy approximation, not a fault-tolerant claim.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402
from photographiq.gkp import SPACING, decode_shift  # noqa: E402

CUTOFF = 80  # two GKP-scale modes share one total-photon cutoff (input + ancilla)
PEAK_WIDTH = ENVELOPE = 0.35
DELTAS = np.linspace(-0.6, 0.6, 9)
N_SEEDS = 16


def displaced_fock_vector(fock_input, delta_q, cutoff):
    pattern = pg.Pattern(inputs=(0,)).append(pg.Displace(0, q=delta_q)).append(pg.Output((0,))).validate()
    state = pg.simulate(pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: fock_input}, seed=0).state
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    vector = np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])
    vector /= np.linalg.norm(vector)
    return pg.FockInput(tuple(vector))


def circular_stats(residuals: np.ndarray, spacing: float):
    angles = residuals / spacing * 2 * np.pi
    mean_vector = np.mean(np.exp(1j * angles))
    circular_mean = float(np.angle(mean_vector) * spacing / (2 * np.pi))
    circular_std = float(np.sqrt(max(0.0, -2 * np.log(abs(mean_vector)))) * spacing / (2 * np.pi))
    return circular_mean, circular_std


def main():
    plt = common.setup_style()
    code_zero = pg.GKPResource(0, peak_width=PEAK_WIDTH, envelope=ENVELOPE)
    fock_zero, _ = code_zero.project(CUTOFF)
    ancilla_plus = pg.gkp.superposition(1, 1, cutoff=CUTOFF, peak_width=PEAK_WIDTH, envelope=ENVELOPE)
    pattern = pg.gkp.correction_pattern(quadrature="q", resource=ancilla_plus)

    rows = []
    for delta in DELTAS:
        delta = float(delta)
        shifted_input = displaced_fock_vector(fock_zero, delta, CUTOFF)

        residuals, corrected_q, ok_seeds = [], [], 0
        for seed in range(N_SEEDS):
            failure = common.safe_cutoff_run(
                lambda seed=seed: pg.simulate(
                    pattern, backend="piquasso-fock", cutoff=CUTOFF, inputs={"in": shifted_input}, seed=3000 + 97 * seed
                ),
                label=f"delta={delta} seed={seed}",
            )
            if not failure["ok"]:
                continue
            result = failure["value"]
            residual, _ = decode_shift(result.records["syndrome"])
            residuals.append(residual)
            corrected_q.append(result.state.quadrature("in")[0])
            ok_seeds += 1

        expected_residual = float(((delta + SPACING / 2) % SPACING) - SPACING / 2)
        if residuals:
            circular_mean, circular_std = circular_stats(np.array(residuals), SPACING)
        else:
            circular_mean = circular_std = None
        rows.append(
            {
                "delta": delta,
                "ok_seeds": ok_seeds,
                "expected_residual": expected_residual,
                "circular_mean_decoded_residual": circular_mean,
                "circular_std_decoded_residual": circular_std,
                "circular_mean_error": None if circular_mean is None else abs(circular_mean - expected_residual),
                "mean_abs_corrected_q": float(np.mean(np.abs(corrected_q))) if corrected_q else None,
                "abs_uncorrected_q": abs(delta),
            }
        )

    ok_rows = [r for r in rows if r["ok_seeds"] > 0]
    # Statistical tolerance: circular_std / sqrt(N) sets the expected standard
    # error of the ensemble mean; a factor of 5 gives headroom against a
    # single flaky run while still catching a genuinely biased estimator.
    failures = [
        r for r in ok_rows
        if r["circular_mean_error"] > 5 * r["circular_std_decoded_residual"] / np.sqrt(max(1, r["ok_seeds"])) + 0.05
    ]
    if failures:
        common.save_csv(failures, "R28_gkp_syndrome_FAILURES")
        raise AssertionError(f"{len(failures)} deltas show a biased syndrome estimate beyond statistical tolerance; see FAILURES csv")

    common.save_result(rows, "R28_gkp_syndrome", extra={"cutoff": CUTOFF, "peak_width": PEAK_WIDTH, "envelope": ENVELOPE, "n_seeds": N_SEEDS})

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    axes[0].errorbar(
        [r["delta"] for r in ok_rows], [r["circular_mean_decoded_residual"] for r in ok_rows],
        yerr=[r["circular_std_decoded_residual"] / np.sqrt(r["ok_seeds"]) for r in ok_rows],
        fmt="o-", label=f"Ensemble mean decoded residual (n={N_SEEDS})", color="#24677b", capsize=3,
    )
    axes[0].plot([r["delta"] for r in ok_rows], [r["expected_residual"] for r in ok_rows], "--", label="Ideal (delta mod spacing)", color="grey")
    axes[0].set_xlabel("Imposed displacement delta")
    axes[0].set_ylabel("Decoded residual (ensemble mean +/- SEM)")
    axes[0].legend(fontsize=7)

    axes[1].plot([r["delta"] for r in ok_rows], [r["mean_abs_corrected_q"] for r in ok_rows], "o-", label="<|corrected mean q|>", color="#c66d27")
    axes[1].plot([r["delta"] for r in ok_rows], [r["abs_uncorrected_q"] for r in ok_rows], "--", label="|uncorrected q| = |delta|", color="grey")
    axes[1].set_xlabel("Imposed displacement delta")
    axes[1].set_ylabel("Residual |mean q| after correction")
    axes[1].legend(fontsize=7)
    fig.suptitle(f"GKP q-syndrome extraction, finite ancilla (cutoff={CUTOFF}, width={PEAK_WIDTH})")
    common.save_figure(fig, "R28_gkp_syndrome")
    plt.close(fig)

    common.print_summary(
        "R28 GKP syndrome extraction (ensemble statistics)",
        deltas=len(rows),
        seeds_per_delta=N_SEEDS,
        max_circular_mean_error=max((r["circular_mean_error"] for r in ok_rows), default=None),
        median_circular_std=float(np.median([r["circular_std_decoded_residual"] for r in ok_rows])) if ok_rows else None,
    )


if __name__ == "__main__":
    main()


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:238: RuntimeWarning: preparation: retained Fock norm 0.99999993; test a larger cutoff
  engine.prepare(c.node, value(c.squeezing), c.state)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99922338; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99999998; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99924706; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99999997; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:238: RuntimeWarning: preparation: retained Fock norm 0.99999994; test a larger cutoff
  engine.prepare(c.node, value(c.squeezing), c.state)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.9992523; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


...

D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99924599; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99924144; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.99999977; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:476: RuntimeWarning: Displacement: retained Fock norm 0.9999994; test a larger cutoff
  self._gate((node,), pq.Displacement(r=abs(alpha), phi=np.angle(alpha)))



=== R28 GKP syndrome extraction (ensemble statistics) ===
  deltas: 9
  seeds_per_delta: 16
  max_circular_mean_error: 0.053402588505609416
  median_circular_std: 0.37290041463438517


## R29: Finite squeezing noise scan (main-text result).

Fixes a compound logical circuit (single-mode rotate+squeeze, and a two-mode
CZ-entangled circuit) and sweeps the MBQC RESOURCE squeezing used to compile
it. For each resource squeezing value, extracts the exact unconditional
Gaussian channel (S, N, d) and reports: the logical-map error against an
independently built NumPy target matrix (expected to stay near machine
precision, since squeezing does not change the ideal map), Tr(N), the
spectral norm/max eigenvalue of the added-noise matrix N, and the resulting
output variance for a fixed coherent input.

In [31]:
"""R29: Finite squeezing noise scan (main-text result).

Fixes a compound logical circuit (single-mode rotate+squeeze, and a two-mode
CZ-entangled circuit) and sweeps the MBQC RESOURCE squeezing used to compile
it. For each resource squeezing value, extracts the exact unconditional
Gaussian channel (S, N, d) and reports: the logical-map error against an
independently built NumPy target matrix (expected to stay near machine
precision, since squeezing does not change the ideal map), Tr(N), the
spectral norm/max eigenvalue of the added-noise matrix N, and the resulting
output variance for a fixed coherent input.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

RESOURCE_SQUEEZING = np.geomspace(0.1, 3.0, 20)
LOGICAL_ANGLE = 0.5
LOGICAL_SQUEEZE = 0.3
CZ_WEIGHT = 0.6
INPUT_ALPHA = 0.3 + 0.2j


def independent_rotation(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def independent_squeezing(r):
    return np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])


def main():
    plt = common.setup_style()

    # Circuit.rotate() is applied before Circuit.squeeze() in construction order,
    # so the composed map is S(squeeze) @ R(angle) (squeeze acts on the already-
    # rotated state), not R @ S.
    single_mode_target = independent_squeezing(LOGICAL_SQUEEZE) @ independent_rotation(LOGICAL_ANGLE)
    two_mode_target = np.eye(4)
    two_mode_target[1, 2] = CZ_WEIGHT
    two_mode_target[3, 0] = CZ_WEIGHT

    single_rows, two_rows = [], []
    for r in RESOURCE_SQUEEZING:
        r = float(r)
        circuit = pg.Circuit(1).rotate(0, LOGICAL_ANGLE).squeeze(0, LOGICAL_SQUEEZE)
        pattern = circuit.compile(squeezing=r)
        channel = pg.gaussian_channel(pattern)
        input_state = pg.GaussianInput.coherent(INPUT_ALPHA)
        applied = channel.apply(input_state.state(0))
        single_rows.append(
            {
                "resource_squeezing": r,
                "map_frobenius_error": common.frobenius_error(channel.matrix, single_mode_target),
                "noise_trace": float(np.trace(channel.noise)),
                "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
                "noise_max_eigenvalue": float(np.linalg.eigvalsh(channel.noise).max()),
                "output_q_variance": float(applied.covariance[0, 0]),
                "output_p_variance": float(applied.covariance[1, 1]),
            }
        )

        circuit2 = pg.Circuit(2).cz(0, 1, CZ_WEIGHT)
        pattern2 = circuit2.compile(squeezing=r)
        channel2 = pg.gaussian_channel(pattern2)
        two_rows.append(
            {
                "resource_squeezing": r,
                "map_frobenius_error": common.frobenius_error(channel2.matrix, two_mode_target),
                "noise_trace": float(np.trace(channel2.noise)),
                "noise_spectral_norm": float(np.linalg.norm(channel2.noise, ord=2)),
                "noise_max_eigenvalue": float(np.linalg.eigvalsh(channel2.noise).max()),
            }
        )

    tolerance = 1e-7
    failures = [r for r in single_rows + two_rows if r["map_frobenius_error"] > tolerance]
    if failures:
        common.save_csv(failures, "R29_finite_squeezing_FAILURES")
        raise AssertionError("Resource squeezing altered the ideal logical map beyond tolerance; see FAILURES csv")

    common.save_result(single_rows, "R29_finite_squeezing_single_mode", extra={"angle": LOGICAL_ANGLE, "squeeze": LOGICAL_SQUEEZE})
    common.save_result(two_rows, "R29_finite_squeezing_cz", extra={"weight": CZ_WEIGHT})

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
    rs = [row["resource_squeezing"] for row in single_rows]
    axes[0].loglog(rs, [row["noise_trace"] for row in single_rows], "o-", label="Rotate+Squeeze circuit", color="#24677b")
    axes[0].loglog(rs, [row["noise_trace"] for row in two_rows], "s-", label="CZ circuit", color="#c66d27")
    axes[0].set_xlabel("Resource squeezing r")
    axes[0].set_ylabel("Tr(N): total added variance")
    axes[0].legend(fontsize=8)

    axes[1].semilogx(rs, [row["output_q_variance"] for row in single_rows], "o-", label="Output q variance", color="#627a36")
    axes[1].semilogx(rs, [row["output_p_variance"] for row in single_rows], "s-", label="Output p variance", color="#a83279")
    axes[1].set_xlabel("Resource squeezing r")
    axes[1].set_ylabel("Output variance (rotate+squeeze circuit)")
    axes[1].legend(fontsize=8)
    fig.suptitle("Finite resource squeezing: MBQC compilation noise floor")
    common.save_figure(fig, "R29_finite_squeezing")
    plt.close(fig)

    common.print_summary(
        "R29 finite squeezing noise scan (main-text)",
        resource_squeezing_points=len(RESOURCE_SQUEEZING),
        max_map_error=max(r["map_frobenius_error"] for r in single_rows + two_rows),
        noise_trace_range=(min(r["noise_trace"] for r in single_rows), max(r["noise_trace"] for r in single_rows)),
    )


if __name__ == "__main__":
    main()



=== R29 finite squeezing noise scan (main-text) ===
  resource_squeezing_points: 20
  max_map_error: 5.978733960281817e-16
  noise_trace_range: (0.02724161871677198, 8.997894673376232)


## R30: Loss / attenuation validation.

Sweeps intensity transmissivity eta through ``photographiq.commands.Loss``
on both Gaussian backends. For a coherent input, checks the exact analytic
relations alpha_out = sqrt(eta)*alpha_in and <n>_out = eta*<n>_in (thermal
occupation held at zero), computed independently with NumPy.

In [32]:
"""R30: Loss / attenuation validation.

Sweeps intensity transmissivity eta through ``photographiq.commands.Loss``
on both Gaussian backends. For a coherent input, checks the exact analytic
relations alpha_out = sqrt(eta)*alpha_in and <n>_out = eta*<n>_in (thermal
occupation held at zero), computed independently with NumPy.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

ETAS = np.linspace(0.05, 1.0, 20)
ALPHA = 0.6 + 0.4j


def main():
    plt = common.setup_style()
    rows = []
    for eta in ETAS:
        eta = float(eta)
        pattern = pg.Pattern(inputs=(0,)).append(pg.Loss(0, eta, 0.0)).append(pg.Output((0,))).validate()
        expected_alpha = np.sqrt(eta) * ALPHA
        expected_mean = np.array([2 * expected_alpha.real, 2 * expected_alpha.imag])
        expected_n = eta * abs(ALPHA) ** 2

        row = {"eta": eta}
        for backend in ("gaussian", "piquasso"):
            state = pg.simulate(pattern, backend=backend, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0).state
            row[f"{backend}_mean_error"] = common.frobenius_error(state.mean, expected_mean)
            row[f"{backend}_photon_number_error"] = abs(state.photon_number(0) - expected_n)
            row[f"{backend}_photon_number"] = state.photon_number(0)
        row["expected_photon_number"] = expected_n
        rows.append(row)

    tolerance = 1e-9
    failures = [
        r for r in rows
        if r["gaussian_mean_error"] > tolerance or r["piquasso_mean_error"] > tolerance
        or r["gaussian_photon_number_error"] > tolerance or r["piquasso_photon_number_error"] > tolerance
    ]
    if failures:
        common.save_csv(failures, "R30_loss_validation_FAILURES")
        raise AssertionError("Loss channel disagreed with the analytic attenuation formulas; see FAILURES csv")

    common.save_result(rows, "R30_loss_validation", extra={"alpha": [ALPHA.real, ALPHA.imag]})

    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    etas = [r["eta"] for r in rows]
    ax.plot(etas, [r["expected_photon_number"] for r in rows], "-", label="Analytic: eta*|alpha|^2", color="grey")
    ax.plot(etas, [r["gaussian_photon_number"] for r in rows], "o", markersize=4, label="NumPy Gaussian backend", color="#24677b")
    ax.plot(etas, [r["piquasso_photon_number"] for r in rows], "x", markersize=5, label="Piquasso backend", color="#c66d27")
    ax.set_xlabel("Transmissivity eta")
    ax.set_ylabel("Output mean photon number")
    ax.legend(fontsize=8)
    common.save_figure(fig, "R30_loss_validation")
    plt.close(fig)

    common.print_summary(
        "R30 loss / attenuation validation",
        eta_points=len(rows),
        max_mean_error=max(max(r["gaussian_mean_error"], r["piquasso_mean_error"]) for r in rows),
        max_photon_error=max(max(r["gaussian_photon_number_error"], r["piquasso_photon_number_error"]) for r in rows),
    )


if __name__ == "__main__":
    main()



=== R30 loss / attenuation validation ===
  eta_points: 20
  max_mean_error: 4.965068306494546e-16
  max_photon_error: 3.3306690738754696e-16


## R31: Thermal noise validation.

Sweeps transmissivity eta and thermal environment occupation nbar through
``photographiq.commands.Loss`` on both Gaussian backends, and compares the
resulting mean/covariance to the independent analytic thermal-attenuation
channel: mean -> sqrt(eta)*mean_in, V -> eta*V_in + (1-eta)*(2*nbar+1)*I
(statistical covariance, vacuum V=I), written directly with NumPy.

In [33]:
"""R31: Thermal noise validation.

Sweeps transmissivity eta and thermal environment occupation nbar through
``photographiq.commands.Loss`` on both Gaussian backends, and compares the
resulting mean/covariance to the independent analytic thermal-attenuation
channel: mean -> sqrt(eta)*mean_in, V -> eta*V_in + (1-eta)*(2*nbar+1)*I
(statistical covariance, vacuum V=I), written directly with NumPy.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

ETAS = [0.2, 0.4, 0.6, 0.8, 1.0]
NBARS = [0.0, 0.3, 0.8, 1.5, 3.0]
ALPHA = 0.5 - 0.3j


def analytic_thermal_channel(eta: float, nbar: float, mean_in: np.ndarray, cov_in: np.ndarray):
    mean_out = np.sqrt(eta) * mean_in
    cov_out = eta * cov_in + (1 - eta) * (2 * nbar + 1) * np.eye(2)
    return mean_out, cov_out


def main():
    plt = common.setup_style()
    input_state = pg.GaussianInput.coherent(ALPHA)
    mean_in = np.array(input_state.mean)
    cov_in = np.array(input_state.covariance)

    rows = []
    for eta in ETAS:
        for nbar in NBARS:
            pattern = pg.Pattern(inputs=(0,)).append(pg.Loss(0, eta, nbar)).append(pg.Output((0,))).validate()
            expected_mean, expected_cov = analytic_thermal_channel(eta, nbar, mean_in, cov_in)

            row = {"eta": eta, "nbar": nbar}
            for backend in ("gaussian", "piquasso"):
                state = pg.simulate(pattern, backend=backend, inputs={0: input_state}, seed=0).state
                row[f"{backend}_mean_error"] = common.frobenius_error(state.mean, expected_mean)
                row[f"{backend}_covariance_error"] = common.frobenius_error(state.covariance, expected_cov)
            rows.append(row)

    tolerance = 1e-9
    failures = [
        r for r in rows
        if r["gaussian_mean_error"] > tolerance or r["piquasso_mean_error"] > tolerance
        or r["gaussian_covariance_error"] > tolerance or r["piquasso_covariance_error"] > tolerance
    ]
    if failures:
        common.save_csv(failures, "R31_thermal_noise_FAILURES")
        raise AssertionError("Thermal loss channel disagreed with the analytic formula; see FAILURES csv")

    common.save_result(rows, "R31_thermal_noise", extra={"alpha": [ALPHA.real, ALPHA.imag]})

    fig, ax = plt.subplots(figsize=(6, 4))
    for nbar in NBARS:
        subset = [r for r in rows if r["nbar"] == nbar]
        variances = []
        for r in subset:
            _, cov = analytic_thermal_channel(r["eta"], r["nbar"], mean_in, cov_in)
            variances.append(cov[0, 0])
        ax.plot([r["eta"] for r in subset], variances, "o-", label=f"nbar={nbar}")
    ax.set_xlabel("Transmissivity eta")
    ax.set_ylabel("Output q variance")
    ax.legend(fontsize=8, title="Thermal occupation")
    ax.set_title("Thermal attenuation channel (analytic curves)")
    common.save_figure(fig, "R31_thermal_noise")
    plt.close(fig)

    common.print_summary(
        "R31 thermal noise validation",
        cases=len(rows),
        max_mean_error=max(max(r["gaussian_mean_error"], r["piquasso_mean_error"]) for r in rows),
        max_covariance_error=max(max(r["gaussian_covariance_error"], r["piquasso_covariance_error"]) for r in rows),
    )


if __name__ == "__main__":
    main()



=== R31 thermal noise validation ===
  cases: 25
  max_mean_error: 4.577566798522237e-16
  max_covariance_error: 1.88411095042053e-15


## R32: Detector inefficiency / noisy homodyne.

Noisy homodyne (``Homodyne(efficiency, noise)``) is supported on both
Gaussian backends and experimentally on the mixed-Fock backend; the pure
Fock backend explicitly rejects it (feature matrix, docs/validation/
feature-matrix.md). This script verifies the documented added-variance
formula nu = (1-efficiency)/efficiency + noise (docs/theory.md,
"Destructive Gaussian measurements") against sampled outcome variance on the
NumPy Gaussian backend, then separately cross-checks a modest mixed-Fock case
against the exact Gaussian prediction (a finite-cutoff approximation).

In [34]:
"""R32: Detector inefficiency / noisy homodyne.

Noisy homodyne (``Homodyne(efficiency, noise)``) is supported on both
Gaussian backends and experimentally on the mixed-Fock backend; the pure
Fock backend explicitly rejects it (feature matrix, docs/validation/
feature-matrix.md). This script verifies the documented added-variance
formula nu = (1-efficiency)/efficiency + noise (docs/theory.md,
"Destructive Gaussian measurements") against sampled outcome variance on the
NumPy Gaussian backend, then separately cross-checks a modest mixed-Fock case
against the exact Gaussian prediction (a finite-cutoff approximation).
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SHOTS = 6000
SEED = 20260913
ALPHA = 0.4 + 0.2j
EFFICIENCIES = [1.0, 0.9, 0.7, 0.5]
NOISES = [0.0, 0.2, 0.6]
MIXED_FOCK_CUTOFF = 10
# The mixed-Fock noisy-homodyne path convolves a dense density matrix with a
# Gaussian electronics kernel via scipy.integrate.quad_vec inside a brentq
# root-find, PER SHOT, whenever noise!=0: measured at ~8s/shot at cutoff=10
# (and >13s/shot at cutoff=20) on the reference machine. Shot counts below
# are deliberately small for the noisy cases and are a coarse cross-check,
# not a high-precision statistical estimate; the exact Gaussian-backend sweep
# above is the statistically powered result.
MIXED_FOCK_SHOTS_NOISELESS = 200
MIXED_FOCK_SHOTS_NOISY = 6


def independent_variance(prior_variance: float, efficiency: float, noise: float) -> float:
    """Documented formula: nu=(1-eta)/eta + noise, outcomes calibrated to the incident quadrature."""
    return prior_variance + (1 - efficiency) / efficiency + noise


def main():
    plt = common.setup_style()
    rows = []
    for efficiency in EFFICIENCIES:
        for noise in NOISES:
            pattern = pg.Pattern(inputs=(0,)).append(
                pg.Measure(0, pg.Homodyne(angle=0.0, efficiency=efficiency, noise=noise), key="m")
            )
            shots = pg.run_shots(
                pattern, SHOTS, backend="gaussian", inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=SEED
            )
            samples = shots.values("m")
            empirical_variance = float(np.var(samples, ddof=1))
            predicted_variance = independent_variance(1.0, efficiency, noise)  # coherent input: prior q variance = 1
            # Sampling error of a variance estimate from N draws scales like
            # variance*sqrt(2/(N-1)); use a 6-sigma band for a robust, non-flaky check.
            sampling_error = predicted_variance * np.sqrt(2 / (SHOTS - 1))
            rows.append(
                {
                    "efficiency": efficiency,
                    "noise": noise,
                    "empirical_variance": empirical_variance,
                    "predicted_variance": predicted_variance,
                    "absolute_error": abs(empirical_variance - predicted_variance),
                    "sampling_error_6sigma": 6 * sampling_error,
                    "within_tolerance": abs(empirical_variance - predicted_variance) <= 6 * sampling_error,
                }
            )

    failures = [r for r in rows if not r["within_tolerance"]]
    if failures:
        common.save_csv(failures, "R32_noisy_measurement_FAILURES")
        raise AssertionError(f"{len(failures)} noisy-homodyne cases exceeded the statistical tolerance; see FAILURES csv")

    # Mixed-Fock cross-check (experimental, finite-cutoff approximation).
    mixed_rows = []
    for efficiency in (1.0, 0.8):
        for noise in (0.0, 0.3):
            pattern = pg.Pattern(inputs=(0,)).append(
                pg.Measure(0, pg.Homodyne(angle=0.0, efficiency=efficiency, noise=noise), key="m")
            )
            shots = MIXED_FOCK_SHOTS_NOISELESS if noise == 0.0 else MIXED_FOCK_SHOTS_NOISY
            failure = common.safe_cutoff_run(
                lambda efficiency=efficiency, noise=noise, shots=shots: pg.run_shots(
                    pattern, shots, backend="piquasso-mixed-fock", cutoff=MIXED_FOCK_CUTOFF,
                    inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=SEED,
                ),
                label=f"mixed-fock eff={efficiency} noise={noise}",
            )
            if not failure["ok"]:
                mixed_rows.append({"efficiency": efficiency, "noise": noise, "ok": False, "error": failure["error"]})
                continue
            samples = failure["value"].values("m")
            empirical_variance = float(np.var(samples, ddof=1))
            predicted_variance = independent_variance(1.0, efficiency, noise)
            mixed_rows.append(
                {
                    "efficiency": efficiency, "noise": noise, "ok": True,
                    "empirical_variance": empirical_variance, "predicted_variance": predicted_variance,
                    "absolute_error": abs(empirical_variance - predicted_variance),
                    "cutoff": MIXED_FOCK_CUTOFF, "shots": shots,
                }
            )

    common.save_result(
        rows, "R32_noisy_measurement",
        extra={"shots": SHOTS, "alpha": [ALPHA.real, ALPHA.imag], "mixed_fock": mixed_rows},
    )
    common.save_csv(mixed_rows, "R32_noisy_measurement_mixed_fock")

    fig, ax = plt.subplots(figsize=(6, 4))
    for noise in NOISES:
        subset = [r for r in rows if r["noise"] == noise]
        ax.plot([r["efficiency"] for r in subset], [r["predicted_variance"] for r in subset], "-", color="grey")
        ax.plot([r["efficiency"] for r in subset], [r["empirical_variance"] for r in subset], "o", label=f"noise={noise}")
    ax.set_xlabel("Detector efficiency")
    ax.set_ylabel("Outcome variance")
    ax.legend(fontsize=8)
    ax.set_title("Noisy homodyne: sampled vs. documented formula (lines=analytic)")
    common.save_figure(fig, "R32_noisy_measurement")
    plt.close(fig)

    common.print_summary(
        "R32 detector inefficiency / noisy homodyne",
        cases=len(rows),
        max_absolute_error=max(r["absolute_error"] for r in rows),
        mixed_fock_ok=sum(1 for r in mixed_rows if r["ok"]),
    )


if __name__ == "__main__":
    main()



=== R32 detector inefficiency / noisy homodyne ===
  cases: 12
  max_absolute_error: 0.0037851122324386033
  mixed_fock_ok: 4


## R33: Fock cutoff convergence.

Runs representative non-Gaussian workloads (cat state, photon subtraction,
cubic phase, GKP resource) at strictly increasing Fock cutoffs using
``photographiq.convergence.cutoff_convergence``, and reports retained norm,
boundary population, photon number, adjacent-cutoff fidelity/trace distance
(from the study itself) plus fidelity to the HIGHEST tested cutoff (computed
separately here). The highest tested cutoff is a reference point, not an
exact/infinite-cutoff ground truth -- convergence must still be judged from
the trend, not assumed from the last value's smallness alone.

In [35]:
"""R33: Fock cutoff convergence.

Runs representative non-Gaussian workloads (cat state, photon subtraction,
cubic phase, GKP resource) at strictly increasing Fock cutoffs using
``photographiq.convergence.cutoff_convergence``, and reports retained norm,
boundary population, photon number, adjacent-cutoff fidelity/trace distance
(from the study itself) plus fidelity to the HIGHEST tested cutoff (computed
separately here). The highest tested cutoff is a reference point, not an
exact/infinite-cutoff ground truth -- convergence must still be judged from
the trend, not assumed from the last value's smallness alone.
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402

import photographiq as pg  # noqa: E402

WORKLOADS = {
    "cat_state": dict(
        pattern=pg.Pattern(inputs=()).append(pg.Prepare(0, state=pg.CatResource(1.5, parity=1))).append(pg.Output((0,))),
        cutoffs=[16, 24, 32, 48, 64],
        node=0,
        kwargs={},
    ),
    "photon_subtraction": dict(
        pattern=pg.non_gaussian.photon_subtraction(0.3),
        cutoffs=[8, 12, 16, 24, 32],
        node="in",
        kwargs={"inputs": {"in": pg.FockInput.number(2)}, "measurement_outcomes": {"count": 1}},
    ),
    "cubic_phase": dict(
        pattern=pg.Pattern(inputs=(0,)).append(pg.CubicPhase(0, 0.15)).append(pg.Output((0,))),
        cutoffs=[16, 24, 32, 48, 64],
        node=0,
        kwargs={"inputs": {0: pg.GaussianInput.coherent(0.3 + 0.0j)}},
    ),
    "gkp_resource": dict(
        pattern=pg.Pattern(inputs=()).append(pg.Prepare(0, state=pg.GKPResource(0, peak_width=0.4, envelope=0.4))).append(pg.Output((0,))),
        cutoffs=[24, 32, 48, 64, 96],
        node=0,
        kwargs={},
    ),
}


def main():
    plt = common.setup_style()
    all_rows = {}
    for name, spec in WORKLOADS.items():
        study = pg.cutoff_convergence(spec["pattern"], spec["cutoffs"], **spec["kwargs"])
        final_state = study.results[-1].state
        rows = []
        for row, result in zip(study.rows, study.results, strict=True):
            fidelity_to_highest = pg.fidelity(result.state, final_state) if row["cutoff"] != spec["cutoffs"][-1] else 1.0
            trace_distance_to_highest = pg.trace_distance(result.state, final_state) if row["cutoff"] != spec["cutoffs"][-1] else 0.0
            rows.append(
                {
                    "workload": name,
                    "cutoff": row["cutoff"],
                    "dimension": common.fock_dimension(len(result.state.nodes), row["cutoff"]),
                    "seconds": row["seconds"],
                    "norm": row["norm"],
                    "minimum_retained_norm": row["minimum_retained_norm"],
                    "maximum_boundary_population": row["maximum_boundary_population"],
                    "photon_number": row["photon_numbers"].get(spec["node"]),
                    "fidelity_to_previous": row["fidelity_to_previous"],
                    "trace_distance_to_previous": row["trace_distance_to_previous"],
                    "fidelity_to_highest_cutoff": fidelity_to_highest,
                    "trace_distance_to_highest_cutoff": trace_distance_to_highest,
                }
            )
        all_rows[name] = rows
        print(f"  {name}: " + ", ".join(f"c={r['cutoff']}:F={r['fidelity_to_highest_cutoff']:.6f}" for r in rows), flush=True)

    flat_rows = [row for rows in all_rows.values() for row in rows]
    common.save_result(flat_rows, "R33_fock_cutoff", extra={"workloads": list(WORKLOADS)})

    fig, axes = plt.subplots(1, len(WORKLOADS), figsize=(4.2 * len(WORKLOADS), 3.6))
    for ax, (name, rows) in zip(axes, all_rows.items(), strict=True):
        infidelities = [max(1e-16, 1 - r["fidelity_to_highest_cutoff"]) for r in rows[:-1]]
        ax.semilogy([r["cutoff"] for r in rows[:-1]], infidelities, "o-", color="#24677b")
        ax.set_xlabel("Fock cutoff")
        ax.set_title(name)
    axes[0].set_ylabel(f"1 - fidelity to cutoff={list(WORKLOADS.values())[0]['cutoffs'][-1]}")
    fig.suptitle("Fock cutoff convergence (highest cutoff is a reference, not exact truth)")
    common.save_figure(fig, "R33_fock_cutoff")
    plt.close(fig)

    common.print_summary(
        "R33 Fock cutoff convergence",
        workloads=list(WORKLOADS),
        min_final_fidelity=min(rows[-2]["fidelity_to_highest_cutoff"] for rows in all_rows.values() if len(rows) > 1),
    )


if __name__ == "__main__":
    main()


  cat_state: c=16:F=1.000000, c=24:F=1.000000, c=32:F=1.000000, c=48:F=1.000000, c=64:F=1.000000


  photon_subtraction: c=8:F=1.000000, c=12:F=1.000000, c=16:F=1.000000, c=24:F=1.000000, c=32:F=1.000000


  cubic_phase: c=16:F=1.000000, c=24:F=1.000000, c=32:F=1.000000, c=48:F=1.000000, c=64:F=1.000000


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:238: RuntimeWarning: preparation: retained Fock norm 0.99911485; test a larger cutoff
  engine.prepare(c.node, value(c.squeezing), c.state)


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:238: RuntimeWarning: preparation: retained Fock norm 0.99998599; test a larger cutoff
  engine.prepare(c.node, value(c.squeezing), c.state)


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\simulator.py:238: RuntimeWarning: preparation: retained Fock norm 0.99999987; test a larger cutoff
  engine.prepare(c.node, value(c.squeezing), c.state)


  gkp_resource: c=24:F=0.999115, c=32:F=0.999986, c=48:F=1.000000, c=64:F=1.000000, c=96:F=1.000000



=== R33 Fock cutoff convergence ===
  workloads: ['cat_state', 'photon_subtraction', 'cubic_phase', 'gkp_resource']
  min_final_fidelity: 0.9999999974475406


## R34: Cubic injection convergence -- 2D scan over cutoff and resource squeezing.

Two-dimensional grid of (Fock cutoff, resource squeezing) for
``photographiq.non_gaussian.cubic_injection`` with a fixed ancilla outcome
m=0 (isolating the residual finite-resource Gaussian envelope from outcome-
dependent feed-forward). Fidelity to an ideal direct CubicPhase(gamma),
computed independently with NumPy/SciPy, is saved as a heatmap-ready CSV.

In [36]:
"""R34: Cubic injection convergence -- 2D scan over cutoff and resource squeezing.

Two-dimensional grid of (Fock cutoff, resource squeezing) for
``photographiq.non_gaussian.cubic_injection`` with a fixed ancilla outcome
m=0 (isolating the residual finite-resource Gaussian envelope from outcome-
dependent feed-forward). Fidelity to an ideal direct CubicPhase(gamma),
computed independently with NumPy/SciPy, is saved as a heatmap-ready CSV.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from scipy.linalg import expm

import common  # noqa: E402

import photographiq as pg  # noqa: E402

GAMMA = 0.15
ALPHA = 0.2 + 0.0j
CUTOFFS = [24, 32, 48, 64, 96]
SQUEEZING_VALUES = [0.1, 0.3, 0.6, 1.0, 1.5, 2.0]
FIXED_OUTCOME = 0.0


def independent_cubic_unitary(gamma, cutoff):
    a = np.diag(np.sqrt(np.arange(1, cutoff)), k=1)
    q = a + a.T
    return expm(1j * gamma * (q @ q @ q) / 6)


def truncated_input_vector(cutoff):
    pattern = pg.Pattern(inputs=(0,)).append(pg.Output((0,))).validate()
    state = pg.simulate(
        pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0
    ).state
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    return np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])


def main():
    plt = common.setup_style()
    rows = []
    for cutoff in CUTOFFS:
        before = truncated_input_vector(cutoff)
        ideal_after = independent_cubic_unitary(GAMMA, cutoff) @ before
        ideal_after /= np.linalg.norm(ideal_after)

        for r in SQUEEZING_VALUES:
            pattern = pg.non_gaussian.cubic_injection(GAMMA, squeezing=r)
            failure = common.safe_cutoff_run(
                lambda: pg.simulate(
                    pattern, backend="piquasso-fock", cutoff=cutoff,
                    inputs={"in": pg.GaussianInput.coherent(ALPHA)},
                    measurement_outcomes={"m": FIXED_OUTCOME}, seed=0,
                ),
                label=f"cutoff={cutoff} r={r}",
            )
            if not failure["ok"]:
                rows.append({"cutoff": cutoff, "resource_squeezing": r, "ok": False, "fidelity": None, "error": failure["error"]})
                continue
            result = failure["value"]
            amplitudes = dict(zip(result.state.basis, result.state.state_vector, strict=True))
            actual = np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])
            fidelity = float(np.clip(abs(np.vdot(ideal_after, actual)) ** 2, 0, 1))
            rows.append(
                {
                    "cutoff": cutoff, "resource_squeezing": r, "ok": True, "fidelity": fidelity,
                    "log10_infidelity": common.safe_log10(1 - fidelity),
                    "retained_norm": result.state.retained_norms[-1] if result.state.retained_norms else None,
                }
            )

    common.save_result(rows, "R34_cubic_convergence", extra={"gamma": GAMMA, "alpha": [ALPHA.real, ALPHA.imag]})

    ok_rows = [r for r in rows if r["ok"]]
    grid = np.full((len(SQUEEZING_VALUES), len(CUTOFFS)), np.nan)
    for r in ok_rows:
        i = SQUEEZING_VALUES.index(r["resource_squeezing"])
        j = CUTOFFS.index(r["cutoff"])
        grid[i, j] = r["fidelity"]

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    mesh = ax.imshow(grid, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(CUTOFFS)))
    ax.set_xticklabels(CUTOFFS)
    ax.set_yticks(range(len(SQUEEZING_VALUES)))
    ax.set_yticklabels(SQUEEZING_VALUES)
    ax.set_xlabel("Fock cutoff")
    ax.set_ylabel("Resource squeezing r")
    ax.set_title(f"Cubic injection fidelity to ideal gate, gamma={GAMMA} (m fixed at 0)")
    fig.colorbar(mesh, ax=ax, label="Fidelity")
    common.save_figure(fig, "R34_cubic_convergence")
    plt.close(fig)

    common.print_summary(
        "R34 cubic injection convergence (2D scan)",
        total_cells=len(rows),
        ok_cells=len(ok_rows),
        max_fidelity=max((r["fidelity"] for r in ok_rows), default=None),
        min_fidelity=min((r["fidelity"] for r in ok_rows), default=None),
    )


if __name__ == "__main__":
    main()


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999865; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99996303; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99999994; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99999474; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99965619; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99999979; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99981295; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999992; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999471; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.9999992; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99944778; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99996586; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99999998; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)
D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99990895; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))


...

D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:465: RuntimeWarning: GaussianTransform: retained Fock norm 0.99999861; test a larger cutoff
  self._gate((u, v), pq.GaussianTransform(passive=passive, active=active))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99999961; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:385: RuntimeWarning: preparation: retained Fock norm 0.99998717; test a larger cutoff
  self.prepare(node, squeezing=state.squeezing)


D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\src\photographiq\backends\fock.py:513: RuntimeWarning: CubicPhase: retained Fock norm 0.99995925; test a larger cutoff
  self._gate((node,), pq.CubicPhase(gamma=gamma))



=== R34 cubic injection convergence (2D scan) ===
  total_cells: 30
  ok_cells: 14
  max_fidelity: 0.9889973966811096
  min_fidelity: 0.9469294912977205


## R35: Fock-space dimension and memory model (analytical, cheap).

Uses the actual PhotoGraphiQ total-photon Fock dimension formula
D = binom(modes+cutoff-1, modes) (``PiquassoFockBackend.dimension``, cross-
checked against ``common.fock_dimension``) over a modes x cutoff grid, and
estimates pure-vector and dense-density-matrix byte costs. The default
``max_dimension`` allocation guard is read directly from the backend class.
Dense-density-matrix bytes are a rough dense upper-style estimate; the actual
mixed-Fock implementation may differ (see docs/performance.md).

In [37]:
"""R35: Fock-space dimension and memory model (analytical, cheap).

Uses the actual PhotoGraphiQ total-photon Fock dimension formula
D = binom(modes+cutoff-1, modes) (``PiquassoFockBackend.dimension``, cross-
checked against ``common.fock_dimension``) over a modes x cutoff grid, and
estimates pure-vector and dense-density-matrix byte costs. The default
``max_dimension`` allocation guard is read directly from the backend class.
Dense-density-matrix bytes are a rough dense upper-style estimate; the actual
mixed-Fock implementation may differ (see docs/performance.md).
"""

from __future__ import annotations

import inspect
import sys
from pathlib import Path

import numpy as np

import common  # noqa: E402

from photographiq.backends.fock import PiquassoFockBackend  # noqa: E402

MODES = list(range(1, 21))
CUTOFFS = list(range(2, 41))


def default_max_dimension() -> int:
    return inspect.signature(PiquassoFockBackend.__init__).parameters["max_dimension"].default


def main():
    plt = common.setup_style()
    max_dimension = default_max_dimension()

    backend = PiquassoFockBackend(cutoff=CUTOFFS[0])
    rows = []
    for modes in MODES:
        for cutoff in CUTOFFS:
            backend.cutoff = cutoff
            dimension_native = backend.dimension(modes)
            dimension_independent = common.fock_dimension(modes, cutoff)
            if dimension_native != dimension_independent:
                raise AssertionError(f"Dimension formulas disagree at modes={modes}, cutoff={cutoff}")
            rows.append(
                {
                    "modes": modes,
                    "cutoff": cutoff,
                    "dimension": dimension_native,
                    "pure_vector_bytes": common.vector_bytes(dimension_native),
                    "dense_density_matrix_bytes_rough_estimate": common.density_bytes(dimension_native),
                    "exceeds_default_max_dimension": dimension_native > max_dimension,
                }
            )

    common.save_result(rows, "R35_fock_dimension_scaling", extra={"default_max_dimension": max_dimension})

    dims = np.array([[common.fock_dimension(m, c) for c in CUTOFFS] for m in MODES])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    mesh = axes[0].imshow(
        np.log10(dims), aspect="auto", origin="lower", cmap="viridis",
        extent=[CUTOFFS[0], CUTOFFS[-1], MODES[0], MODES[-1]],
    )
    axes[0].set_xlabel("Cutoff")
    axes[0].set_ylabel("Modes")
    axes[0].set_title(r"$\log_{10}$(Fock dimension)")
    fig.colorbar(mesh, ax=axes[0])

    for modes in (1, 2, 4, 8, 16, 20):
        axes[1].semilogy(CUTOFFS, [common.vector_bytes(common.fock_dimension(modes, c)) for c in CUTOFFS], "o-", label=f"{modes} modes", markersize=3)
    axes[1].axhline(common.vector_bytes(max_dimension), color="red", linestyle="--", label="default max_dimension")
    axes[1].set_xlabel("Cutoff")
    axes[1].set_ylabel("Pure state-vector bytes")
    axes[1].legend(fontsize=6)
    fig.suptitle("Fock-space dimension and memory scaling (D = binom(modes+cutoff-1, modes))")
    common.save_figure(fig, "R35_fock_dimension_scaling")
    plt.close(fig)

    common.print_summary(
        "R35 Fock-space dimension and memory model",
        grid_points=len(rows),
        default_max_dimension=max_dimension,
        max_dimension_in_grid=max(r["dimension"] for r in rows),
        n_exceeding_default_guard=sum(r["exceeds_default_max_dimension"] for r in rows),
    )


if __name__ == "__main__":
    main()



=== R35 Fock-space dimension and memory model ===
  grid_points: 780
  default_max_dimension: 1000000
  max_dimension_in_grid: 2794563003870330
  n_exceeding_default_guard: 415


## R36: Gaussian runtime scaling.

Benchmarks preparing and entangling an N-mode line resource
(``CVGraph.line(N)``) through the NumPy Gaussian backend and the Piquasso
Gaussian backend, with warm-up runs and repeated timing measurements
(median, IQR, std). Timing is a local observation of this machine, not a
comparative performance claim (docs/performance.md, docs/validation.md).

In [38]:
"""R36: Gaussian runtime scaling.

Benchmarks preparing and entangling an N-mode line resource
(``CVGraph.line(N)``) through the NumPy Gaussian backend and the Piquasso
Gaussian backend, with warm-up runs and repeated timing measurements
(median, IQR, std). Timing is a local observation of this machine, not a
comparative performance claim (docs/performance.md, docs/validation.md).
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402
import metadata  # noqa: E402

import photographiq as pg  # noqa: E402

MODE_COUNTS = [10, 20, 50, 100, 200]


def build_pattern(n_modes: int) -> pg.Pattern:
    graph = pg.CVGraph.line(n_modes, squeezing=1.0, inputs=(0,))
    return pg.Pattern(graph)


def main():
    plt = common.setup_style()
    rows = []
    for n_modes in MODE_COUNTS:
        pattern = build_pattern(n_modes)
        for backend in ("gaussian", "piquasso"):
            warmup, repeats = (2, 9) if n_modes <= 50 else (1, 5)
            timing = common.benchmark(
                lambda pattern=pattern, backend=backend: pg.simulate(pattern, backend=backend, seed=0),
                warmup=warmup, repeats=repeats,
            )
            rows.append(
                {
                    "modes": n_modes,
                    "commands": len(pattern.commands),
                    "backend": backend,
                    "median_seconds": timing["median_seconds"],
                    "std_seconds": timing["std_seconds"],
                    "iqr_seconds": timing["iqr_seconds"],
                    "min_seconds": timing["min_seconds"],
                    "max_seconds": timing["max_seconds"],
                    "repeats": timing["repeats"],
                    "warmup": timing["warmup"],
                }
            )
            print(f"  modes={n_modes} backend={backend}: median={timing['median_seconds']:.5f}s", flush=True)

    common.save_result(rows, "R36_gaussian_runtime", extra={"environment": metadata.collect()})

    fig, ax = plt.subplots(figsize=(6, 4))
    for backend, color in (("gaussian", "#24677b"), ("piquasso", "#c66d27")):
        subset = [r for r in rows if r["backend"] == backend]
        ax.errorbar(
            [r["modes"] for r in subset], [r["median_seconds"] for r in subset],
            yerr=[r["iqr_seconds"] / 2 for r in subset], fmt="o-", label=backend, color=color, capsize=3,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of modes (line resource)")
    ax.set_ylabel("Median runtime (s)")
    ax.legend(fontsize=8)
    ax.set_title("Gaussian backend runtime scaling (this machine only)")
    common.save_figure(fig, "R36_gaussian_runtime")
    plt.close(fig)

    common.print_summary("R36 Gaussian runtime scaling", mode_counts=MODE_COUNTS)


if __name__ == "__main__":
    main()


  modes=10 backend=gaussian: median=0.01948s


  modes=10 backend=piquasso: median=0.07619s


  modes=20 backend=gaussian: median=0.04853s


  modes=20 backend=piquasso: median=0.22616s


  modes=50 backend=gaussian: median=0.46604s


...

  modes=100 backend=gaussian: median=4.90946s


  modes=100 backend=piquasso: median=19.65894s


  modes=200 backend=gaussian: median=49.67803s


  modes=200 backend=piquasso: median=193.57475s



=== R36 Gaussian runtime scaling ===
  mode_counts: [10, 20, 50, 100, 200]


## R37: Compiler runtime scaling (with warm-up and repeated timing).

Benchmarks ``Circuit.compile`` alone (not execution) for a growing number of
alternating rotation/CZ logical gates on two modes, using warm-up runs and
repeated ``time.perf_counter`` measurements (median, IQR, std). Records
generated resource nodes, commands and measurements alongside timing.

In [39]:
"""R37: Compiler runtime scaling (with warm-up and repeated timing).

Benchmarks ``Circuit.compile`` alone (not execution) for a growing number of
alternating rotation/CZ logical gates on two modes, using warm-up runs and
repeated ``time.perf_counter`` measurements (median, IQR, std). Records
generated resource nodes, commands and measurements alongside timing.
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402
import metadata  # noqa: E402

import photographiq as pg  # noqa: E402

GATE_COUNTS = [1, 2, 5, 10, 20, 50, 100]
SQUEEZING = 1.0


def build_circuit(n_gates: int) -> pg.Circuit:
    circuit = pg.Circuit(2)
    for i in range(n_gates):
        if i % 2 == 0:
            circuit.rotate(i % 2, 0.1 + 0.01 * i)
        else:
            circuit.cz(0, 1, 0.2 + 0.01 * i)
    return circuit


def main():
    plt = common.setup_style()
    rows = []
    for n_gates in GATE_COUNTS:
        circuit = build_circuit(n_gates)
        warmup, repeats = (2, 9) if n_gates <= 20 else (1, 5)
        timing = common.benchmark(
            lambda circuit=circuit: circuit.compile(squeezing=SQUEEZING, return_trace=True),
            warmup=warmup, repeats=repeats,
        )
        pattern, trace = timing["result"]
        measurements = sum(1 for c in pattern.commands if isinstance(c, pg.Measure))
        rows.append(
            {
                "n_gates": n_gates,
                "median_compile_seconds": timing["median_seconds"],
                "std_compile_seconds": timing["std_seconds"],
                "iqr_compile_seconds": timing["iqr_seconds"],
                "repeats": timing["repeats"],
                "warmup": timing["warmup"],
                "resource_nodes": len(pattern.graph.nodes) - circuit.modes,
                "commands": len(pattern.commands),
                "measurements": measurements,
            }
        )
        print(f"  n_gates={n_gates}: median={timing['median_seconds']:.5f}s, nodes={rows[-1]['resource_nodes']}", flush=True)

    common.save_result(rows, "R37_compiler_runtime", extra={"environment": metadata.collect()})

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    axes[0].errorbar(
        [r["n_gates"] for r in rows], [r["median_compile_seconds"] for r in rows],
        yerr=[r["iqr_compile_seconds"] / 2 for r in rows], fmt="o-", color="#24677b", capsize=3,
    )
    axes[0].set_xscale("log")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Number of logical gates")
    axes[0].set_ylabel("Median compile time (s)")

    axes[1].plot([r["n_gates"] for r in rows], [r["resource_nodes"] for r in rows], "o-", label="Resource nodes", color="#c66d27")
    axes[1].plot([r["n_gates"] for r in rows], [r["commands"] for r in rows], "s-", label="Commands", color="#627a36")
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Number of logical gates")
    axes[1].legend(fontsize=8)
    fig.suptitle("Compiler runtime scaling (this machine only)")
    common.save_figure(fig, "R37_compiler_runtime")
    plt.close(fig)

    common.print_summary("R37 compiler runtime scaling", gate_counts=GATE_COUNTS)


if __name__ == "__main__":
    main()


  n_gates=1: median=0.00380s, nodes=4


  n_gates=2: median=0.01043s, nodes=12


  n_gates=5: median=0.02397s, nodes=28


  n_gates=10: median=0.04524s, nodes=60


  n_gates=20: median=0.09806s, nodes=120


  n_gates=50: median=0.24026s, nodes=300


  n_gates=100: median=0.53090s, nodes=600



=== R37 compiler runtime scaling ===
  gate_counts: [1, 2, 5, 10, 20, 50, 100]


## R38: PhotoGraphiQ overhead versus raw Piquasso.

Builds matched N-mode Gaussian workloads (per-mode squeeze+rotate, plus a CZ
chain) and times two independent execution paths with warm-up and repeated
measurements: (1) one single raw ``piquasso`` program executed once with one
``GaussianSimulator`` call (the natural, most efficient native-engine usage),
and (2) the identical physical operations through
``photographiq.simulate(..., backend="piquasso")``, which executes one native
Program/Simulator per gate (see ``backends/piquasso.py``). Raw Piquasso is
the numerical engine reference; PhotoGraphiQ adds graph/pattern/MBQC
abstractions on top, and the ratio T_PG/T_PQ quantifies that added cost on
this machine (not a comparative benchmark claim across machines).

In [40]:
"""R38: PhotoGraphiQ overhead versus raw Piquasso.

Builds matched N-mode Gaussian workloads (per-mode squeeze+rotate, plus a CZ
chain) and times two independent execution paths with warm-up and repeated
measurements: (1) one single raw ``piquasso`` program executed once with one
``GaussianSimulator`` call (the natural, most efficient native-engine usage),
and (2) the identical physical operations through
``photographiq.simulate(..., backend="piquasso")``, which executes one native
Program/Simulator per gate (see ``backends/piquasso.py``). Raw Piquasso is
the numerical engine reference; PhotoGraphiQ adds graph/pattern/MBQC
abstractions on top, and the ratio T_PG/T_PQ quantifies that added cost on
this machine (not a comparative benchmark claim across machines).
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import piquasso as pq

import common  # noqa: E402
import metadata  # noqa: E402

import photographiq as pg  # noqa: E402

MODE_COUNTS = [5, 10, 20, 40]
HBAR = 2.0


def raw_piquasso_run(n_modes: int):
    with pq.Program() as program:
        pq.Q(*range(n_modes)) | pq.Vacuum()
        for i in range(n_modes):
            pq.Q(i) | pq.Squeezing(r=0.1 + 0.01 * i)
            pq.Q(i) | pq.Phaseshifter(phi=0.2 + 0.01 * i)
        for i in range(n_modes - 1):
            pq.Q(i, i + 1) | pq.ControlledZ(s=0.3)
    return pq.GaussianSimulator(d=n_modes, config=pq.Config(hbar=HBAR)).execute(program).state


def build_pattern(n_modes: int) -> pg.Pattern:
    pattern = pg.Pattern(inputs=tuple(range(n_modes)))
    for i in range(n_modes):
        pattern.append(pg.Squeeze(i, 0.1 + 0.01 * i))
        pattern.append(pg.Rotate(i, 0.2 + 0.01 * i))
    for i in range(n_modes - 1):
        pattern.append(pg.Entangle(i, i + 1, 0.3))
    pattern.append(pg.Output(tuple(range(n_modes))))
    return pattern.validate()


def main():
    plt = common.setup_style()
    rows = []
    for n_modes in MODE_COUNTS:
        pattern = build_pattern(n_modes)
        warmup, repeats = (2, 9) if n_modes <= 10 else (1, 5)

        raw_timing = common.benchmark(lambda n=n_modes: raw_piquasso_run(n), warmup=warmup, repeats=repeats)
        pg_timing = common.benchmark(
            lambda pattern=pattern: pg.simulate(pattern, backend="piquasso", seed=0), warmup=warmup, repeats=repeats
        )

        rows.append(
            {
                "modes": n_modes,
                "gates": 3 * n_modes - 1,
                "raw_piquasso_median_seconds": raw_timing["median_seconds"],
                "raw_piquasso_iqr_seconds": raw_timing["iqr_seconds"],
                "photographiq_median_seconds": pg_timing["median_seconds"],
                "photographiq_iqr_seconds": pg_timing["iqr_seconds"],
                "overhead_ratio": pg_timing["median_seconds"] / raw_timing["median_seconds"],
            }
        )
        print(f"  modes={n_modes}: raw={raw_timing['median_seconds']:.5f}s pg={pg_timing['median_seconds']:.5f}s ratio={rows[-1]['overhead_ratio']:.2f}", flush=True)

    common.save_result(rows, "R38_piquasso_overhead", extra={"environment": metadata.collect()})

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
    modes = [r["modes"] for r in rows]
    axes[0].loglog(modes, [r["raw_piquasso_median_seconds"] for r in rows], "o-", label="Raw Piquasso (1 program)", color="#24677b")
    axes[0].loglog(modes, [r["photographiq_median_seconds"] for r in rows], "s-", label="PhotoGraphiQ (piquasso backend)", color="#c66d27")
    axes[0].set_xlabel("Modes")
    axes[0].set_ylabel("Median runtime (s)")
    axes[0].legend(fontsize=7)

    axes[1].plot(modes, [r["overhead_ratio"] for r in rows], "o-", color="#627a36")
    axes[1].set_xlabel("Modes")
    axes[1].set_ylabel(r"Overhead ratio $T_{PG}/T_{PQ}$")
    fig.suptitle("PhotoGraphiQ overhead over raw Piquasso (this machine only)")
    common.save_figure(fig, "R38_piquasso_overhead")
    plt.close(fig)

    common.print_summary("R38 PhotoGraphiQ overhead vs. raw Piquasso", mode_counts=MODE_COUNTS)


if __name__ == "__main__":
    main()


  modes=5: raw=0.00731s pg=0.05245s ratio=7.17


  modes=10: raw=0.01294s pg=0.13048s ratio=10.08


  modes=20: raw=0.02606s pg=0.32116s ratio=12.33


  modes=40: raw=0.05072s pg=1.13880s ratio=22.45



=== R38 PhotoGraphiQ overhead vs. raw Piquasso ===
  mode_counts: [5, 10, 20, 40]


## R39: Serialization round-trip.

Serializes a nontrivial adaptive pattern (labelled nodes, ``Parameter`` and
``Outcome`` expression dependencies, an outcome-dependent measurement angle)
through the public versioned JSON API (``Pattern.to_json``/``Pattern.from_json``,
backed by ``photographiq.serialization``), and verifies the restored pattern
has structurally equal commands, preserved labels/expression dependencies,
and identical seeded execution behavior.

In [41]:
"""R39: Serialization round-trip.

Serializes a nontrivial adaptive pattern (labelled nodes, ``Parameter`` and
``Outcome`` expression dependencies, an outcome-dependent measurement angle)
through the public versioned JSON API (``Pattern.to_json``/``Pattern.from_json``,
backed by ``photographiq.serialization``), and verifies the restored pattern
has structurally equal commands, preserved labels/expression dependencies,
and identical seeded execution behavior.
"""

from __future__ import annotations

import sys
from pathlib import Path

import common  # noqa: E402

import photographiq as pg  # noqa: E402


def build_pattern():
    adjacency = [[0, 1, 0], [1, 0, 1], [0, 1, 0]]
    graph = pg.CVGraph.from_adjacency(adjacency, labels=("alice", "bob", "carol"), squeezing=1.0, inputs=("alice",))
    pattern = pg.Pattern(graph)
    pattern.measure("alice", pg.Homodyne.q(), key="m0")
    pattern.displace("bob", q=-pg.Outcome("m0"))
    pattern.measure("bob", pg.Homodyne(angle=0.1 * pg.Outcome("m0") + pg.Parameter("theta")), key="m1")
    pattern.displace("carol", q=-pg.Outcome("m1"), p=pg.Outcome("m0") + pg.Outcome("m1"))
    pattern.append(pg.Output(("carol",)))
    return pattern.validate()


def main():
    pattern = build_pattern()
    text = pattern.to_json()
    restored = pg.Pattern.from_json(text)

    checks = {
        "inputs_preserved": restored.inputs == pattern.inputs,
        "outputs_preserved": restored.outputs == pattern.outputs,
        "commands_structurally_equal": restored.commands == pattern.commands,
        "parameters_preserved": restored.parameters == pattern.parameters,
        "dependency_graph_edges_preserved": sorted(restored.dependencies().edges()) == sorted(pattern.dependencies().edges()),
    }

    original_result = pg.simulate(
        pattern, backend="gaussian", inputs={"alice": pg.GaussianInput.coherent(0.3 + 0.1j)},
        parameters={"theta": 0.4}, seed=7,
    )
    restored_result = pg.simulate(
        restored, backend="gaussian", inputs={"alice": pg.GaussianInput.coherent(0.3 + 0.1j)},
        parameters={"theta": 0.4}, seed=7,
    )
    checks["identical_seeded_outcomes"] = original_result.outcomes == restored_result.outcomes
    checks["identical_seeded_output_mean"] = common.frobenius_error(
        original_result.state.mean, restored_result.state.mean
    ) < 1e-12
    checks["identical_seeded_output_covariance"] = common.frobenius_error(
        original_result.state.covariance, restored_result.state.covariance
    ) < 1e-12

    failures = [name for name, ok in checks.items() if not ok]
    if failures:
        common.save_json({"failures": failures}, "R39_serialization_FAILURES")
        raise AssertionError(f"Serialization round-trip checks failed: {failures}")

    common.save_json(checks, "R39_serialization")
    common.write_metadata("R39_serialization")
    common.save_raw(text, "R39_serialization_pattern.json")

    common.print_summary("R39 serialization round-trip", **checks)


if __name__ == "__main__":
    main()



=== R39 serialization round-trip ===
  inputs_preserved: True
  outputs_preserved: True
  commands_structurally_equal: True
  parameters_preserved: True
  dependency_graph_edges_preserved: True
  identical_seeded_outcomes: True
  identical_seeded_output_mean: True
  identical_seeded_output_covariance: True


## R40: Seed reproducibility / shots.

Verifies that ``photographiq.run_shots`` gives bit-identical outcomes for
repeated runs with the same seed, and that two different seeds draw from
statistically indistinguishable distributions (a two-sample Kolmogorov-
Smirnov test) while producing different raw sample sequences.

In [42]:
"""R40: Seed reproducibility / shots.

Verifies that ``photographiq.run_shots`` gives bit-identical outcomes for
repeated runs with the same seed, and that two different seeds draw from
statistically indistinguishable distributions (a two-sample Kolmogorov-
Smirnov test) while producing different raw sample sequences.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from scipy import stats

import common  # noqa: E402

import photographiq as pg  # noqa: E402

SHOTS = 2000
ALPHA = 0.3 + 0.2j


def build_pattern():
    return pg.Pattern(inputs=(0,)).append(pg.Measure(0, pg.Homodyne.q(), key="m"))


def main():
    plt = common.setup_style()
    pattern = build_pattern()
    inputs = {0: pg.GaussianInput.coherent(ALPHA)}

    run_a = pg.run_shots(pattern, SHOTS, backend="gaussian", inputs=inputs, seed=42)
    run_b = pg.run_shots(pattern, SHOTS, backend="gaussian", inputs=inputs, seed=42)
    run_c = pg.run_shots(pattern, SHOTS, backend="gaussian", inputs=inputs, seed=43)

    samples_a, samples_b, samples_c = run_a.values("m"), run_b.values("m"), run_c.values("m")

    identical_same_seed = bool(np.array_equal(samples_a, samples_b))
    identical_cross_seed = bool(np.array_equal(samples_a, samples_c))

    ks_same_distribution = stats.ks_2samp(samples_a, samples_c)
    ks_self_consistency = stats.ks_2samp(samples_a, samples_b)  # trivially identical arrays -> statistic 0

    mean_a, mean_c = float(np.mean(samples_a)), float(np.mean(samples_c))
    theoretical_mean = 2 * ALPHA.real  # q-quadrature mean of a coherent state

    summary = {
        "shots": SHOTS,
        "identical_outcomes_same_seed": identical_same_seed,
        "identical_outcomes_different_seeds": identical_cross_seed,
        "ks_statistic_seed42_vs_seed43": float(ks_same_distribution.statistic),
        "ks_pvalue_seed42_vs_seed43": float(ks_same_distribution.pvalue),
        "ks_statistic_seed42_vs_itself": float(ks_self_consistency.statistic),
        "sample_mean_seed42": mean_a,
        "sample_mean_seed43": mean_c,
        "theoretical_mean": theoretical_mean,
        "mean_error_seed42": abs(mean_a - theoretical_mean),
        "mean_error_seed43": abs(mean_c - theoretical_mean),
    }

    checks = {
        "same_seed_is_bit_identical": identical_same_seed,
        "different_seeds_are_not_identical": not identical_cross_seed,
        "different_seeds_pass_ks_same_distribution": ks_same_distribution.pvalue > 0.01,
        "sample_means_close_to_theory": summary["mean_error_seed42"] < 0.1 and summary["mean_error_seed43"] < 0.1,
    }
    failures = [name for name, ok in checks.items() if not ok]
    if failures:
        common.save_json({"summary": summary, "failures": failures}, "R40_seeded_shots_FAILURES")
        raise AssertionError(f"Seed reproducibility checks failed: {failures}")

    common.save_json({**summary, "checks": checks}, "R40_seeded_shots")
    common.write_metadata("R40_seeded_shots")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(samples_a, bins=40, alpha=0.6, label="seed=42", color="#24677b", density=True)
    ax.hist(samples_c, bins=40, alpha=0.6, label="seed=43", color="#c66d27", density=True)
    ax.axvline(theoretical_mean, color="black", linestyle="--", label="Theoretical mean")
    ax.set_xlabel("Homodyne outcome (q)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.set_title(f"Seeded shot reproducibility (n={SHOTS} each)")
    common.save_figure(fig, "R40_seeded_shots")
    plt.close(fig)

    common.print_summary("R40 seed reproducibility / shots", **checks)


if __name__ == "__main__":
    main()



=== R40 seed reproducibility / shots ===
  same_seed_is_bit_identical: True
  different_seeds_are_not_identical: True
  different_seeds_pass_ks_same_distribution: True
  sample_means_close_to_theory: True


## R41: JAX deterministic gradient validation (experimental; runs only if JAX is installed).

For rotation, displacement, squeezing, cubic phase, Kerr and loss gates
(each with one JAX-differentiable ``Parameter``), compares
``jax.grad`` of ``photographiq.autodiff.expectation`` against central finite
differences of the SAME function at several step sizes h, per the package's
documented validation approach (docs/validation/feature-matrix.md:
"Estimator convergence: Four finite-difference steps"). This checks internal
autodiff/finite-difference consistency of the differentiable Fock generator,
not agreement with an external analytic formula.

In [43]:
"""R41: JAX deterministic gradient validation (experimental; runs only if JAX is installed).

For rotation, displacement, squeezing, cubic phase, Kerr and loss gates
(each with one JAX-differentiable ``Parameter``), compares
``jax.grad`` of ``photographiq.autodiff.expectation`` against central finite
differences of the SAME function at several step sizes h, per the package's
documented validation approach (docs/validation/feature-matrix.md:
"Estimator convergence: Four finite-difference steps"). This checks internal
autodiff/finite-difference consistency of the differentiable Fock generator,
not agreement with an external analytic formula.
"""

from __future__ import annotations
try:

    import sys
    from pathlib import Path

    import common  # noqa: E402

    if not common.HAS_JAX:
        print("SKIPPED: jax is not installed; see requirements_notes.md")
        sys.exit(0)

    import jax  # noqa: E402

    jax.config.update("jax_enable_x64", True)
    import jax.numpy as jnp  # noqa: E402

    import photographiq as pg  # noqa: E402
    from photographiq.autodiff import expectation  # noqa: E402

    CUTOFF = 12
    ALPHA = 0.3 + 0.2j
    T0 = 0.4
    STEP_SIZES = [1e-2, 1e-3, 1e-4, 1e-5]

    GATES = {
        "rotation": (lambda t: pg.Rotate(0, t), "q"),
        "displacement": (lambda t: pg.Displace(0, q=t), "q"),
        "squeezing": (lambda t: pg.Squeeze(0, t), "photon_number"),
        "cubic_phase": (lambda t: pg.CubicPhase(0, t), "p"),
        "kerr": (lambda t: pg.Kerr(0, t), "q"),
        "loss": (lambda t: pg.Loss(0, t, 0.0), "photon_number"),
    }


    def main():
        plt = common.setup_style()
        inputs = {0: pg.GaussianInput.coherent(ALPHA)}
        rows = []

        for name, (gate_builder, observable) in GATES.items():
            gate = gate_builder(pg.Parameter("t"))
            pattern = pg.Pattern(inputs=(0,)).append(gate).append(pg.Output((0,)))
            # Keep the loss transmissivity in (0,1]: evaluate around t0 scaled down.
            t0 = T0 if name != "loss" else 0.5

            def f(t, pattern=pattern, observable=observable):
                return expectation(pattern, {"t": t}, cutoff=CUTOFF, inputs=inputs, observable=observable)

            grad_value = float(jax.grad(f)(jnp.asarray(t0)))
            for h in STEP_SIZES:
                fd_value = float((f(t0 + h) - f(t0 - h)) / (2 * h))
                relative_error = abs(grad_value - fd_value) / max(abs(grad_value), 1e-12)
                rows.append(
                    {
                        "gate": name,
                        "observable": observable,
                        "t0": t0,
                        "step_size": h,
                        "jax_gradient": grad_value,
                        "finite_difference_gradient": fd_value,
                        "absolute_error": abs(grad_value - fd_value),
                        "relative_error": relative_error,
                    }
                )
            print(f"  {name}: jax_grad={grad_value:.8f}", flush=True)

        # The finite-difference approximation should agree with autodiff to a few
        # times h^2 (central-difference truncation error) for a well-behaved h;
        # too-small h instead suffers floating-point cancellation. Require the
        # BEST of the tested step sizes to be accurate, not every one of them.
        tolerance = 1e-5
        best_by_gate = {}
        for row in rows:
            best_by_gate[row["gate"]] = min(best_by_gate.get(row["gate"], float("inf")), row["relative_error"])
        failures = [gate for gate, err in best_by_gate.items() if err > tolerance]
        if failures:
            common.save_csv([r for r in rows if r["gate"] in failures], "R41_jax_gradients_FAILURES")
            raise AssertionError(f"No tested step size matched autodiff for gates: {failures}; see FAILURES csv")

        common.save_result(rows, "R41_jax_gradients", extra={"cutoff": CUTOFF, "alpha": [ALPHA.real, ALPHA.imag]})

        fig, ax = plt.subplots(figsize=(6.5, 4.2))
        for name in GATES:
            subset = [r for r in rows if r["gate"] == name]
            ax.loglog([r["step_size"] for r in subset], [max(1e-16, r["relative_error"]) for r in subset], "o-", label=name)
        ax.set_xlabel("Finite-difference step size h")
        ax.set_ylabel("Relative gradient error vs. JAX autodiff")
        ax.legend(fontsize=7)
        ax.set_title("JAX autodiff vs. central finite differences (experimental)")
        common.save_figure(fig, "R41_jax_gradients")
        plt.close(fig)

        common.print_summary("R41 JAX deterministic gradient validation", gates=list(GATES), best_relative_errors=best_by_gate)


    if __name__ == "__main__":
        main()
except SystemExit:
    pass


  rotation: jax_grad=-0.60207540


  displacement: jax_grad=1.00000000


  squeezing: jax_grad=0.98503815


  cubic_phase: jax_grad=1.18840449


  kerr: jax_grad=-0.78343505


  loss: jax_grad=0.13000000



=== R41 JAX deterministic gradient validation ===
  gates: ['rotation', 'displacement', 'squeezing', 'cubic_phase', 'kerr', 'loss']
  best_relative_errors: {'rotation': 2.332356517973767e-11, 'displacement': 1.5543122344756597e-14, 'squeezing': 7.941889810031469e-11, 'cubic_phase': 1.2192284179281408e-10, 'kerr': 8.081591556509901e-11, 'loss': 8.753681540313734e-15}


## R42: Conditional / estimator gradients (experimental, statistical estimators).

Tests the two generic Monte Carlo gradient-estimator primitives in
``photographiq.autodiff`` against known analytical identities:

1. ``score_function_surrogate`` (REINFORCE/likelihood-ratio estimator) on a
   parametrized Bernoulli branch, compared to the exact analytic gradient of
   E[f(X)] for X ~ Bernoulli(sigmoid(theta)).
2. ``gaussian_sample`` (pathwise/reparameterization primitive) on a simple
   quadratic objective, compared to the exact analytic pathwise gradient.

Both are Monte Carlo estimators: agreement is statistical (bias and standard
error over repeated seeds), not exact-precision agreement. This is
explicitly experimental functionality (docs/validation/feature-matrix.md).

In [44]:
"""R42: Conditional / estimator gradients (experimental, statistical estimators).

Tests the two generic Monte Carlo gradient-estimator primitives in
``photographiq.autodiff`` against known analytical identities:

1. ``score_function_surrogate`` (REINFORCE/likelihood-ratio estimator) on a
   parametrized Bernoulli branch, compared to the exact analytic gradient of
   E[f(X)] for X ~ Bernoulli(sigmoid(theta)).
2. ``gaussian_sample`` (pathwise/reparameterization primitive) on a simple
   quadratic objective, compared to the exact analytic pathwise gradient.

Both are Monte Carlo estimators: agreement is statistical (bias and standard
error over repeated seeds), not exact-precision agreement. This is
explicitly experimental functionality (docs/validation/feature-matrix.md).
"""

from __future__ import annotations
try:

    import sys
    from pathlib import Path

    import common  # noqa: E402

    if not common.HAS_JAX:
        print("SKIPPED: jax is not installed; see requirements_notes.md")
        sys.exit(0)

    import jax  # noqa: E402

    jax.config.update("jax_enable_x64", True)
    import jax.numpy as jnp  # noqa: E402
    import numpy as np  # noqa: E402

    from photographiq.autodiff import gaussian_sample, score_function_surrogate  # noqa: E402

    N_SAMPLES = 20000
    N_REPEATS = 10
    THETA0 = 0.3
    F_ONE, F_ZERO = 2.0, -1.0  # values attached to the two Bernoulli branches


    def score_function_case():
        def analytic_gradient(theta):
            p = jax.nn.sigmoid(theta)
            return (F_ONE - F_ZERO) * p * (1 - p)

        exact_gradient = float(analytic_gradient(THETA0))

        rows = []
        for repeat in range(N_REPEATS):
            rng = np.random.default_rng(1000 + repeat)
            p0 = float(jax.nn.sigmoid(THETA0))
            outcomes = rng.random(N_SAMPLES) < p0  # samples from Bernoulli(p(theta0))
            values = jnp.where(jnp.asarray(outcomes), F_ONE, F_ZERO)

            def loss(theta, outcomes=outcomes, values=values):
                p = jax.nn.sigmoid(theta)
                log_probs = jnp.where(jnp.asarray(outcomes), jnp.log(p), jnp.log1p(-p))
                return score_function_surrogate(values, log_probs)

            estimate = float(jax.grad(loss)(jnp.asarray(THETA0)))
            rows.append(estimate)

        rows = np.array(rows)
        return {
            "estimator": "score_function_surrogate",
            "exact_gradient": exact_gradient,
            "mean_estimate": float(rows.mean()),
            "bias": float(rows.mean() - exact_gradient),
            "standard_error": float(rows.std(ddof=1) / np.sqrt(N_REPEATS)),
            "n_samples_per_repeat": N_SAMPLES,
            "n_repeats": N_REPEATS,
        }


    def pathwise_case():
        sigma = 0.7

        def analytic_gradient(theta):
            return 2.0 * theta  # d/dtheta E_z[(theta+sigma*z)^2] = 2*theta since E[z]=0

        exact_gradient = float(analytic_gradient(THETA0))

        rows = []
        for repeat in range(N_REPEATS):
            rng = np.random.default_rng(2000 + repeat)
            z_batch = rng.standard_normal((N_SAMPLES, 1))

            def loss(theta, z_batch=z_batch):
                mean = jnp.array([theta])
                covariance = jnp.array([[sigma**2]])
                samples = jax.vmap(lambda z: gaussian_sample(mean, covariance, z))(jnp.asarray(z_batch))
                return jnp.mean(samples[:, 0] ** 2)

            estimate = float(jax.grad(loss)(THETA0))
            rows.append(estimate)

        rows = np.array(rows)
        return {
            "estimator": "gaussian_sample (pathwise)",
            "exact_gradient": exact_gradient,
            "mean_estimate": float(rows.mean()),
            "bias": float(rows.mean() - exact_gradient),
            "standard_error": float(rows.std(ddof=1) / np.sqrt(N_REPEATS)),
            "n_samples_per_repeat": N_SAMPLES,
            "n_repeats": N_REPEATS,
        }


    def main():
        plt = common.setup_style()
        results = [score_function_case(), pathwise_case()]

        # A statistically consistent unbiased estimator should have |bias| within
        # a few standard errors of zero; this is not an exact-precision check.
        failures = [r for r in results if abs(r["bias"]) > 5 * r["standard_error"] + 1e-6]
        if failures:
            common.save_json(failures, "R42_stochastic_gradients_FAILURES")
            raise AssertionError("An estimator's bias exceeded its statistical tolerance; see FAILURES json")

        common.save_json({"results": results}, "R42_stochastic_gradients")
        common.write_metadata("R42_stochastic_gradients")

        fig, ax = plt.subplots(figsize=(5.5, 3.8))
        names = [r["estimator"] for r in results]
        ax.errorbar(
            range(len(results)), [r["mean_estimate"] for r in results],
            yerr=[5 * r["standard_error"] for r in results], fmt="o", capsize=4, color="#24677b", label="Estimate +/- 5 SE",
        )
        ax.plot(range(len(results)), [r["exact_gradient"] for r in results], "x", markersize=10, color="#c66d27", label="Exact gradient")
        ax.set_xticks(range(len(results)))
        ax.set_xticklabels(names, rotation=15, ha="right", fontsize=8)
        ax.set_ylabel("Gradient estimate")
        ax.legend(fontsize=8)
        ax.set_title("Stochastic gradient estimators (experimental)")
        common.save_figure(fig, "R42_stochastic_gradients")
        plt.close(fig)

        common.print_summary("R42 conditional / estimator gradients", results=results)


    if __name__ == "__main__":
        main()
except SystemExit:
    pass



=== R42 conditional / estimator gradients ===
  results: [{'estimator': 'score_function_surrogate', 'exact_gradient': 0.7333749350722375, 'mean_estimate': 0.7330200980043476, 'bias': -0.00035483706788985714, 'standard_error': 0.00036830127328416507, 'n_samples_per_repeat': 20000, 'n_repeats': 10}, {'estimator': 'gaussian_sample (pathwise)', 'exact_gradient': 0.6, 'mean_estimate': 0.5890576154568313, 'bias': -0.010942384543168648, 'standard_error': 0.003075018088609748, 'n_samples_per_repeat': 20000, 'n_repeats': 10}]


## Aggregate manuscript tables (`generate_tables.py`)

In [45]:
import importlib
import generate_tables
importlib.reload(generate_tables)
generate_tables.main()


Generating manuscript summary tables...
  wrote compiler_accuracy.csv / compiler_accuracy.md (5 rows)
  wrote backend_agreement.csv / backend_agreement.md (2 rows)
  wrote piquasso_validation.csv / piquasso_validation.md (12 rows)
  wrote graphix_structure.csv / graphix_structure.md (4 rows)


  wrote non_gaussian_validation.csv / non_gaussian_validation.md (4 rows)
  wrote convergence_summary.csv / convergence_summary.md (5 rows)
  wrote performance_summary.csv / performance_summary.md (15 rows)
  wrote feature_status.csv / feature_status.md (12 rows)
Done. See D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\paper_experiments\tables
